# Лабораторная работа 1 — рабочий notebook

## 0. Окружение и настройки

In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import IPython

print("Версия Python:", sys.version)

print('numpy :', np.__version__)
print('pandas:', pd.__version__)
print('seaborn:', sns.__version__)
print('IPython:', IPython.__version__)

In [ ]:
# Журнал решений: одна запись на одно решение
LOG_COLUMNS = [
    'field', 'issue', 'scope', 'n_rows',
    'value_before', 'action', 'value_after', 'rationale',
]

cleaning_log_entries = {}


def add_log(key, field, issue, scope, n_rows,
            value_before, action, value_after, rationale):
    """Добавляет решение; повторный вызов с тем же ключом обновляет запись"""
    cleaning_log_entries[key] = {
        'field': field,
        'issue': issue,
        'scope': scope,
        'n_rows': int(n_rows),
        'value_before': str(value_before),
        'action': action,
        'value_after': str(value_after),
        'rationale': rationale,
    }

Проверим расположение clients.csv и transactions.csv

In [ ]:
# Проверка наличия данных
from pathlib import Path

DATA_DIR = Path("data/raw")

CLIENTS_PATH = DATA_DIR / "clients.csv"
TRANSACTIONS_PATH = DATA_DIR / "transactions.csv"

## 1. Загрузка сырых данных

### 1.1. Наивная загрузка

In [ ]:
if CLIENTS_PATH.is_file() and TRANSACTIONS_PATH.is_file():
    df_clients_naive = pd.read_csv(CLIENTS_PATH, encoding='utf-8')
    df_transactions_naive = pd.read_csv(TRANSACTIONS_PATH, encoding='cp1251')
    for name, preview in [("clients", df_clients_naive),
                      ("transactions", df_transactions_naive)]:
        
        display(preview.head(2))
    

вывод:
- clients.csv весь прочитан в таблицу (5702, 16)
- transactions.csv прочитан некорректно: получен один столбец вместо отдельных полей, видно, что разделителем является ';'

In [ ]:
# Типы всех столбцов
print("\n Типы данных всех столбцов:")
print(df_clients_naive.dtypes)

вывод:
типы требуют уточнения и не совсем корректны:
 - даты(birth_date, registration_date, card_open_date, app_last_login) представлены str, а не спец типом даты
  - satisfaction_score содержит целые оценки, а не дробные

### 1.2. Правильная загрузка clients

In [ ]:
from IPython.display import display
df_clients = pd.read_csv(
    CLIENTS_PATH,
    encoding='utf-8',
    dtype={
        'client_id': 'str',
        'city': 'str',
        'branch_code': 'str',  # Читаем как строку, так как в начале есть нули, которые при преобразовании в int станут незначащими и исчезнут
        'gender': 'category',
        'segment': 'category',
        'loyalty_tier': 'category',
        'card_type': 'category',
        'income_source': 'category',
        'has_mobile_app': 'category',
        'marketing_opt_in': 'category',
        'satisfaction_score': 'Int64',    # Это порядковая шкала, удобно Int так как порядок тогда уже задан 
        'monthly_income': 'float64',
    }
)

n_loaded = len(df_clients)

add_log(
    key='clients_read_types',
    field=('client_id; branch_code; gender; segment; loyalty_tier; card_type; income_source; has_mobile_app; marketing_opt_in; satisfaction_score; monthly_income'),
    issue='При загрузке необходимо задать смысловые типы столбцов',
    scope='Все строки clients.csv при корректном чтении',
    n_rows=n_loaded,
    value_before=('Представление в CSV; при пробном чтении branch_code интерпретировался как число'),
    action='Заданы типы при чтении',
    value_after=('Идентификаторы — строки; категориальные признаки — category; satisfaction_score — Int64; monthly_income — float64'),
    rationale=(
        'Сохранить ведущие нули кодов отделений, '
        'пропуски целочисленной оценки и смысловые типы признаков. '
        'Содержательные значения не подменяются.'
    ),
)

df_clients_raw = df_clients.copy(deep=True)

### 1.3. Правильная загрузка transactions

In [ ]:
# Загрузка транзакций с приведением типов и очисткой числовых значений
df_transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    sep=';',
    encoding='cp1251',
    dtype={
        'txn_id': 'str',
        'client_id': 'str',
        'currency': 'category',
        'category': 'category',
        'mcc_code': 'category',
        'channel': 'category',
        'status': 'category'
    },
    parse_dates=['txn_datetime'],
    date_format={'txn_datetime': 'ISO8601'},
    converters={
         # Обработка суммы: удаление пробелов, замена запятой на точку, защита от невалидных пропусков
        'amount': lambda value: float(
            ''.join(value.split()).replace(',', '.')
        ) if value.strip().casefold() not in {
            '', 'nan', 'na', 'n/a', 'null', 'none'
        } else np.nan
    }
)

n_transactions_loaded = len(df_transactions)

# Паспорт таблицы: типы, уникальные значения и пропуски
txn_overview = pd.DataFrame({
    'Тип': df_transactions.dtypes.astype(str),
    'Уникальных': df_transactions.nunique(),
    'Пропусков': df_transactions.isna().sum(),
    'Пропуски, %': df_transactions.isna().mean().mul(100),
})


display(txn_overview.round(2))
display(df_transactions.head(3))

# Сохраняем список исходных столбцов
txn_source_columns = df_transactions.columns.tolist()

# Фиксируем чтение и преобразование типов
add_log(
    key='transactions_read_types',
    field='transactions.*',
    issue='Нужны корректное чтение CSV и смысловые типы полей',
    scope='Все строки transactions.csv',
    n_rows=n_transactions_loaded,
    value_before='CSV: разделитель ;, кодировка cp1251, десятичная запятая',
    action='Заданы типы; преобразованы сумма и дата при чтении',
    value_after=(f'{len(df_transactions)} строк, {df_transactions.shape[1]} столбцов'),
    rationale=('Идентификаторы сохранены строками; сумма числовая; дата datetime. Строки не удалены.'),
)

вывод:
таблица transactions содержит 629'930 строк и 9 столбцов. Все столбцы прочитаны с нужными типами. Пропусков не обнаружено

## 2. Аудит качества

Здесь ничего не исправляем — находим и измеряем. Даты и суммы операций уже технически прочитаны загрузчиком, но содержательная очистка ещё не выполнена. Повторы до и после нормализации — разные результаты.

### 2.1. Объём, типы, ключи, единица наблюдения
Анкета клиента и операция — разные единицы наблюдения. Проверяем client_id и txn_id на их таблицах. Доли считаем от числа строк соответствующей таблицы.


In [ ]:
# Копии для аудита до содержательной очистки.
audit_clients_initial = df_clients.copy(deep=True)
audit_transactions_initial = df_transactions.copy(deep=True)
for name, table in [('clients', audit_clients_initial), ('transactions', audit_transactions_initial)]:
    
    passport = pd.DataFrame({'тип': table.dtypes.astype(str), 'уникальных': table.nunique(),
        'пропусков': table.isna().sum(), 'пропуски, %': table.isna().mean().mul(100)})
    display(passport.round(2))
    display(table.head(3))


### 2.2. Дубли

In [ ]:
# Измеряем повторы, ничего не удаляем.
audit_duplicates = []
for name, table, key in [('clients', audit_clients_initial, 'client_id'),
                         ('transactions', audit_transactions_initial, 'txn_id')]:
    text = table[key].astype('string')
    empty = text.isna() | text.str.strip().eq('').fillna(False)
    involved = ~empty & text.duplicated(keep=False)
    audit_duplicates.append({'Таблица': name, 'Всего строк': len(table),
        'Пустых ключей': int(empty.sum()),
        'Уникальных непустых ключей': int(text.loc[~empty].nunique()),
        'Лишних полных копий': int(table.duplicated().sum()),
        'Строк с повторным непустым ключом': int(involved.sum()),
        'Доля строк с повторным ключом, %': 100*involved.sum()/len(table)})
display(pd.DataFrame(audit_duplicates).set_index('Таблица').round(2))


### 2.3. Соответствие данных описанию полей из задания

In [ ]:
# Список столбцов для анализа
cat_cols = [
    'gender', 'city', 'segment', 'loyalty_tier', 'card_type',
    'income_source', 'has_mobile_app', 'satisfaction_score', 'marketing_opt_in'
]

# Частота категорий в каждом столбце
for col in cat_cols:
    print(f"\n Столбец: {col}")
    print(df_clients[col].value_counts(dropna=False))  # dropna=False — показывает NaN тоже

вывод:
- Есть разные написания одних и тех же городов
- Нет мобильного приложения(has_mobile_app) у 2140 и у 2140 клиентов NaT в app_last_login, проверим совпадение этих групп
- 1549 клиентов не указали источник зарплаты. Нужно посмотреть опять же связи с другими признаками
- 648 клиента отказались от участия маркетинговой коммуникации(не ошибка), поэтому они не будут включены ни в одну из групп контактной кампании, но не исключаем их из общего анализа 
- 3416 пропуска в satisfaction_score нельзя ничем заменить. Далее проверим есть ли связи с другими признаками

In [ ]:
# Список категориальных колонок транзакций для анализа и последующей нормализации
TXN_CAT_COLS = [ 'currency', 'category', 'mcc_code', 'channel', 'status']

# Смотрим исходные написания и частоты до нормализации
for col in TXN_CAT_COLS:
    counts = df_transactions[col].value_counts(dropna=False)

    category_overview = pd.DataFrame({
        'Количество': counts,
        'Доля, %': counts.div(len(df_transactions)).mul(100),
    })

    
    display(category_overview.round(2))

вывод:
пропусков в категориальных столбцах нет, категориальные столбцы имеют ограниченные количества категорий. Выявлены разные варианты написания категорий в некоторых столбцах.

Решение: объединяем известные варианты написания

In [ ]:
amount = df_transactions['amount']

quality = pd.Series({
    'Всего операций': len(df_transactions),
    'Пропусков во всех полях': df_transactions.isna().sum().sum(),
    'Пропусков суммы': amount.isna().sum(),
    'Бесконечных сумм': np.isinf(
        amount.to_numpy(dtype=float, na_value=np.nan)
    ).sum(),
    'Нулевых сумм': amount.eq(0).sum(),
    'Отрицательных сумм': amount.lt(0).sum(),
    'Положительных сумм': amount.gt(0).sum(),
}, name='Количество')

display(quality.to_frame())

# Показываем примеры только при наличии проблемных сумм
amount_to_check = (
    amount.isna()
    | ~np.isfinite(amount)
    | amount.eq(0)
)

if amount_to_check.any():
    display(
        df_transactions.loc[
            amount_to_check,
            ['txn_id', 'amount', 'currency', 'status'],
        ].head(10)
    )

вывод:
пропусков, бесконечных сумм и нулевых значений нет, но есть отрицательные суммы, которые надо проверить, например, на согласованность знака суммы и статуса

In [ ]:
# Знак суммы сопоставляем со статусом до нормализации.
audit_sign = pd.Series('Не определён', index=audit_transactions_initial.index)
audit_amount = audit_transactions_initial['amount']
audit_finite = audit_amount.notna() & np.isfinite(audit_amount)
audit_sign.loc[audit_finite & audit_amount.lt(0)] = 'Отрицательная'
audit_sign.loc[audit_finite & audit_amount.eq(0)] = 'Нулевая'
audit_sign.loc[audit_finite & audit_amount.gt(0)] = 'Положительная'
display(pd.crosstab(audit_transactions_initial['status'].astype('string').fillna('Нет статуса'),audit_sign))



### 2.4. Связь между таблицами

In [ ]:
# Сравниваем множества ключей, не размножая операции merge с повторными анкетами.
audit_client_ids = set(audit_clients_initial.client_id.dropna())
audit_operation_ids = set(audit_transactions_initial.client_id.dropna())
audit_found = audit_transactions_initial.client_id.isin(audit_client_ids)
display(pd.Series({'Операций всего': len(audit_transactions_initial),
    'Операций с найденным ID клиента': int(audit_found.sum()),
    'Операций без найденного ID': int((~audit_found).sum()),
    'Доля операций без найденного ID, %': 100*(~audit_found).mean(),
    'Уникальных непустых ID без анкеты': len(audit_operation_ids-audit_client_ids),
    'ID анкет без операций': len(audit_client_ids-audit_operation_ids)},name='Значение').to_frame())


вывод:
есть разные написания городов, повторные записи клиентов и операций, технический доход −999 и операции без клиентской анкеты. Отрицательные суммы нужно сопоставлять с возвратами. Количества до исправлений приведены выше; результаты после фильтров не подменяют этот аудит.

## 3. Препроцессинг

### 3.1. Типы, даты, числа


## 3. Предобработка данных

### 3.1. Преобразование дат и проверка типов


In [ ]:
DATE_FORMATS = {
    "birth_date": "%Y-%m-%d",
    "registration_date": "%d.%m.%Y",
    "card_open_date": "%d.%m.%Y",
    "app_last_login": "%Y-%m",
}

rows = []
for col, fmt in DATE_FORMATS.items():
    raw = df_clients[col]
    parsed = pd.to_datetime(raw.str.strip(), format=fmt, errors="coerce")
    rows.append({
        "поле": col, "пропусков в CSV": raw.isna().sum(),
        "не распознано": (raw.notna() & parsed.isna()).sum(),
        "NaT после разбора": parsed.isna().sum(),
    })
    df_clients[col] = parsed

    

date_parse_audit = pd.DataFrame(rows).set_index("поле")

# Записываем преобразование клиентских дат после сводки проверки.
add_log(key='clients_parse_dates',field='; '.join(DATE_FORMATS),
    issue='Даты прочитаны из CSV как текст',scope='Все клиентские анкеты до дедублирования',
    n_rows=len(df_clients),value_before='Текстовые даты в форматах CSV',
    action='Преобразованы в datetime по явно заданным форматам',
    value_after=date_parse_audit.to_string(),
    rationale=f'Форматы: {DATE_FORMATS}. Пропуски и ошибки разбора учтены отдельно. '
              'app_last_login содержит месяц: первый день — техническое представление, не день входа.')


display(date_parse_audit)
display(df_clients.head(3))



In [ ]:
overview = pd.DataFrame({
    "тип": df_clients.dtypes.astype(str),
    "уникальных": df_clients.nunique(),
    "пропусков": df_clients.isna().sum(),
    "пропуски, %": (df_clients.isna().mean() * 100).round(2),
})
display(overview)

### 3.2. Категориальные признаки и повторные записи

#### 3.2.1. Города: одно название

In [ ]:
# Приведём все города к одному написанию
import re

def normalize_city(city):
    """Brings the city to its standard appearance"""
    if pd.isna(city):
        return city
    
    city = city.strip()  # убираем пробелы
    # Убираем "г. " или "город " в начале
    city = re.sub(r'^(г\.|город)\s*', '', city, flags=re.IGNORECASE)
    city = city.title() # Один регистр
    
    abbreviations = {
        'Спб': 'Санкт-Петербург',
        'Санкт Петербург': 'Санкт-Петербург',
        'Н.Новгород': 'Нижний Новгород',
        'Н. Новгород': 'Нижний Новгород',
        'Ростов-На-Дону': 'Ростов-на-Дону',
    }
    
    return abbreviations.get(city, city)

city_before = df_clients['city'].copy()
city_after = city_before.apply(normalize_city)

# Учитываем только действительно изменившиеся значения
city_changed = (city_before.astype('string').fillna('')
        .ne(city_after.astype('string').fillna(''))
)

df_clients['city'] = city_after

add_log(
    key='clients_city_normalization',
    field='city',
    issue='Несколько написаний одного города',
    scope='normalize_city(city) отличается от исходного значения',
    n_rows=city_changed.sum(),
    value_before=( f'{city_before.nunique()} написаний; например «г. Москва», «МОСКВА», «СПб», «Санкт Петербург»'),
    action='Нормализация',
    value_after=f'{city_after.nunique()} единых названий городов',
    rationale=('Унифицированы регистр, префиксы и сокращения. Географический смысл сохранён; n_rows — число изменённых значений до удаления дублей.'),
)

print(df_clients['city'].value_counts(dropna=False))

вывод:
разные написания одного и того же города привели к одному названию, в итоги получили 6 городов

#### 3.2.2. Приложение и месяц входа

In [ ]:
# Проверяет соответствие отсутствия даты последнего входа и отсутствие мобильного приложения 
mask_anomaly = (
    (df_clients['has_mobile_app'] == 'Нет') & (df_clients['app_last_login'].notna())  
    |
    (df_clients['has_mobile_app'] == 'Да') & (df_clients['app_last_login'].isna())    
)

df_anomalies = df_clients[mask_anomaly].copy()




# Смотрим на аномалии
display(df_anomalies[['client_id', 'has_mobile_app', 'app_last_login']].head(20))

# Тепловая карта
app_login_check = pd.crosstab(
    df_clients['has_mobile_app'],
    df_clients['app_last_login'].isna().map({True: 'Нет даты', False: 'Есть дата'})
).reindex(columns=['Есть дата', 'Нет даты'], fill_value=0)

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.heatmap(app_login_check, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
ax.set(title='Приложение и дата входа', xlabel='Дата последнего входа', ylabel='Есть приложение')
ax.grid(False)
plt.tight_layout()
plt.show()

вывод:
даты последнего входа не указаны только у людей, у которых нет приложения. Данный пропуск ничем не заменяем

#### 3.2.3. Полные копии клиентских анкет

In [ ]:
n_clients_raw = len(df_clients)
n_unique_clients_raw = df_clients["client_id"].nunique()
# Проверяем, что нет пустых client_id — иначе нельзя корректно считать дубликаты
missing_id = (df_clients["client_id"].isna() | df_clients["client_id"].str.strip().eq("").fillna(False))
if missing_id.any():
    raise ValueError("Найдены пустые client_id: сначала разберите эти строки отдельно.")

fully_duplicated_rows = df_clients.duplicated().sum()
display(pd.Series({
    "строк до удаления повторов": n_clients_raw,
    "уникальных client_id": n_unique_clients_raw,
    "строк с повторяющимся ID": df_clients["client_id"].duplicated(keep=False).sum(),
    "лишних полных копий": fully_duplicated_rows,
}, name="количество").to_frame())

# Сохраняем удаляемые копии в отдельный df для журнала и последующей проверки
clients_full_duplicates = df_clients.loc[df_clients.duplicated()].copy()
df_clients = df_clients.drop_duplicates().copy()
n_after_full = len(df_clients)
n_full_removed = n_clients_raw - n_after_full

add_log(
    key='clients_full_duplicates', field='clients.*',
    issue='Полностью совпадающие клиентские анкеты',
    scope='df_clients.duplicated(keep="first") после нормализации',
    n_rows=n_full_removed, value_before=f'{n_clients_raw} строк',
    action='Удалены полные дубликаты', value_after=f'{n_after_full} строк',
    rationale=('Единица наблюдения — клиент. Полные копии анкеты не должны увеличивать его вес. Сохранено первое вхождение каждой одинаковой строки.'),
)



вывод:
обнаружено 40 полный дубликатов, которые были удалены(слишком маловероятно открытие двух абсолютно одинаковых карт в один день одним пользователем).
 
Остальные частичные дубликаты с одинаковым client_id нужно проверить: какие признаки отличаются(т.е. возможно ли что это 2 или более карт у одного клиента)

#### 3.2.4. Поиск конфликтующих строк

In [ ]:
# Проверяем все неидентичные записи клиентов с повторяющимся ID
conflict_mask = df_clients['client_id'].duplicated(keep=False)

clients_conflicts = (
    df_clients.loc[conflict_mask]
    .sort_values('client_id')
    .copy()
)




# Количество разных значений каждого поля внутри client_id
different = clients_conflicts.groupby('client_id', observed=True).nunique(dropna=False).gt(1)

# Число клиентов с расхождениями по каждому полю
conflict_counts = different.sum()

display(
    conflict_counts[conflict_counts > 0]
    .to_frame('n_clients_with_conflict')
)

conflict_cols = different.columns[different.any()].tolist()
display(clients_conflicts[['client_id'] + conflict_cols])

вывод:
после удаления полных копий остались 30 строк, относящихся к 15 client_id, отличия: 13 по городам и 15 по сегментам, так как остальные поля совпадают.
Удалять это строки нет оснований, так это клиент мог поменять со временем свои данные, но и определить какая запись сейчас действительна невозможно.
Решение: Сохраняем по одной строке на клиента. У всех 15 клиентов заменяет конфликтующие city и segment пропусками, так как нет вариантов определить какая запись более актуальная

#### 3.2.5. Объединение строк — решение конфликтов

In [ ]:
# Берём исходные конфликтующие анкеты
clients_conflicts_original = clients_conflicts.copy()
resolved_clients = clients_conflicts_original.copy()

# Определяем, какие поля расходятся у каждого клиента
different = clients_conflicts_original.groupby('client_id', observed=True).nunique(dropna=False).gt(1)


conflict_cols = different.columns[different.any()].tolist()
conflict_ids = clients_conflicts_original['client_id'].unique()

# Заменяем только противоречивые поля в исходных анкетах
for col in conflict_cols:
    ids = different.index[different[col]]

    selected = resolved_clients['client_id'].isin(ids)

    # Сохраняем примеры до замены
    before_examples = (
        resolved_clients.loc[selected, col]
        .astype('string')
        .drop_duplicates()
        .tolist()
    )

    resolved_clients.loc[selected, col] = pd.NA

    add_log(
        key=f'clients_conflict_{col}', field=col,
        issue='Противоречивые значения у одного client_id',
        scope='Клиенты с несколькими значениями поля среди конфликтующих анкет',
        n_rows=selected.sum(), value_before=before_examples,
        action='Заменено пропуском',
        value_after=f'NA в {int(selected.sum())} исходных версиях анкет; {len(ids)} уникальных клиентов',
        rationale='Достоверную версию определить нельзя. Заменяется только противоречивое поле; согласованные признаки клиента сохранены. n_rows — число затронутых версий до объединения.',
    )

# После устранения противоречий объединяем версии анкет в одну запись на клиента
n_versions_before = len(resolved_clients)

resolved_clients = resolved_clients.drop_duplicates()

n_versions_removed = n_versions_before - len(resolved_clients)

add_log(
    key='clients_conflict_versions_merge', field='client_id; clients.*',
    issue='После устранения противоречий версии анкеты совпадают',
    scope='resolved_clients после замены конфликтующих полей',
    n_rows=n_versions_removed,
    value_before=f'{n_versions_before} версий; {resolved_clients["client_id"].nunique()} клиентов',
    action='Объединены версии; удалены лишние копии',
    value_after=f'{len(resolved_clients)} анкет',
    rationale='После устранения противоречий сохраняем одну анкету на клиента, не удаляя самого клиента из базы.',
)



# Заменяем прежний результат обработки исправленными анкетами
df_clients = pd.concat(
    [
        df_clients.loc[~df_clients['client_id'].isin(conflict_ids)],
        resolved_clients,
    ],
    ignore_index=True,
)


display(
    df_clients.loc[
        df_clients['client_id'].isin(conflict_ids),
        ['client_id', 'city', 'segment'],
    ].sort_values('client_id')
)


вывод:
после объединения 30 конфликтных анкет в 15 записей справочник содержит 5647 уникальных client_id. Удалены лишние представления клиента, а не сами клиенты.

#### 3.2.6. Категории операций

In [ ]:
normalization_rows = []

for col in TXN_CAT_COLS:
    before = df_transactions[col].astype('string')

    # Убираем внешние и повторные пробелы
    after = (
        before
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )

    # Пустая строка означает отсутствие значения
    after = after.mask(after.eq(''))

    if col == 'currency':
        after = after.str.upper()

    elif col == 'status':
        after = after.str.casefold()

    elif col == 'channel':
        after = after.str.capitalize().replace({
            'Сбп': 'СБП',
        })

    elif col == 'category':
        after = after.str.capitalize().replace({
            'Кафе/рестораны': 'Кафе и рестораны',
            'Связь и интернет': 'Связь и Интернет',
            'Азс': 'АЗС',
        })

    # Считаем изменения, не принимая два пропуска за различие
    changed = (
        before.ne(after).fillna(False)
        | before.isna().ne(after.isna())
    )

    df_transactions[col] = after.astype('category')

    normalization_rows.append({
        'Поле': col,
        'Вариантов до': before.nunique(),
        'Вариантов после': after.nunique(),
        'Изменено строк': int(changed.sum()),
        'Пропусков после': int(after.isna().sum()),
    })

    add_log(
        key=f'transactions_normalize_{col}',
        field=col,
        issue='Различия в написании категориальных значений',
        scope='Строки, изменившиеся при нормализации',
        n_rows=changed.sum(),
        value_before=f'{before.nunique()} заполненных вариантов',
        action='Нормализация написаний',
        value_after=f'{after.nunique()} заполненных вариантов',
        rationale=('Унифицированы пробелы, регистр и известные варианты названий. Пропуски не заполнены; смысл категорий и значения MCC не переопределяются.'),
    )

# Проверяем результат преобразований
normalization_summary = (
    pd.DataFrame(normalization_rows)
    .set_index('Поле')
)
display(normalization_summary)


display(
    df_transactions['category']
    .value_counts(dropna=False)
    .to_frame('Количество')
)

# Сохраняем таблицу после нормализации, до удаления повторов
transactions_normalized = df_transactions.copy(deep=True)

вывод:
категориальные значения приведены к единому виду

#### 3.2.7. Проверка уникальности операций

In [ ]:
# Проверяем ключ до удаления повторов
txn_id = df_transactions['txn_id'].astype('string')

missing_txn_id = (
    txn_id.isna()
    | txn_id.str.strip().eq('').fillna(False)
)

n_before_duplicates = len(df_transactions)
n_unique_before = txn_id.nunique()

uniqueness_summary = pd.Series({
    'Всего строк': n_before_duplicates,
    'Уникальных заполненных txn_id': n_unique_before,
    'Пустых txn_id': missing_txn_id.sum(),
    'Лишних полных копий': df_transactions.duplicated().sum(),
    'Строк с повторяющимся заполненным txn_id': (
        ~missing_txn_id
        & txn_id.duplicated(keep=False)
    ).sum(),
}, name='Количество')

display(uniqueness_summary.to_frame())

# Для диагностики убираем полные копии только во временной таблице
transactions_no_copies = df_transactions.drop_duplicates()

conflict_id = transactions_no_copies['txn_id'].astype('string')
filled_id = (
    conflict_id.notna()
    & conflict_id.str.strip().ne('').fillna(False)
)

transaction_conflicts = (
    transactions_no_copies.loc[filled_id & conflict_id.duplicated(keep=False)]
    .sort_values('txn_id')
    .copy()
)

# Какие поля различаются у одного идентификатора
differing_fields = (
    transaction_conflicts
    .groupby('txn_id', observed=True)
    .nunique(dropna=False)
    .gt(1)
)



display(
    differing_fields.sum()
    .loc[lambda values: values.gt(0)]
    .to_frame('Идентификаторов с различием')
)

# Показываем несколько групп целиком
example_ids = (
    transaction_conflicts['txn_id']
    .drop_duplicates()
    .head(3)
)

display(
    transaction_conflicts.loc[transaction_conflicts['txn_id'].isin(example_ids)]
)

вывод:
для данной таблицы полные копии не несут никакого смысла это просто ошибки, поэтому полные копии надо удалить, а основные не полные копии проверяем.
Решение: удалить полные копии, проверить конфиктующие пары

#### 3.2.8. Полные копии операций

In [ ]:
# Сохраняем удаляемые копии для аудита
transaction_duplicates = df_transactions.loc[
    df_transactions.duplicated()
].copy()

df_transactions = df_transactions.drop_duplicates().copy()




add_log(
    key='transactions_full_duplicates',
    field='transactions.*',
    issue='Полностью совпадающие строки, включая txn_id',
    scope='Полные копии после нормализации категорий',
    n_rows=len(transaction_duplicates),
    value_before=f'{n_before_duplicates} строк',
    action='Удалены полные копии',
    value_after=f'{len(df_transactions)} строк',
    rationale=( 'Одна операция не должна учитываться несколько раз. Сохранено первое вхождение; удалённые копии доступны в transaction_duplicates.'),
)

вывод:
удалены все полные копии

#### 3.2.9. Разные версии времени операции

In [ ]:
# обновляем диагностику после удаления полных копий
filled_id = (
    df_transactions['txn_id'].notna()
    & df_transactions['txn_id'].astype('string').str.strip().ne('').fillna(False)
)
transaction_conflicts = df_transactions.loc[
    filled_id & df_transactions['txn_id'].duplicated(keep=False)
].copy()

other_cols = [
    col for col in df_transactions.columns
    if col not in ['txn_id', 'txn_datetime']
]

# Совпадение всех полей, кроме времени
same_other_fields = (
    transaction_conflicts
    .groupby('txn_id', observed=True)[other_cols]
    .nunique(dropna=False)
    .le(1)
    .all(axis=1)
)

dates = transaction_conflicts['txn_datetime']

time_check = (
    transaction_conflicts
    .assign(
        calendar_date=dates.dt.normalize(),
        is_midnight=dates.notna() & dates.eq(dates.dt.normalize()),
    )
    .groupby('txn_id', observed=True)
    .agg(
        n_rows=('txn_datetime', 'size'),
        n_valid_dates=('txn_datetime', 'count'),
        n_days=('calendar_date', 'nunique'),
        n_midnight=('is_midnight', 'sum'),
    )
)

time_check['same_other_fields'] = same_other_fields

resolvable = (
    time_check['n_rows'].eq(2)
    & time_check['n_valid_dates'].eq(2)
    & time_check['n_days'].eq(1)
    & time_check['n_midnight'].eq(1)
    & time_check['same_other_fields']
)

resolvable_ids = time_check.index[resolvable]

display(pd.Series({
    'Конфликтующих txn_id': len(time_check),
    'Соответствуют правилу пары': resolvable.sum(),
    'Требуют отдельного разбора': (~resolvable).sum(),
}, name='Количество').to_frame())

if (~resolvable).any():
    display(time_check.loc[~resolvable].head(10))

# Удаляем полуночную версию только у проверенных пар:
# один txn_id, один день, остальные поля совпадают,
# ровно одна из двух записей имеет время 00:00:00
n_before = len(df_transactions)
dates_all = df_transactions['txn_datetime']

remove_midnight = (
    df_transactions['txn_id'].isin(resolvable_ids)
    & dates_all.notna()
    & dates_all.eq(dates_all.dt.normalize())
)

# Сохраняем удаляемые строки для аудита
removed_midnight_rows = df_transactions.loc[
    remove_midnight
].copy()

df_transactions = df_transactions.loc[
    ~remove_midnight
].copy()

n_removed = len(removed_midnight_rows)

# Повторно проверяем конфликты после удаления
transaction_conflicts = (
    df_transactions.loc[
        df_transactions['txn_id'].notna()
        & df_transactions['txn_id'].duplicated(keep=False)
    ]
    .sort_values(['txn_id', 'txn_datetime'])
    .copy()
)

# Итог обработки
display(pd.Series({
    'Строк до обработки': n_before,
    'Разрешено конфликтующих txn_id': (
        removed_midnight_rows['txn_id'].nunique()
    ),
    'Удалено полуночных строк': n_removed,
    'Строк после обработки': len(df_transactions),
    'Осталось конфликтующих txn_id': (
        transaction_conflicts['txn_id'].nunique()
    ),
    'Осталось конфликтующих строк': len(transaction_conflicts),
}, name='Количество').to_frame())

# Показываем сами записи, для которых правило не решило конфликт
if not transaction_conflicts.empty:
    
    display(transaction_conflicts)
else:
    pass

# Фиксируем решение в существующем журнале
add_log(
    key='transactions_midnight_versions',
    field='txn_id; txn_datetime',
    issue='Разные версии времени одной операции',
    scope=('Две записи одного txn_id за один календарный день; остальные поля совпадают; ровно одна запись имеет время 00:00:00'
    ),
    n_rows=n_removed,
    value_before=(f'{n_before} строк; {len(resolvable_ids)} пар соответствуют правилу'),
    action='Удалена полуночная версия, сохранена версия с ненулевым временем',
    value_after=(f'{len(df_transactions)} строк; неразрешённых txn_id: {transaction_conflicts["txn_id"].nunique()}'
    ),
    rationale=('Принято допущение, что полуночная версия менее точна. Удаление применяется только к проверенным парам. Самостоятельные операции в полночь не удаляются. Удалённые строки сохранены в removed_midnight_rows; неразрешённые записи — в transaction_conflicts.'),
)

вывод:
все конфликтующие строки отличаются только временем. У всех дубликатов есть строчка со временем реальной покупки и временем 00:00:00, поэтому дубликаты с 00:00:00 удалили. Правило применяется к проверенным парам

### 3.3. Пропуски
#### 3.3.1. Техническое значение дохода и флаги


In [ ]:
LOW_INCOME_LIMIT = 10_000

# Сохраняем значения до замены для сравнения
income_before = df_clients['monthly_income'].copy()
n_minus999 = income_before.eq(-999).sum()

# Другие отрицательные значения требуют отдельного решения
other_negative = income_before.lt(0) & income_before.ne(-999)


# Заменяем некорректный код пропуском, не удаляя клиентов
df_clients['monthly_income'] = income_before.replace(-999, np.nan)
income = df_clients['monthly_income']

add_log(
    key='clients_income_minus999',
    field='monthly_income',
    issue='Отрицательное значение −999 в поле дохода',
    scope='monthly_income == -999 после обработки повторов',
    n_rows=n_minus999,
    value_before=f'−999; пропусков до обработки: {int(income_before.isna().sum())}',
    action='Заменено пропуском',
    value_after=f'NaN; всего пропусков: {int(income.isna().sum())}',
rationale=('−999 не является достоверным неотрицательным доходом; заменён пропуском без удаления клиента. Точный доход неизвестен.'),
)

df_clients['income_missing'] = income.isna()
df_clients['income_low_flag'] = (income.gt(0) & income.lt(LOW_INCOME_LIMIT)).fillna(False)

add_log(
    key='clients_income_missing_flag',
    field='income_missing',
    issue='Нужно отмечать отсутствие суммы после очистки',
    scope='Все клиенты; monthly_income.isna()',
    n_rows=len(df_clients),
    value_before='Флаг отсутствовал',
    action='Создан флаг',
    value_after=f'True у {int(df_clients["income_missing"].sum())} клиентов',
    rationale='Отделить отсутствие суммы от её размера. Учитываются исходные пропуски и заменённые −999.',
)

add_log(
    key='clients_income_low_flag',
    field='income_low_flag',
    issue='Нужно выделить низкие положительные доходы',
    scope=f'Все клиенты; 0 < monthly_income < {LOW_INCOME_LIMIT}',
    n_rows=len(df_clients),
    value_before='Флаг отсутствовал',
    action='Создан диагностический флаг',
    value_after=f'True у {int(df_clients["income_low_flag"].sum())} клиентов',
    rationale='Порог используется для дополнительной проверки и анализа чувствительности, а не как доказательство ошибки. Пропуски не относятся к низким доходам.',
)


# Сводная таблица до и после обработки
income_check = pd.DataFrame({
    'До обработки': [income_before.notna().sum(), income_before.isna().sum(), n_minus999],
    'После обработки': [income.notna().sum(), income.isna().sum(), income.eq(-999).sum()],
}, index=['Доход заполнен', 'Доход отсутствует', 'Значение −999'])
display(income_check)

add_log(
    key='clients_low_income_retained',
    field='monthly_income',
    issue='Необычно низкие положительные суммы',
    scope=f'0 < monthly_income < {LOW_INCOME_LIMIT}',
    n_rows=df_clients['income_low_flag'].sum(),
    value_before=f'Положительные суммы ниже {LOW_INCOME_LIMIT} ₽',
    action='Оставлено',
    value_after='Суммы сохранены; группа отмечена income_low_flag',
    rationale='Причина низких сумм не установлена. Не умножать на 1000 и не заменять медианой. Влияние проверяется сравнением результатов с группой и без неё.',
)

add_log(
    key='clients_high_income_retained',
    field='monthly_income',
    issue='Доходы выше 1 млн ₽',
    scope='monthly_income > 1000000',
    n_rows=income.gt(1_000_000).sum(),
    value_before='Доходы выше 1000000 ₽',
    action='Оставлено',
    value_after='Суммы сохранены без ограничения сверху',
    rationale='Проверен контекст клиентов; подтверждения ошибки нет. Крупная сумма сама по себе не основание для удаления.',
)

Решение: 
- Значения −999 заменены пропусками, так как очень похожи на значения(флаг) которые кто-то ставил при отсутствии дохода. Достоверно определить нельзя.
- 175 положительных доходов ниже 10'000 отмечены флагом(income_low_flag) (Похоже единицы измерения когда-то перепутали и забыли написать, что это тыщи, так как по форме распределения это значения не вписываются

#### 3.3.2 Исследование отсутствующих сведений

In [ ]:
# Создаем флаг отсутствия "не указан"
source_text = (
    df_clients['income_source']
    .astype('string')
    .str.strip()
    .str.casefold()
)

df_clients['income_source_missing'] = (
    source_text.isna()
    | source_text.isin(['', 'не указан'])
)

df_clients['score_missing'] = df_clients['satisfaction_score'].isna()
df_clients['city_missing'] = df_clients['city'].isna()
df_clients['segment_missing'] = df_clients['segment'].isna()

# Три основных показателя для дальнейших сравнений
flags = [
    'income_missing',
    'income_source_missing',
    'score_missing',
]

flag_labels = {
    'income_missing': 'Сумма дохода',
    'income_source_missing': 'Источник дохода',
    'score_missing': 'Оценка',
    'city_missing': 'Город',
    'segment_missing': 'Сегмент',
}

summary_flags = flags + ['city_missing', 'segment_missing']

missing_summary = pd.DataFrame({
    'Отсутствует': df_clients[summary_flags].sum(),
    'Доля базы, %': df_clients[summary_flags].mean().mul(100),
}).rename(index=flag_labels)

display(missing_summary.round(2))

new_missing_flags = {
    'income_source_missing': ('income_source', 'NaN, пустая строка или «Не указан» после удаления краевых пробелов и приведения к нижнему регистру'),
    'score_missing': ('satisfaction_score', 'satisfaction_score.isna()'),
    'city_missing': ('city', 'city.isna()'),
    'segment_missing': ('segment', 'segment.isna()'),
}

for flag, (source_field, rule) in new_missing_flags.items():
    add_log(
        key=f'clients_create_{flag}',
        field=flag,
        issue=f'Нужно учитывать отсутствие сведений в {source_field}',
        scope=f'Все клиенты; условие True: {rule}',
        n_rows=len(df_clients),
        value_before='Отдельный флаг отсутствовал',
        action='Создан флаг',
        value_after=f'Флаг рассчитан для {len(df_clients)} клиентов; True у {int(df_clients[flag].sum())}',
        rationale='Отдельно учитывать полноту сведений в групповых сравнениях и при выборе наблюдений для конкретного анализа. Исходные значения не изменяются и не заполняются.',
    )


вывод:
- сумма и источник дохода нужно проверить связь. 
- пропуски городов и сегментов получились при объединении одинаковый id клиентов

*месяц последнего входа тоже уже было объяснено раньше месте

#### 3.3.3 Полнота по сегментам

In [ ]:
def missing_by(group, flag):
    """Number of clients and proportion of missing values per group"""
    result = (
        df_clients.groupby(group, observed=True, dropna=False)[flag]
        .agg(n_clients='size', n_missing='sum', missing_pct='mean') # mean=sum/mean 
    )
    result['missing_pct'] = result['missing_pct'].mul(100)
    return result


segment_group = df_clients.groupby('segment', observed=True, dropna=False)

segment_counts = segment_group.size()
segment_rates = segment_group[flags].mean().mul(100)

segment_table = segment_rates.rename(columns=flag_labels).copy()
segment_table.insert(0, 'Клиентов', segment_counts)

display(segment_table.round(2))

# Подписи
heatmap_data = segment_rates.rename(columns=flag_labels).copy()
heatmap_data.index = [
    f'{segment if pd.notna(segment) else "Сегмент отсутствует"} '
    f'(n={int(n)})'
    for segment, n in zip(segment_rates.index, segment_counts)
]

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='Blues', vmin=0, vmax=100, linewidths=0.5, cbar_kws={'label': 'Отсутствует внутри группы, %'}, ax=ax)
ax.set(title='Полнота сведений по клиентским сегментам', xlabel='', ylabel='')
ax.tick_params(axis='both', labelrotation=0)
plt.tight_layout()
plt.show()


вывод:
В массовом сегменте больше всего пропусков информацию о доходах и реже остальных оставляли оценку.

#### 3.3.4 Сумма дохода и источник

In [ ]:
source_presence = missing_by('income_source_missing', 'income_missing'
).reindex([False, True])

source_presence.index = [
    'Источник известен',
    'Источник неизвестен',
]

display(source_presence.round(2))

plot_data = source_presence.loc[
    source_presence['n_clients'].gt(0)
]

labels = [
    f'{name} (n={int(row["n_clients"])})'
    for name, row in plot_data.iterrows()
]

fig, ax = plt.subplots(figsize=(9, 3.5))

bars = ax.barh(labels, plot_data['missing_pct'])
ax.bar_label(bars, labels=[f'{value:.1f}%' for value in plot_data['missing_pct']], padding=4)
ax.set(title='Отсутствие суммы дохода при разной полноте источника', xlabel='Сумма дохода отсутствует, % группы', ylabel='', xlim=(0, max(5, plot_data['missing_pct'].max() * 1.3)))
ax.invert_yaxis()
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

unknown_source_with_income = (
    df_clients['income_source_missing']
    & ~df_clients['income_missing']
).sum()



вывод:
клиент, которые не указали источник дохода, пропускали сумму дохода в ~5 раз чаще, чем клиенты с зарплатой в другом банке или в нашем банке. 

#### 3.3.5 Полнота оценок

In [ ]:
score_by_loyalty = missing_by('loyalty_tier', 'score_missing')

# Сортировка по доле пропусков облегчает сравнение
score_by_loyalty = score_by_loyalty.sort_values('missing_pct')

display(score_by_loyalty.round(2))

labels = [
    f'{name if pd.notna(name) else "Уровень отсутствует"} '
    f'(n={int(row["n_clients"])})'
    for name, row in score_by_loyalty.iterrows()
]

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.barh(labels, score_by_loyalty['missing_pct'])
ax.bar_label(bars, labels=[f'{value:.1f}%' for value in score_by_loyalty['missing_pct']], padding=4)
ax.set(title='Отсутствие оценок по уровню лояльности', xlabel='Оценка отсутствует, % группы', ylabel='', xlim=(0, 105))
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

# Вычисляем разброс доли пропусков между группами лояльности для оценки неоднородности
score_range = (score_by_loyalty['missing_pct'].max() - score_by_loyalty['missing_pct'].min())


вывод:
Клиенты базового и серебряного уровней в ~2 раза реже оставляют оценки, чем клиенты золотого и платинового уровней.
Решение: satisfaction_score оставляем с пропусками

#### 3.3.6 Совместные пропуски

In [ ]:
n_missing = df_clients[flags].sum(axis=1)
complete_mask = n_missing.eq(0)

completeness = (
    n_missing.value_counts()
    .reindex(range(4), fill_value=0)
    .rename_axis('Отсутствует полей из трёх')
    .to_frame('Клиентов')
)

completeness['Доля базы, %'] = (
    completeness['Клиентов'].div(len(df_clients)).mul(100)
)

display(completeness.round(2))

# Оцениваем, как удаление неполных анкет исказит распределение клиентов по сегментам
segment_for_compare = (
    df_clients['segment']
    .astype('string')
    .fillna('Сегмент отсутствует')
)

all_counts = segment_for_compare.value_counts()
complete_counts = (
    segment_for_compare.loc[complete_mask]
    .value_counts()
    .reindex(all_counts.index, fill_value=0)
)

composition = pd.DataFrame({
    'Вся база, клиентов': all_counts,
    'Полные анкеты, клиентов': complete_counts,
})

composition['Вся база, %'] = (
    all_counts.div(len(df_clients)).mul(100)
)

n_complete = int(complete_mask.sum())

composition['Полные анкеты, %'] = (
    complete_counts.div(n_complete).mul(100)
    if n_complete > 0
    else np.nan
)

composition['Изменение доли, п.п.'] = (composition['Полные анкеты, %'] - composition['Вся база, %'])

display(composition.round(2))




# Фиксируем в журнале решение не удалять и не заполнять пропуски, чтобы не вносить искусственные искажения
retained_missing = {
    'monthly_income': ('income_missing', 'Неизвестный доход не равен нулю. Общее заполнение изменит распределение.'),
    'income_source': ('income_source_missing', 'Источник нельзя достоверно восстановить по наличию или размеру суммы.'),
    'satisfaction_score': ('score_missing', 'Заполнение создало бы ответы клиентов, которые оценку не предоставили.'),
}

for field, (flag, reason) in retained_missing.items():
    add_log(
        key=f'clients_keep_missing_{field}',
        field=field,
        issue='Отсутствуют содержательные сведения',
        scope=f'{flag} == True',
        n_rows=df_clients[flag].sum(),
        value_before='Пропуск или обозначение отсутствующих сведений',
        action='Оставлено без заполнения',
        value_after='Исходное отсутствие сведений сохранено',
        rationale=reason,
    )

add_log(
    key='clients_no_global_dropna',
    field='clients.*',
    issue='Неполные анкеты по сумме, источнику дохода или оценке',
    scope='Хотя бы один из трёх основных флагов равен True',
    n_rows=(~complete_mask).sum(),
    value_before='Анкеты с одним или несколькими отсутствующими полями',
    action='Оставлено; общее удаление неполных строк не применяется',
    value_after=f'В базе сохранены все {len(df_clients)} клиентов',
    rationale='Для каждого исследования нужны свои поля. Отсутствие оценки или дохода не должно автоматически исключать клиента из анализа операций.',
)

вывод: если удалять все строки с пропусками очень уменьшится выборка, поэтому для каждого исследования надо выбирать только необходимые признаки
- клиентскую базу сохраняем без общего удаления неполных анкет
- доходы и оценки не заполняем нулями, медианами или средним
- для каждого расчёта подбираем наблюдения по наличию необходимых сведений 

### 3.4. Выбросы и логическая согласованность
Обнаружение и решение — разные шаги. Диагностические графики рядом с решениями сохранены из начала работы.


#### 3.4.1. Возраст и даты

In [ ]:
REFERENCE_DATE = pd.Timestamp('2025-06-30')

date_cols = [
    'birth_date',
    'registration_date',
    'card_open_date',
    'app_last_login',
]
for col in date_cols:
    pass


def full_years(birth_date, event_date):
    """
    Calculates the age in full years between the date of birth and the event.
    При неизвестной дате возвращает пропуск.
    """
    # Проверяем, наступил ли день рождения в году события
    before_birthday = (
        (event_date.dt.month < birth_date.dt.month)
        | ((event_date.dt.month == birth_date.dt.month)
           & (event_date.dt.day < birth_date.dt.day))
    )

    years = event_date.dt.year - birth_date.dt.year - before_birthday.astype(int)

    return years.where(birth_date.notna() & event_date.notna()).astype('Int64')


# Создаём Series с контрольной датой для каждого клиента
reference_dates = pd.Series(REFERENCE_DATE, index=df_clients.index)

for target, event in {
    'age': reference_dates,
    'age_at_registration': df_clients['registration_date'],
    'age_at_card_open': df_clients['card_open_date'],
}.items():
    df_clients[target] = full_years(df_clients['birth_date'], event)
    
add_log(
    key='clients_age_features', field='age; age_at_registration; age_at_card_open',
    issue='Для анализа нужны возрасты на дату среза и событий',
    scope='Все клиенты после обработки повторных анкет',
    n_rows=len(df_clients), value_before='Производные возрастные признаки отсутствовали',
    action='Созданы признаки',
    value_after=f'Полные годы на {REFERENCE_DATE:%d.%m.%Y}, на дату регистрации и открытия карты',
    rationale='Возраст рассчитан с учётом наступления дня рождения. Исходные даты не изменяются.',
)


age_fields = ['age', 'age_at_registration', 'age_at_card_open']
display(df_clients[age_fields].describe().round(1))
display(df_clients[date_cols].isna().sum().to_frame('неизвестна дата'))

age = df_clients['age'].dropna()

# Диаграмма распределения клиентов по возрасту
plt.figure(figsize=(12, 5))
# discrete=True рисует отдельные столбцы для каждого целого возраста, а не сплошную кривую
sns.histplot(age, discrete=True, color='skyblue', edgecolor='black')
plt.title('Распределение клиентов по возрасту (полных лет)')
plt.xlabel('Возраст')
plt.ylabel('Количество клиентов')
plt.grid(axis='y', alpha=0.7)
plt.tight_layout()
plt.show()

# Столбцы для ручного просмотра анкет
age_cols = [
    'client_id', 'birth_date', 'age',
    'registration_date', 'age_at_registration',
    'card_open_date', 'age_at_card_open', 'card_type',
]

# Самые молодые
display(df_clients.sort_values('age')[age_cols].head(5))
# Самые пожилые
display(df_clients.sort_values('age', ascending=False)[age_cols].head(5))

вывод:
возраст клиентов на 30.06.2025 составляет от 23 до 88 лет. Возраста на данный момент полностью подходят для любых операций. Но нужно ещё проверить их возраст на момент регистрации и открытия карты.
Возраст 88 на диаграмме выглядит анамально и не всписывается в общую форму. Его нужно исследовать подробнее, пока нельзя сказать, что это точно ошибка

#### 3.4.2. Аномальный возраст 88

In [ ]:
# диагностический флаг
AGE_TO_CHECK = 88

df_clients['age_88_flag'] = (
    df_clients['age'].eq(AGE_TO_CHECK).fillna(False)
)

add_log(
    key='clients_age88_flag',
    field='age_88_flag',
    issue='Нужно выделить скопление наблюдений возраста 88 лет',
    scope='Все клиенты; True при age == AGE_TO_CHECK',
    n_rows=len(df_clients),
    value_before='Флаг отсутствовал',
    action='Создан диагностический флаг',
    value_after=f'True у {int(df_clients["age_88_flag"].sum())} клиентов',
    rationale='Возраст сам по себе возможен. Флаг позволяет проверить группу отдельно без удаления клиентов и изменения даты рождения.',
)

add_log(
    key='clients_age88_retained',
    field='birth_date; age',
    issue='Скопление наблюдений на максимальном возрасте',
    scope=f'age == {AGE_TO_CHECK}',
    n_rows=df_clients['age_88_flag'].sum(),
    value_before=f'Возраст {AGE_TO_CHECK} лет',
    action='Оставлено',
    value_after='Даты рождения и возраст сохранены',
    rationale='Подтверждения ошибки нет. Повышенная частота значения не является достаточным основанием для удаления.',
)


age_88 = df_clients.loc[df_clients['age_88_flag']].copy()



# Проверяем, не связан ли всплеск с повторением одной даты рождения


display(
    age_88['birth_date']
    .value_counts()
    .head(10)
    .to_frame('клиентов')
)

# Смотрим даты и возраст на момент прошлых событий
display(
    age_88[ ['client_id', 'birth_date', 'age', 'registration_date', 'age_at_registration', 'card_open_date', 'age_at_card_open']]
    .sort_values('birth_date')
    .head(15)
)

вывод:
Даты рождения, регистрации и открытия карты все очень разные, что не похоже на заглушку. Но установить причину не удалось. Люди 88 возраста составляют 2, 18 % от всех.
Решение:
 - сохраняем клиентов
 - оставляем диагностический флаг(age_88_flag)

#### 3.4.3. Возрастные рамки и даты

In [ ]:
birth = df_clients['birth_date']
reg = df_clients['registration_date']
card = df_clients['card_open_date']
login_month = df_clients['app_last_login'].dt.to_period('M')

age_flags = pd.DataFrame({
    'birth_in_future': birth.gt(REFERENCE_DATE),
    'age_over_110': df_clients['age'].gt(110),
    'age_under_14': df_clients['age'].lt(14),
    'card_open_under_14': df_clients['age_at_card_open'].lt(14),
    'credit_card_under_18': (df_clients['card_type'].eq('Кредитная')
                           & df_clients['age_at_card_open'].lt(18)),
}).fillna(False)
date_flags = pd.DataFrame({
    'registration_before_birth': reg.lt(birth),
    'card_before_birth': card.lt(birth),
    'registration_in_future': reg.gt(REFERENCE_DATE),
    'card_open_in_future': card.gt(REFERENCE_DATE),
    'card_before_registration': card.lt(reg),
    'login_after_snapshot_month': login_month.gt(REFERENCE_DATE.to_period('M')),
    'login_before_registration_month': login_month.lt(reg.dt.to_period('M')),
    'login_before_birth_month': login_month.lt(birth.dt.to_period('M')),
}).fillna(False)
checks = pd.concat([age_flags, date_flags], axis=1)
display(checks.sum().to_frame('клиентов с сигналом'))
problem_clients = df_clients.loc[checks.any(axis=1)].copy()
view_cols = ['client_id', *date_cols, *age_fields, 'card_type']


add_log(
    key='clients_credit_under_18_retained',
    field='card_type; card_open_date; age_at_card_open',
    issue='Кредитная карта открыта в возрасте до 18 лет',
    scope='age_flags["credit_card_under_18"] == True',
    n_rows=age_flags['credit_card_under_18'].sum(),
    value_before='Тип текущей карты — кредитная; возраст открытия ниже 18',
    action='Оставлено',
    value_after='Тип карты и даты сохранены; случаи доступны в аудите',
    rationale='Не установлено, какое поле ошибочно или отличается по смыслу. Не исправлять значения предположением; учитывать ограничение при анализе возраста открытия.',
)

add_log(
    key='clients_early_login_retained',
    field='app_last_login; registration_date',
    issue='Месяц последнего входа раньше месяца регистрации',
    scope='date_flags["login_before_registration_month"] == True',
    n_rows=date_flags['login_before_registration_month'].sum(),
    value_before='Месяц входа < месяца регистрации',
    action='Оставлено',
    value_after='Даты сохранены; противоречия доступны в аудите',
    rationale='Достоверное значение неизвестно. Не подменять даты; сравнение учитывает месячную точность app_last_login.',
)


# сохраняем флаги отдельно
client_date_audit = checks.set_axis(df_clients['client_id'], axis=0)


вывод:
- у 3 клиентов на данный момент тип карты указан как "кредитная", а возраст при открытии этой карты составляет 16-17 лет. Это может свидетельствовать либо о несогласованности атрибутов, либо о правонарушении, либо об эмансипации.
 - у 15 клиентов месяц последнего входа предшествует месяцу регистрации. 
Проанализируем подробнее:

#### 3.4.4. Примеры временных противоречий

In [ ]:
# Два диагностических графика на одном рисунке
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Возраст при открытии кредитной карты
credit = df_clients.loc[
    df_clients['card_type'].eq('Кредитная')
    & df_clients['age'].notna()
    & df_clients['age_at_card_open'].notna()
].copy()

young = credit['age_at_card_open'].lt(18)

for mask, color, label, size in [
    (~young, 'blue', '18 лет и старше', 15),
    (young, 'crimson', 'Младше 18 лет', 65),
]:
    axes[0].scatter(
        credit.loc[mask, 'age'].astype(float),
        credit.loc[mask, 'age_at_card_open'].astype(float),
        color=color, s=size, alpha=0.6,
        label=f'{label} (n={int(mask.sum())})',
    )

axes[0].axhline(18, color='darkorange', linestyle='--', label='Ориентир: 18 лет')

# Подпишем только несколько подозрительных точек
for _, row in credit.loc[young].head(5).iterrows():
    axes[0].annotate(
        row['client_id'],
        (float(row['age']), float(row['age_at_card_open'])),
        xytext=(5, 5), textcoords='offset points', fontsize=8,
    )

axes[0].set(
    title='Возраст при открытии кредитной карты',
    xlabel='Возраст на дату среза, лет',
    ylabel='Возраст при открытии, лет',
)
axes[0].legend(fontsize=8)

# 2. Месяц регистрации и месяц последнего входа
timeline = df_clients.loc[
    df_clients['registration_date'].notna()
    & df_clients['app_last_login'].notna()
].copy()

registration_month = timeline['registration_date'].dt.to_period('M')
last_login_month = timeline['app_last_login'].dt.to_period('M')
early_login = last_login_month.lt(registration_month)

# На графике обе даты представлены началом соответствующего месяца
x = registration_month.dt.to_timestamp()
y = last_login_month.dt.to_timestamp()

for mask, color, label, size in [
    (~early_login, 'blue', 'Вход не раньше регистрации', 15),
    (early_login, 'crimson', 'Вход раньше регистрации', 65),
]:
    axes[1].scatter(
        x.loc[mask], y.loc[mask],
        color=color, s=size, alpha=0.6,
        label=f'{label} (n={int(mask.sum())})',
    )

if not timeline.empty:
    lower = min(x.min(), y.min())
    upper = max(x.max(), y.max())

    axes[1].plot(
        [lower, upper], [lower, upper],
        color='darkorange', linestyle='--',
        label='Один месяц',
    )

axes[1].set(
    title='Регистрация и последний вход в приложение',
    xlabel='Месяц регистрации',
    ylabel='Месяц последнего входа',
)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Временно показываем все столбцы анкет
with pd.option_context('display.max_columns', None):
    
    display(df_clients.loc[date_flags['login_before_registration_month']].sort_values('client_id').head(15))
    
    display(df_clients.loc[age_flags['credit_card_under_18']].sort_values('client_id').head(3))

вывод:
графики показывают красными точкам клиентов, которые имееют несоответствие. Решение: 
- клиентов сохраняем
- даты не исправляем
- сохраняем флаги
- 15 клиентов со спорным месяцем входа не будет использовать для выводов об активности

#### 3.4.5. Распределение дохода

In [ ]:
# Для анализа распределения берём доходы без пропусков
income_known = df_clients['monthly_income'].dropna().copy()

p95 = income_known.quantile(0.95)
income_for_plot = income_known.loc[income_known.le(p95)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_samples = [
    (axes[0], income_known, 'Все известные доходы'),
    (axes[1], income_for_plot, f'Увеличенный участок: доходы до P95 ({p95:,.0f} ₽)'),
]

# Строим два графика: полный диапазон и обрезанный по 95-му перцентилю
for ax, values, title in plot_samples:
    sns.histplot(x=values, bins=50, edgecolor='white', ax=ax)
    # На обеих панелях — характеристики полной выборки известных доходов
    ax.axvline(income_known.median(), color='crimson', linestyle='--', label=f'Медиана всех доходов: {income_known.median():,.0f} ₽')
    ax.axvline(income_known.mean(), color='darkorange', linestyle='-.', label=f'Среднее всех доходов: {income_known.mean():,.0f} ₽')
    ax.axvline(LOW_INCOME_LIMIT, color='gray', linestyle=':', label='Порог проверки: 10 000 ₽')
    ax.set(title=f'{title}\nn = {len(values)}', xlabel='Ежемесячный доход, ₽', ylabel='Количество клиентов')
    ax.ticklabel_format(axis='x', style='plain')
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()



#### 3.4.6. Описательная статистика и чувствительность

In [ ]:
# Сравниваем расширенную описательную статистику полной выборки и выборки без низких доходов
income_without_low = income.loc[
    income.notna() & ~df_clients['income_low_flag']
]

samples = {
    'Все известные доходы': income_known,
    'Без доходов ниже 10 000 ₽': income_without_low,
}

rows = {}

for name, values in samples.items():
    row = values.describe( percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    row['IQR'] = values.quantile(0.75) - values.quantile(0.25)
    row['Асимметрия'] = values.skew()
    row['Избыточный эксцесс'] = values.kurt()
    rows[name] = row

income_stats = pd.DataFrame(rows).rename(index={
    'count': 'Количество',
    'mean': 'Среднее',
    'std': 'Стандартное отклонение',
    'min': 'Минимум',
    '1%': 'P1',
    '5%': 'P5',
    '25%': 'Q1',
    '50%': 'Медиана',
    '75%': 'Q3',
    '95%': 'P95',
    '99%': 'P99',
    'max': 'Максимум',
})

display(income_stats.round(2))

# Оцениваем изменение основных показателей
sensitivity = pd.DataFrame({
    'Все известные доходы': [income_known.mean(), income_known.median()],
    'Без низких доходов': [income_without_low.mean(), income_without_low.median()],
}, index=['Среднее', 'Медиана'])

sensitivity['Изменение, %'] = (sensitivity['Без низких доходов']
                               .div(sensitivity['Все известные доходы'])
                               .sub(1)
                               .mul(100))
display(sensitivity.round(2))

Вывод: длинный правых хвост, поэтому прежде всего будем использовать медиану и квартили
Решение: клиентов с низкими доходами не удаляем. Используем выборку без низких доходов для дальнейшего анализа.

#### 3.4.7. Крайние доходы

In [ ]:
# Считаем количество наблюдений на границах распределения (нули, экстремумы)
income_cols = ['client_id', 'monthly_income', 'income_source', 'segment', 'loyalty_tier', 'city']

income_limits = pd.Series({
    'Отрицательные значения после обработки': income.lt(0).sum(),
    'Нулевые доходы': income.eq(0).sum(),
    'Положительные доходы ниже 10 000 ₽': df_clients['income_low_flag'].sum(),
    'Доходы выше 1 000 000 ₽': income.gt(1_000_000).sum(),
}, name='Количество')

display(income_limits.to_frame())

# Выводим примеры анкет с минимальными и максимальными доходами для ручной проверки

display(df_clients.loc[income.gt(0)].nsmallest(5, 'monthly_income')[income_cols])


display(df_clients.nlargest(5, 'monthly_income')[income_cols])

Решение: 
- низкие положительные доходы сохраняем с income_low_flag;
- крупные доходы не ограничиваем сверху;
- ничем не заменяем ни медианой, ни умножением на 1000, так как ошибку не получается подтвердить.

#### 3.4.8. Низкие доходы по группам

In [ ]:
# профиль низких доходов по группам
low_income_profiles = {}

for group in ['income_source', 'segment']:
    profile = (
        df_clients.groupby(group, observed=True, dropna=False)
        .agg(
            n_clients=('client_id', 'size'),
            income_known=('monthly_income', 'count'),
            low_income=('income_low_flag', 'sum'),
        )
    )

    # Рассчитываем долю низких доходов среди известных, защищаясь от деления на ноль
    profile['low_among_known_pct'] = (
        profile['low_income']
        .div(profile['income_known'].replace(0, np.nan))
        .mul(100)
    )

    low_income_profiles[group] = profile

    
    display(profile.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

titles = {
    'income_source': 'По источнику дохода',
    'segment': 'По сегменту',
}

for ax, (group, profile) in zip(axes, low_income_profiles.items()):
    plot_data = (
        profile.loc[profile['income_known'].gt(0)]
        .sort_values('low_among_known_pct')
    )

    plot_data = profile.loc[profile['income_known'].gt(0)].sort_values('low_among_known_pct')
    labels = [f'{name if pd.notna(name) else "Пропуск"} (n={int(row["income_known"])})' for name, row in plot_data.iterrows()]
    bars = ax.barh(labels, plot_data['low_among_known_pct'])
    

    # Подписываем значения на столбцах и ограничиваем ось X для лучшей читаемости
    ax.bar_label(bars, labels=[f'{value:.1f}%' for value in plot_data['low_among_known_pct']], padding=4)
    ax.set(title=titles[group], xlabel='Доля низких доходов среди известных, %', ylabel='')
    ax.set_xlim(0, max(5, plot_data['low_among_known_pct'].max() * 1.3))
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

Вывод: закономерностей не найдено

#### 3.4.9. Форма распределения и Q–Q

In [ ]:
from scipy import stats

positive = income.loc[income.gt(0)].astype(float)

positive_without_low = positive.loc[~df_clients.loc[positive.index,'income_low_flag']]

shape_samples = {
    'Доход, все положительные': positive,
    'ln(дохода), все положительные': np.log(positive),
    'ln(дохода), без низких': np.log(positive_without_low),
}

shape_comparison = pd.DataFrame({
    name: {
        'Количество': len(values),
        'Асимметрия': values.skew(),
        'Избыточный эксцесс': values.kurt(),
    }
    for name, values in shape_samples.items()
}).T

display(shape_comparison.round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# Строим Q-Q графики через scipy.stats.probplot, выделяя цветом клиентов с низким доходом относительно опорной прямой
for ax, (name, values) in zip(axes, shape_samples.items()):
    values = values.dropna().sort_values()

    if len(values) < 4 or values.nunique() < 2:
        ax.set_title(name)
        ax.text(
            0.5, 0.5, 'Недостаточно данных',
            ha='center', va='center',
            transform=ax.transAxes,
        )
        continue

    # получаем квантили и параметры опорной прямой
    (theoretical, observed), (slope, intercept, _) = (
        stats.probplot(values.to_numpy(), dist='norm')
    )

    # выделяем низкие доходы
    low_flag = df_clients.loc[values.index, 'income_low_flag'].to_numpy(dtype=bool)

    for mask, color, label in [
        (~low_flag, 'steelblue', 'Доход от 10 000 ₽'), 
        (low_flag, 'crimson', 'Доход ниже 10 000 ₽')
    ]:
        if mask.any():
            ax.scatter(theoretical[mask], observed[mask], s=10, alpha=0.5, color=color, label=label)

    ax.plot(theoretical, slope * theoretical + intercept, color='darkorange', label='Опорная прямая')
    ax.set(title=f'{name}\nn = {len(values)}', xlabel='Теоретические квантили N(0, 1)', ylabel='Выборочные квантили')
    ax.legend(fontsize=7)
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

вывод:
изначально есть правосторонняя асимметрия. После логарифмирования асимметрия становится отрицательной из-за низких доходов, и после их отрезания видно более близкое положение точек к прямой(норм распред)

Общий вывод по доходам:
  - Значения -999 заменены на пропуски. 
  - Созданы столбцы с флагами 
    - income_missing — отсутствие дохода после обработки (10,87% клиентов)
    - income_low_flag — положительный доход ниже 10 000 (175 клиентов)
  - логарифмирование приближает распределения дохода к нормальному, без учёта низких доходов
Дальше:
 1) Клиенты с низким доходом не удалены, используем их для анализа, где доход не важен
 2) Результаты, зависящие от дохода, лучше сравнивать с вариантом с и без income_low_flag

#### 3.4.11. Сумма операции и статус

In [ ]:
amount = df_transactions['amount']

quality = pd.Series({
    'Всего операций': len(df_transactions),
    'Пропусков во всех полях': df_transactions.isna().sum().sum(),
    'Пропусков суммы': amount.isna().sum(),
    'Бесконечных сумм': np.isinf(
        amount.to_numpy(dtype=float, na_value=np.nan)
    ).sum(),
    'Нулевых сумм': amount.eq(0).sum(),
    'Отрицательных сумм': amount.lt(0).sum(),
    'Положительных сумм': amount.gt(0).sum(),
}, name='Количество')

display(quality.to_frame())

# Показываем примеры только при наличии проблемных сумм
amount_to_check = (
    amount.isna()
    | ~np.isfinite(amount)
    | amount.eq(0)
)

if amount_to_check.any():
    display(
        df_transactions.loc[
            amount_to_check,
            ['txn_id', 'amount', 'currency', 'status'],
        ].head(10)
    )

In [ ]:
status = df_transactions['status'].astype('string')

# Невалидные суммы выделяем отдельно, а не считаем положительными
valid_amount = amount.notna() & np.isfinite(amount)

sign = pd.Series(
    'Не определён',
    index=df_transactions.index,
    name='Знак суммы',
)

sign.loc[valid_amount & amount.lt(0)] = 'Отрицательная'
sign.loc[valid_amount & amount.eq(0)] = 'Нулевая'
sign.loc[valid_amount & amount.gt(0)] = 'Положительная'

sign_counts = pd.crosstab(
    status.fillna('Статус отсутствует'),
    sign,
).reindex(
    columns=[ 'Отрицательная', 'Нулевая', 'Положительная', 'Не определён'],
    fill_value=0,
)

# Цвет показывает долю внутри статуса, подпись — число строк
sign_pct = (
    sign_counts
    .div(sign_counts.sum(axis=1), axis=0)
    .mul(100)
)

fig, ax = plt.subplots(figsize=(9, 3.5))

sns.heatmap(
    sign_pct,
    annot=sign_counts,
    fmt=',d',
    cmap='Blues',
    vmin=0,
    vmax=100,
    linewidths=0.5,
    cbar_kws={'label': 'Доля внутри статуса, %'},
    ax=ax,
)

ax.set(
    title='Согласованность знака суммы и статуса',
    xlabel='Знак суммы',
    ylabel='Статус',
)
ax.tick_params(axis='both', labelrotation=0)

plt.tight_layout()
plt.show()

# явный контроль противоречий и запись решения
expected_statuses = {'успешно', 'отклонено', 'возврат'}
unknown_status = status.isna() | ~status.isin(expected_statuses)
refund = status.eq('возврат').fillna(False)
non_refund = status.isin(['успешно', 'отклонено'])
sign_conflict = valid_amount & (
    (refund & amount.ge(0)) | (non_refund & amount.le(0))
)

transactions_amount_issues = df_transactions.loc[
    amount_to_check | unknown_status | sign_conflict
].copy()

add_log(
    key='transactions_keep_refunds', field='amount; status',
    issue='Отрицательные суммы требуют содержательной интерпретации',
    scope='Конечные отрицательные суммы со статусом возврат',
    n_rows=(valid_amount & amount.lt(0) & refund).sum(),
    value_before='Отрицательная сумма у возврата',
    action='Оставлено без изменения знака',
    value_after=(
    f'Возвратов с отрицательной конечной суммой сохранено: {int((valid_amount & amount.lt(0) & refund).sum())}; противоречий знака и известного статуса: {int(sign_conflict.sum())}; неизвестных или отсутствующих статусов: {int(unknown_status.sum())}; сумм, требующих проверки: {int(amount_to_check.sum())}'
),
    rationale=('Отрицательный знак возврата сохранён; абсолютное значение не подставляется. Диагностика выполнена после обработки повторов, до последующих исключений операций по датам. Возможные нарушения отдельно сохранены в transactions_amount_issues; эта запись не означает их автоматического исправления.'),
)

вывод:
отрицательные суммы это не ошибки! Возвраты сохраняют со знаком минус, успешные и отклонённые операции имеют положительную сумму.
Решение: возвраты сохраняем со знаком минус.

### 3.5. Фильтры перед расчётом активности
#### 3.5.1. Период

In [ ]:
# Даты преобразованы при загрузке данных.
period_start = pd.Timestamp('2024-01-01')
period_end_exclusive = pd.Timestamp('2025-07-01')

dates = df_transactions['txn_datetime']

missing_date = dates.isna()
before_period = dates.lt(period_start)
after_period = dates.ge(period_end_exclusive)

in_period = (
    dates.notna()
    & dates.ge(period_start)
    & dates.lt(period_end_exclusive)
)

period_summary = pd.Series({
    'Всего операций': len(df_transactions),
    'Без даты': missing_date.sum(),
    'Раньше 01.01.2024': before_period.sum(),
    'Начиная с 01.07.2025': after_period.sum(),
    'В исследуемом периоде': in_period.sum(),
}, name='Количество')

display(period_summary.to_frame())




# Отдельные выборки: исходную таблицу не перезаписываем.
transactions_period = df_transactions.loc[in_period].copy()

transactions_outside_period = df_transactions.loc[
    before_period | after_period
].copy()

transactions_without_date = df_transactions.loc[missing_date].copy()

if not transactions_outside_period.empty:
    
    display(
        transactions_outside_period[
            ['txn_id', 'client_id', 'txn_datetime',
             'amount', 'currency', 'status']
        ].sort_values('txn_datetime').head(10)
    )



add_log(
    key='transactions_period_selection', field='txn_datetime',
    issue='Часть операций не относится к исследуемому периоду',
    scope='Все операции после обработки повторов',
    n_rows=len(transactions_outside_period) + len(transactions_without_date),
    value_before=f'{len(df_transactions)} операций всех дат',
    action='Создана отдельная выборка периода; исходная очищенная таблица сохранена',
    value_after=f'{len(transactions_period)} в периоде; {len(transactions_outside_period)} вне периода; {len(transactions_without_date)} без даты',
    rationale='Границы [2024-01-01, 2025-07-01); внепериодные и недатированные записи сохранены отдельно.',
)

Вывод:
есть 3 операции предшествующая исследуемому периоду и 78 операций после исследуемого периода. Нельзя использовать операции не из исследуемого периода.
Решение: внепериодные операции сохраняем в transactions_outside_period,
но не используем для исследования заданного периода

#### 3.5.2. Даты и наличие анкеты

In [ ]:
client_date_cols = ['registration_date', 'card_open_date']

# Присоединяем только поля, необходимые для проверки
transactions_dates_check = transactions_period.merge(
    df_clients[['client_id', *client_date_cols]],
    on='client_id',
    how='left',
    validate='many_to_one',
    indicator='client_match',
)

# Определяем наличие клиентской анкеты
matched = transactions_dates_check['client_match'].eq('both')
no_client = ~matched

# Сравниваем календарные дни, не учитывая время внутри дня
txn_day = transactions_dates_check['txn_datetime'].dt.normalize()

registration_day = (
    transactions_dates_check['registration_date'].dt.normalize()
)

card_open_day = (
    transactions_dates_check['card_open_date'].dt.normalize()
)

# Проверка доступна, если найдена анкета и заполнены обе даты
date_checkable_registration = (
    matched
    & txn_day.notna()
    & registration_day.notna()
)

date_checkable_card = (
    matched
    & txn_day.notna()
    & card_open_day.notna()
)

# Отмечаем операции раньше даты регистрации
transactions_dates_check['before_registration'] = (
    date_checkable_registration
    & txn_day.lt(registration_day)
)

# Отмечаем операции раньше даты открытия карты
transactions_dates_check['before_card_open'] = (
    date_checkable_card
    & txn_day.lt(card_open_day)
)

# Сохраняем операции без найденной клиентской анкеты
transactions_without_client = transactions_dates_check.loc[
    no_client
].copy()

# Сохраняем операции хотя бы с одним временным противоречием
date_conflicts = transactions_dates_check.loc[
    transactions_dates_check[
        ['before_registration', 'before_card_open']
    ].any(axis=1)
].copy()

# Фиксируем диагностику после создания всех признаков и выборок
add_log(
    key='transactions_date_diagnostic_flags',
    field='before_registration; before_card_open',
    issue=('Нужно проверить согласованность дат операций и клиентских событий'),
    scope=('Операции периода в диагностической таблице transactions_dates_check'),
    n_rows=len(transactions_dates_check),
    value_before='Диагностические признаки отсутствовали',
    action=('Созданы признаки проверки дат и сохранены временные противоречия'),
    value_after=(f'До регистрации: {int(transactions_dates_check["before_registration"].sum())}; до открытия карты: {int(transactions_dates_check["before_card_open"].sum())}; операций хотя бы с одним нарушением: {len(date_conflicts)}; доступна проверка регистрации: {int(date_checkable_registration.sum())}; доступна проверка открытия карты: {int(date_checkable_card.sum())}; без анкеты: {int(no_client.sum())}'),
    rationale=('Сравниваются календарные дни. Признаки созданы только в диагностической таблице; рабочие операции не изменены. False не доказывает согласованность, если нет анкеты или необходимой даты. Результат относится к этапу до последующих исключений.'),
)

# Сводка результатов диагностики
date_summary = pd.DataFrame({
    'Количество': [
        len(transactions_dates_check),
        no_client.sum(),
        transactions_dates_check.loc[
            no_client, 'client_id'
        ].nunique(),
        (matched & registration_day.isna()).sum(),
        (matched & card_open_day.isna()).sum(),
        date_checkable_registration.sum(),
        date_checkable_card.sum(),
        transactions_dates_check['before_registration'].sum(),
        transactions_dates_check['before_card_open'].sum(),
        len(date_conflicts),
    ],
}, index=[
    'Всего операций в диагностической таблице',
    'Операций без клиента в очищенном справочнике',
    'Несопоставленных заполненных client_id',
    'Сопоставленных операций без даты регистрации',
    'Сопоставленных операций без даты открытия карты',
    'Доступна проверка относительно регистрации',
    'Доступна проверка относительно открытия карты',
    'Операций до регистрации',
    'Операций до открытия карты',
    'Операций хотя бы с одним временным противоречием',
])

# Для операционных показателей считаем долю от всех строк таблицы
total_ops = len(transactions_dates_check)

date_summary['Доля от всех операций, %'] = (
    date_summary['Количество']
    .div(total_ops if total_ops else np.nan)
    .mul(100)
    .round(4)
)

# Число уникальных клиентов не делим на количество операций
date_summary.loc[
    'Несопоставленных заполненных client_id',
    'Доля от всех операций, %',
] = np.nan

display(date_summary)

# Показываем примеры операций без анкеты и их распределение по MCC
if not transactions_without_client.empty:
    

    display(
        transactions_without_client[
            ['txn_id','client_id','txn_datetime','status','channel','mcc_code',]
        ].head(10)
    )

    

    mcc_summary = (
        transactions_without_client['mcc_code']
        .value_counts(dropna=False)
        .to_frame('Количество операций')
    )

    mcc_summary['Доля, %'] = (
        mcc_summary['Количество операций']
        .div(len(transactions_without_client))
        .mul(100)
        .round(2)
    )

    display(mcc_summary)

# Показываем примеры обнаруженных временных противоречий
if not date_conflicts.empty:
    

    display(
        date_conflicts[
            ['txn_id','client_id','txn_datetime','registration_date','card_open_date','channel','status','category','before_registration','before_card_open']
        ]
        .sort_values(['client_id', 'txn_datetime'])
        .head(10)
    )

вывод:
- для 26 583 операций — 4,23% выборки — не найдены анкеты
в очищенном клиентском справочнике.
- найдена 1 операция до открытия карты

Решение: 
- операции без клиентской анкеты не удаляем, а сохраняем в transactions_without_client и продолжаем исследование в 2.7. Отсутсвие клиента не говорит, что операция неверная
- операция до открытия карты не может быть выполнена, поэтому удаляем её

#### 3.5.3. Решение по операции до открытия карты

In [ ]:
# Выбираем карточные операции до указанной даты открытия карты
mask_to_drop = (
    date_conflicts['before_card_open']
    & date_conflicts['channel'].eq('Карта').fillna(False)
)

# Сохраняем записи с клиентскими датами и результатами проверки
rows_to_drop = date_conflicts.loc[mask_to_drop].copy()
txn_ids_to_drop = rows_to_drop['txn_id'].unique()

# Считаем фактические строки для удаления в каждой таблице
remove_from_main = df_transactions['txn_id'].isin(txn_ids_to_drop)
remove_from_period = transactions_period['txn_id'].isin(txn_ids_to_drop)

n_main_before = len(df_transactions)
n_period_before = len(transactions_period)

n_removed_main = int(remove_from_main.sum())
n_removed_period = int(remove_from_period.sum())

# Удаляем из общей рабочей таблицы и выборки периода
df_transactions = df_transactions.loc[
    ~remove_from_main
].copy()

transactions_period = transactions_period.loc[
    ~remove_from_period
].copy()

# Заменяет прежнюю запись об исключении только из выборки периода
add_log(
    key='transactions_before_card_exclusion',
    field='txn_datetime; card_open_date; channel',
    issue='Карточная операция раньше указанной даты открытия карты',
    scope='before_card_open == True и channel == Карта',
    n_rows=n_removed_main,
    value_before=(f'df_transactions: {n_main_before} строк; transactions_period: {n_period_before} строк'
    ),
    action='Удалено из рабочей таблицы и выборки периода',
    value_after=(f'df_transactions: {len(df_transactions)} строк; transactions_period: {len(transactions_period)} строк; в rows_to_drop сохранено {len(rows_to_drop)} записей'),
rationale=('Применено принятое в анализе правило: карточные операции раньше указанной даты открытия карты исключаются. Правило предполагает, что card_open_date относится к карте, использованной в операции. Какое именно исходное поле ошибочно, не установлено. Удалённые записи сохранены в rows_to_drop; исходный CSV не изменён. n_rows — число удалённых строк общей рабочей таблицы; уникальность txn_id проверена. Удаление тех же операций из периодной копии повторно не суммируется.'),
)

if not rows_to_drop.empty:
    display(rows_to_drop[['txn_id', 'client_id', 'txn_datetime','registration_date','card_open_date','channel', 'status', 'category',]])

вывод:
удалена одна карточная операция, выполненная раньше указанной даты открытия карты. Копия сохранена в rows_to_drop

#### 3.5.4. Полнота связи после фильтров

In [ ]:
# Создаём свежую диагностическую выборку
link_analysis = transactions_period.copy()

link_analysis['client_missing'] = (
    ~link_analysis['client_id'].isin(df_clients['client_id'])
)

# Фиксируем создание диагностического флага
add_log(
    key='transactions_diagnostic_client_missing',
    field='client_missing',
    issue='Нужно выделить операции без клиентской анкеты',
    scope='Все операции диагностической таблицы link_analysis',
    n_rows=len(link_analysis),
    value_before='Диагностический флаг отсутствовал',
    action='Создан флаг в отдельной таблице',
    value_after=(f'True у {int(link_analysis["client_missing"].sum())} операций'),
    rationale=('client_missing показывает отсутствие client_id в очищенном клиентском справочнике. Рабочие таблицы не изменены.'),
)

unknown_transactions = link_analysis.loc[
    link_analysis['client_missing']
].copy()

overall_rate = (link_analysis['client_missing'].sum() / len(link_analysis) * 100)

month = link_analysis['txn_datetime'].dt.to_period('M')

monthly_link = (
    link_analysis
    .groupby(month)['client_missing']
    .agg(total='size', missing='sum')
    .reindex(pd.period_range('2024-01', '2025-06', freq='M'))
)

monthly_link[['total', 'missing']] = (
    monthly_link[['total', 'missing']].fillna(0).astype(int)
)

monthly_link['missing_pct'] = (
    monthly_link['missing']
    .div(monthly_link['total'].replace(0, float('nan')))
    .mul(100)
)

fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(
    monthly_link.index.astype(str),
    monthly_link['missing_pct'],
    marker='o',
    color='steelblue',
)

ax.axhline(
    overall_rate,
    color='darkorange',
    linestyle='--',
    label=f'Весь период: {overall_rate:.2f}%',
)

ax.set(
    title='Доля операций без сопоставленного клиента по месяцам',
    xlabel='Месяц',
    ylabel='Доля операций, %',
)

ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.2)
ax.legend()

sns.despine()
plt.tight_layout()
plt.show()

display(monthly_link.round(2))

вывод:
доля операций без клиентской анкеты снижается. Причина отсутствия анкет по этим данным не установлена.


#### 3.5.5. Разделение по наличию анкеты

In [ ]:
# Разделяем операции по наличию клиентской анкеты
client_found = transactions_period['client_id'].isin(
    df_clients['client_id']
)

transactions_linked = transactions_period.loc[
    client_found
].copy()

transactions_unlinked = transactions_period.loc[
    ~client_found
].copy()


# Отдельно считаем операции и разные client_id
link_summary = pd.DataFrame({
    'Операций': [
        len(transactions_period),
        len(transactions_linked),
        len(transactions_unlinked),
    ],
    'Уникальных заполненных client_id': [
        transactions_period['client_id'].nunique(),
        transactions_linked['client_id'].nunique(),
        transactions_unlinked['client_id'].nunique(),
    ],
}, index=[
    'Все операции периода',
    'Анкета найдена',
    'Анкета не найдена',
])

link_summary['Доля операций, %'] = (
    link_summary['Операций']
    .div(len(transactions_period) if len(transactions_period) else np.nan)
    .mul(100)
)

display(link_summary.round(2))

# количество операций на каждый отсутствующий в справочнике ID
unlinked_clients_summary = (
    transactions_unlinked
    .groupby('client_id', dropna=False)
    .agg(
        n_operations=('txn_id', 'size'),
        first_operation=('txn_datetime', 'min'),
        last_operation=('txn_datetime', 'max'),
    )
    .sort_values(
        ['n_operations', 'first_operation'],
        ascending=[False, True],
    )
    .rename(columns={
        'n_operations': 'Количество операций',
        'first_operation': 'Первая операция',
        'last_operation': 'Последняя операция',
    })
)







# Фиксируем разделение по доступности клиентских данных
add_log(
    key='transactions_client_link_selection',
    field='client_id',
    issue='Для части операций не найдена клиентская анкета',
    scope='Операции периода после проверки дат',
    n_rows=len(transactions_unlinked),
    value_before=f'{len(transactions_period)} операций периода',
    action='Созданы отдельные выборки; операции не удалены',
    value_after=(f'{len(transactions_linked)} операций с анкетой; {len(transactions_unlinked)} без анкеты; {transactions_unlinked["client_id"].nunique()} несопоставленных заполненных client_id'),
    rationale=('Все операции сохраняются для общего анализа. Для связей с клиентскими характеристиками используется transactions_linked.'),
)

вывод:
для 26'583 (4.23%) оперций не найдены 353 клиента

Решение:
 - не удаляем из transactions_period
 - для анализа клиентских характеристик исспользуем transactions_linked.


#### 3.5.6. Выборка успешных оплат


In [ ]:
# Выбираем успешные операции с положительной конечной суммой
successful = transactions_period.loc[
    transactions_period['status']
    .astype('string')
    .eq('успешно')
    .fillna(False)
    & transactions_period['amount'].gt(0)
    & np.isfinite(transactions_period['amount'])
].copy()

# список валют больше не зависит от amount_stats
currencies = [
    currency
    for currency in ['RUB', 'USD', 'EUR']
    if successful['currency'].eq(currency).any()
]

add_log(
    key='transactions_successful_selection',
    field='status; amount',
    issue='Для распределения покупок нужна отдельная выборка',
    scope='Операции исследуемого периода после очистки',
    n_rows=len(transactions_period) - len(successful),
    value_before=f'{len(transactions_period)} операций всех статусов',
    action='Создана отдельная выборка successful',
    value_after=f'{len(successful)} успешных операций для анализа сумм',
    rationale=('Выбраны положительные конечные суммы со статусом успешно. Возвраты и отклонённые попытки остаются в transactions_period. Валюты анализируются отдельно.'),
)

#### 3.5.7. Согласованность MCC и категории

In [ ]:
# Создаём отдельную таблицу для диагностики
category_check = transactions_period.copy()

# Приводим поля к строкам для проверки заполненности
mcc_text = (
    category_check['mcc_code']
    .astype('string')
    .str.strip()
)

category_text = (
    category_check['category']
    .astype('string')
    .str.strip()
)

valid_pair = (
    mcc_text.notna()
    & mcc_text.ne('').fillna(False)
    & category_text.notna()
    & category_text.ne('').fillna(False)
)



# Сохраняем неполные пары отдельно, не удаляя их из рабочей таблицы
category_missing_pairs = category_check.loc[~valid_pair].copy()

category_check = category_check.loc[valid_pair].copy()
category_check['mcc_code'] = mcc_text.loc[valid_pair]
category_check['category'] = category_text.loc[valid_pair]

# Считаем частоты сочетаний
counts = pd.crosstab(
    category_check['mcc_code'],
    category_check['category'],
)

# Основную категорию назначаем только при единственном максимуме
max_count = counts.max(axis=1)
unique_leader = counts.eq(max_count, axis=0).sum(axis=1).eq(1)
main_category = counts.idxmax(axis=1).where(unique_leader)

mcc_category_summary = pd.DataFrame({
    'Основная категория': main_category,
    'Всего операций': counts.sum(axis=1),
    'Операций в основной категории': max_count.where(unique_leader),
})

mcc_category_summary['Доля основной категории, %'] = (
    mcc_category_summary['Операций в основной категории']
    .div(mcc_category_summary['Всего операций'])
    .mul(100)
)

mcc_category_summary['Других операций'] = (
    mcc_category_summary['Всего операций']
    - mcc_category_summary['Операций в основной категории']
)

mcc_category_summary['Доля других операций, %'] = (
    mcc_category_summary['Других операций']
    .div(mcc_category_summary['Всего операций'])
    .mul(100)
)

display(mcc_category_summary.round(2))

# Сохраняем ожидаемое по частоте соответствие только для диагностики
category_check['category_by_mcc_mode'] = (
    category_check['mcc_code'].map(main_category)
)

assessed = category_check['category_by_mcc_mode'].notna()

# При неопределённой основной категории оставляем NA,
# а не записываем False — такую операцию мы не оценили
category_check['category_differs'] = (
    category_check['category']
    .ne(category_check['category_by_mcc_mode'])
    .astype('boolean')
    .where(assessed)
)

category_mismatches = category_check.loc[
    category_check['category_differs'].fillna(False)
].copy()






add_log(
    key='transactions_mcc_category_diagnostics',
    field='mcc_code; category; category_by_mcc_mode; category_differs',
    issue='Категория операции может отличаться от наиболее частой категории MCC',
    scope='Диагностика операций периода с заполненной парой MCC–категория',
    n_rows=len(category_check),
    value_before='Диагностическое соответствие MCC–категория отсутствовало',
    action=('Созданы диагностические признаки; исходные категории сохранены'
    ),
    value_after=(f'Оценено операций: {int(assessed.sum())}; расхождений с модальной категорией: {len(category_mismatches)}; без однозначной модальной категории: {int((~assessed).sum())}; без заполненной пары: {len(category_missing_pairs)}'),
    rationale=('Основная категория определена по единственному максимуму частоты внутри MCC в исследуемой выборке. Это эмпирический ориентир, а не внешний справочник. Расхождение не доказывает ошибку, поэтому category не исправляется и операции не удаляются. При отсутствии единственного максимума category_differs оставлен неопределённым. Неполные пары сохранены отдельно.'),
)


if not category_mismatches.empty:
    

    display(
        category_mismatches[
            ['txn_id', 'client_id', 'txn_datetime','mcc_code', 'category', 'category_by_mcc_mode','channel', 'status',]
        ].head(10)
    )

вывод:
у каждого из 9 МСС имеется одна преобладающая категория, её доля порядка 99%, и 1673 операции с другими категориями. 
Решение: сохраняем расхождения в category_mismatches.

#### 3.5.8. Воронка операций

In [ ]:
# Показываем последовательность и разные ветви отбора.
transaction_funnel = pd.DataFrame({'Этап':[
    'Загружено','После полных повторов','После временных версий',
    'Период до исключения карточной операции','Период после исключения',
    'С анкетой: отдельная ветвь','Успешные: отдельная ветвь'],
    'Строк':[n_transactions_loaded,n_transactions_loaded-len(transaction_duplicates),
             n_main_before,n_period_before,len(transactions_period),len(transactions_linked),len(successful)]})
display(transaction_funnel)



## 4. Журнал изменений
Записи добавляются рядом с действиями; здесь показываем накопленное представление. Не инициализируем пустой DataFrame поверх словаря. Итоговый CSV сохраняется после всех глав в разделе 9.


In [ ]:
cleaning_log = pd.DataFrame(list(cleaning_log_entries.values()),columns=LOG_COLUMNS)
display(cleaning_log)cleaning_log 

## 5. Разведочный анализ
Графиков столько, сколько нужно для исследования; в отчёт войдут не все.

### 5.1. Распределения количественных признаков
Возраст и доход уже исследованы в 3.4 для обоснования решений. Не запускаем их очистку повторно. Ниже сохранены распределения сумм операций; валюты рассматриваются раздельно.


In [ ]:
# Для сравнения покупок не включаем возвраты и отклонённые попытки.
plot_data = df_transactions.loc[
    df_transactions['status'].astype('string')
    .str.strip().str.casefold().eq('успешно').fillna(False)
    & df_transactions['amount'].gt(0),
    ['currency', 'category', 'amount'],
].copy()

# Объединяем варианты написания категорий в выборке для графиков.
plot_data['category'] = (
    plot_data['category'].astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.capitalize()
    .replace({
        'Кафе/рестораны': 'Кафе и рестораны',
        'Азс': 'АЗС',
        'Связь и интернет': 'Связь и Интернет',
    })
    .fillna('Категория не указана')
)

for currency in ['RUB', 'USD', 'EUR']:
    data = plot_data.loc[plot_data['currency'].eq(currency)]

    if data.empty:
        continue

    # Сортируем категории по типичному размеру покупки.
    category_stats = (
        data.groupby('category', observed=True)['amount']
        .agg(n='size', median='median')
        .sort_values('median')
    )

    fig, ax = plt.subplots(figsize=(11, 6))

    sns.boxplot(
        data=data,
        x='amount',
        y='category',
        order=category_stats.index.tolist(),
        color='steelblue',
        whis=1.5,
        flierprops={
            'marker': 'o',
            'markersize': 3,
            'alpha': 0.35,
            'markerfacecolor': 'none',
        },
        ax=ax,
    )

    # Шкалу меняем после построения:
    # квартильные границы и усы рассчитаны по исходным суммам.
    ax.set_xscale('log')

    ax.set_yticks(
        range(len(category_stats)),
        labels=[
            f'{category} (n={int(n):,})'.replace(',', ' ')
            for category, n in category_stats['n'].items()
        ],
    )

    ax.set(
        title=f'Суммы успешных операций по категориям — {currency}',
        xlabel=f'Сумма, {currency} — логарифмическая шкала',
        ylabel='',
    )

    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)

    plt.tight_layout()
    plt.show()

In [ ]:
from scipy import stats

if currencies:
    fig, axes = plt.subplots(len(currencies), 3, figsize=(16, 3.8 * len(currencies)), squeeze=False)
    shape_rows = []
    for row, currency in enumerate(currencies):
        values = successful.loc[successful['currency'].eq(currency), 'amount']
        p99 = values.quantile(0.99)
        visible = values.loc[values.le(p99)]
        log_values = np.log(values)
        ax_main, ax_log, ax_qq = axes[row]
        # Основная масса сумм без верхнего хвоста — только для графика
        sns.histplot(x=visible, bins=45, ax=ax_main, color='steelblue')
        # Среднее и медиана 
        ax_main.axvline(values.mean(),color='crimson',linestyle='-.',linewidth=1.5,label=f'Среднее: {values.mean():,.2f}')
        ax_main.axvline(values.median(),color='darkorange',linestyle='--',linewidth=1.5,label=f'Медиана: {values.median():,.2f}')
        ax_main.legend(fontsize=8)
        ax_main.set(title=f'{currency}: до P99; скрыто {int(values.gt(p99).sum())}',
                    xlabel=f'Сумма, {currency}', ylabel='Количество')
        
         # Все суммы в логарифмической шкале
        sns.histplot(x=log_values, bins=45, ax=ax_log, color='steelblue')
        # Среднее и медиана 
        ax_log.axvline(log_values.mean(),color='crimson',linestyle='-.',linewidth=1.5,label=f'Среднее: {log_values.mean():,.2f}')
        ax_log.axvline(log_values.median(),color='darkorange',linestyle='--',linewidth=1.5,label=f'Медиана: {log_values.median():,.2f}')
        ax_log.legend(fontsize=8)
        ax_log.set(title=f'{currency}: все ln(суммы), n={len(values)}',
                   xlabel='ln(суммы)', ylabel='Количество')
        
        # Сравнение логарифмов сумм с нормальным распределением
        if len(values) >= 3 and values.nunique() > 1:
            (theoretical, observed), (slope, intercept, _) = stats.probplot(log_values, dist='norm')
            ax_qq.scatter(theoretical, observed, s=7, alpha=0.35)
            ax_qq.plot(theoretical, slope * theoretical + intercept, color='darkorange')
        ax_qq.set(title=f'{currency}: Q–Q ln(суммы)', xlabel='Квантили N(0, 1)', ylabel='Квантили ln(суммы)')
        shape_rows.append({'Валюта': currency, 'Асимметрия суммы': values.skew(),
                           'Асимметрия ln': log_values.skew(), 'Эксцесс ln': log_values.kurt()})
        for ax in axes[row]:
            sns.despine(ax=ax)
    plt.tight_layout()
    
    shape_summary = (
        pd.DataFrame(shape_rows)
        .set_index('Валюта')
    )
    
    plt.show()
    display(pd.DataFrame(shape_rows).set_index('Валюта').round(3))

вывод:
изначально распределение далеко от нормально из-за хвоста из очень больших сумм, однако после логарифмирования сумм распределение приближется к нормальному и достаточно хорошо ложится Q-Q диаграмме на квантиль нормального распределения

### 5.2. Динамика по месяцам

#### 5.2.1. Подготовка показателей для графиков
Расчёт метрик нужен до графиков. Сохраняем исходный вычислительный блок здесь, а в разделе 6 — их полное описание и ограничения. Повторно метрики в разделе 6 не считаем.


In [ ]:
BASE_START = pd.Timestamp('2024-01-01')
CURRENT_START = pd.Timestamp('2025-01-01')
ACTIVITY_END = pd.Timestamp('2025-07-01')
ACTIVITY_MONTHS = pd.period_range('2024-01', '2025-06', freq='M')

period_ok = (
    transactions_period['txn_datetime'].ge(BASE_START)
    & transactions_period['txn_datetime'].lt(ACTIVITY_END)
)

# Справочник сохраняет все исходные клиентские признаки и флаги
activity_clients = df_clients.copy()
status_activity = transactions_period['status'].astype('string')
payment_mask = (
    status_activity.eq('успешно').fillna(False)
    & transactions_period['amount'].gt(0)
    & np.isfinite(transactions_period['amount'])
)
activity_payments_all = transactions_period.loc[payment_mask].copy()
linked_mask = activity_payments_all['client_id'].isin(activity_clients['client_id'])
activity_payments = activity_payments_all.loc[linked_mask].copy()
# Несопоставленные оплаты сохраняем 
activity_payments_unlinked = activity_payments_all.loc[~linked_mask].copy()
activity_payments['month'] = activity_payments['txn_datetime'].dt.to_period('M')

display(pd.Series({
    'Операций периода всех статусов': len(transactions_period),
    'Успешных оплат': len(activity_payments_all),
    'Успешных оплат с анкетой': len(activity_payments),
    'Успешных оплат без анкеты': len(activity_payments_unlinked),
}, name='Количество').to_frame())

Решение: для анализа клиентов будем использовать только оплаты с найденным клиентов в clients. Несопоставленные оплаты сохраняем в activity_payments_unlinked.

#### 5.2.1.1 Доступность истории
Берём клиентов зарегистрированных в банке в исследуемый период

In [ ]:
registration = activity_clients['registration_date']
activity_clients['activity_history_group'] = np.select(
    [registration.isna(), registration.ge(ACTIVITY_END), registration.gt(BASE_START)],
    ['Дата регистрации неизвестна', 'Регистрация после периода', 'Нет полной базовой истории'],
    default='Полная история для сравнения',
)
activity_clients['activity_comparable'] = registration.notna() & registration.le(BASE_START)

history_summary = activity_clients['activity_history_group'].value_counts().to_frame('Клиентов')
history_summary['Доля базы, %'] = history_summary['Клиентов'].div(len(activity_clients)).mul(100)
display(history_summary.round(2))

# Не смешиваем реальные нули с отсутствующей историей
registration_by_id = activity_clients.set_index('client_id')['registration_date']
payment_registration = activity_payments['client_id'].map(registration_by_id)
activity_pre_registration = activity_payments.loc[
    payment_registration.notna()
    & activity_payments['txn_datetime'].dt.normalize().lt(payment_registration.dt.normalize())
].copy()


вывод:
для сравнения и анлиза используем 5'542 клиента

#### 5.2.1.2 Помесячная таблица
Соберём таблицу для каждого клиента по каждому месяцу с количеством операций с учётом даты регистрации

In [ ]:
# Подсчёт оплат для каждой пары: клиент-месяц
monthly_counts = (
    activity_payments.groupby(['client_id', 'month'], observed=True)
    .size().rename('successful_count')
)
monthly_index = pd.MultiIndex.from_product(
    [activity_clients['client_id'], ACTIVITY_MONTHS], names=['client_id', 'month']
)
# Заполнение пропусков нулями
client_month_activity = monthly_counts.reindex(monthly_index, fill_value=0).reset_index()
# Учитываем дату регистрации
client_month_activity['registration_date'] = client_month_activity['client_id'].map(registration_by_id)
month_start = client_month_activity['month'].dt.to_timestamp()

client_month_activity['month_available'] = (
    client_month_activity['registration_date'].notna()
    & client_month_activity['registration_date'].le(month_start)
)
# successful_count_observed — найденные операции с учётом доступности месяца(регистрации).
client_month_activity['successful_count_observed'] = (
    client_month_activity['successful_count'].astype('Int64')
    .where(client_month_activity['month_available'])
)

display(client_month_activity.head(5))

#### 5.2.1.3 Активность 
Будем считать период '2024-01'-'2024-12' - базовым(base), а '2025-01':'2025-06' - текущим(current). Для анализа динамики.
активность = число оплат/число месяцев

In [ ]:
# Одна строка = один клиент; каждый месяц - столбец
counts_wide = (
    client_month_activity.pivot(index='client_id', columns='month', values='successful_count')
    .reindex(columns=ACTIVITY_MONTHS, fill_value=0)
)
base_counts = counts_wide.loc[:, '2024-01':'2024-12']
current_counts = counts_wide.loc[:, '2025-01':'2025-06']
metrics = pd.DataFrame(index=counts_wide.index)
# Счётчик оплат
metrics['payments_base'] = base_counts.sum(axis=1)
metrics['payments_current'] = current_counts.sum(axis=1)
# Счётчик активности
metrics['active_months_base'] = base_counts.gt(0).sum(axis=1)
metrics['active_months_current'] = current_counts.gt(0).sum(axis=1)
metrics['activity_last3'] = counts_wide.loc[:, '2025-04':'2025-06'].sum(axis=1).div(3)
metrics['no_payments_last3'] = counts_wide.loc[:, '2025-04':'2025-06'].sum(axis=1).eq(0)

# Объединим метрику и клиентов
client_activity = activity_clients.merge(metrics.reset_index(), on='client_id', how='left', validate='one_to_one')
# Средняя активность за месяц
comparable = client_activity['activity_comparable']
client_activity['activity_base'] = client_activity['payments_base'].div(12).where(comparable)
client_activity['activity_current'] = client_activity['payments_current'].div(6).where(comparable)
client_activity['activity_last3'] = client_activity['activity_last3'].where(comparable)
# Оценим абсолютное и относительное изменение
client_activity['activity_change'] = client_activity['activity_current'] - client_activity['activity_base']
positive_base = comparable & client_activity['activity_base'].gt(0)
client_activity['activity_change_pct'] = (
    client_activity['activity_current'].div(client_activity['activity_base'].where(positive_base))
    .sub(1).mul(100)
)

# Сравниваем целые числа без ошибок округления:
# current / 6 < base / 12 эквивалентно 2 * current < base.
base = client_activity['payments_base']
current = client_activity['payments_current']
client_activity['activity_state'] = np.select(
    [~comparable, base.eq(0) & current.eq(0), base.eq(0) & current.gt(0),
     base.gt(0) & current.eq(0), current.mul(2).lt(base), current.mul(2).eq(base)],
    ['Недостаточно сопоставимой истории', 'Нет оплат в обоих периодах',
     'Появилась активность', 'Прекратились оплаты', 'Снижение', 'Без изменения'],
    default='Рост',
)
# Флаг снижения
client_activity['activity_declined'] = ( 
    comparable & base.gt(0) & current.mul(2).lt(base)
)
display(client_activity[[
    'client_id', 'activity_history_group', 'payments_base', 'payments_current',
    'activity_base', 'activity_current', 'activity_change', 'activity_change_pct',
    'activity_state',
]].head(5))


#### 5.2.1.4 Расчёт и результаты


In [ ]:
# Сверка счётчиков без изменения данных
count_check = pd.DataFrame({
    'По операциям': [
        len(activity_payments),
        int(activity_payments['txn_datetime'].lt(CURRENT_START).sum()),
        int(activity_payments['txn_datetime'].ge(CURRENT_START).sum()),
    ],
    'По клиентским метрикам': [
        int((client_activity['payments_base'] + client_activity['payments_current']).sum()),
        int(client_activity['payments_base'].sum()),
        int(client_activity['payments_current'].sum()),
    ],
}, index=['Весь период', '2024 год', 'Январь–июнь 2025'])
count_check['Совпадает'] = count_check['По операциям'].eq(count_check['По клиентским метрикам'])
display(count_check)


state_summary = client_activity['activity_state'].value_counts().to_frame('Клиентов')
state_summary['Доля всей базы, %'] = state_summary['Клиентов'].div(len(client_activity)).mul(100)
display(state_summary.round(2))

comparison_sample = client_activity.loc[client_activity['activity_comparable']].copy()
display(comparison_sample[[
    'activity_base', 'activity_current', 'activity_change', 'activity_change_pct'
]].describe().round(2))

# Распределение метрик
eligible_base = comparison_sample['payments_base'].gt(0)
n_base = int(eligible_base.sum())
n_declined = int(comparison_sample['activity_declined'].sum())
n_stopped = int(comparison_sample['activity_state'].eq('Прекратились оплаты').sum())

вывод:
Сопоставимая история доступна у 5'542 клиентов. Из них 5'240 имели успешные оплаты в базовом периоде. 
У 5 клиентов с положительной базой нет успешных оплат за январь–июнь 2025 года. нижение явно перевешивает рост 67.84% к 30.59%

#### 5.2.1.5 Проверка формы (диаграмма рассеяния)

In [ ]:
comparison_ids = comparison_sample['client_id']
if len(comparison_ids):
    fixed_monthly = counts_wide.loc[comparison_ids].mean(axis=0)
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    x_months = fixed_monthly.index.to_timestamp()
    axes[0].plot(x_months, fixed_monthly.values, marker='o')
    axes[0].axvline(CURRENT_START, color='gray', linestyle='--', label='Начало оцениваемого периода')
    axes[0].set(title=f'Фиксированная группа: n = {len(comparison_ids):,}',
                xlabel='Месяц', ylabel='Успешных оплат на клиента за месяц')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].legend(fontsize=8)
    axes[1].scatter(comparison_sample['activity_base'], comparison_sample['activity_current'],
                    s=12, alpha=0.3)
    upper = max(comparison_sample['activity_base'].max(), comparison_sample['activity_current'].max(), 1)
    axes[1].plot([0, upper], [0, upper], color='darkorange', linestyle='--', label='Без изменения')
    axes[1].set(title='Частота оплат: базовый и текущий периоды',
                xlabel='2024: оплат в месяц', ylabel='Январь–июнь 2025: оплат в месяц')
    axes[1].legend()
    for ax in axes:
        sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()
else:
    pass


вывод:
помесячная линия показывает изменение частоты в фиксированной группе. На диаграмме сравнения точки ниже диагонали соответствуют снижению, выше — росту. 
- По форме линии можно предположить сезонность, так как оценивая форму '2024-01'-'2024-05' и '2025-01'-'2025-05' видно сходство, но слишком мало данных(период) для подтверждения этой гипотезы.
- По диаграмме рассеяния видно, есть и снижение и рост, основная часть клиентов сосредоточена в области небольших частот. 

In [ ]:

    add_log(
        key='clients_activity_metrics',
        field='activity_base; activity_current; activity_change; activity_change_pct; activity_state',
        issue='Нужно определить и сопоставить месячную платёжную активность',
        scope='Все клиенты; основное сравнение для регистрации не позднее 01.01.2024',
        n_rows=len(client_activity),
        value_before='Клиентские метрики активности отсутствовали',
        action='Созданы client_activity и клиентско-месячная таблица; исходные данные сохранены',
        value_after=(f'{len(client_activity)} клиентов; {len(comparison_sample)} сопоставимы; '
                     f'{len(client_month_activity)} клиентско-месячных строк'),
        rationale=('Успешные положительные конечные оплаты по всем каналам и валютам. '
                   'Для полной истории количество делится на 12 и 6 месяцев; '
                   'месяцы без оплат учитываются нулями. При нулевой базе процент не определяется. '
                   'Недостаточная история не является основанием удаления клиента.'),
    )


вывод:
Определена основная метрика активности: среднее количество оплат за месяц.
Снижение активности преобладает над ростом

Преимущества метрики удобно сравнивать разные периоды времени и легко разделяется по признакам(категориям), например, категория трат, способ.

Недостатки метрики не показывает изменения сумм операций, не учитывает операции в других банках. Усреднение может не отражать прекращение оплат в определённый период месяца(заканчивается зарплата к концу месяца - денег нет)

#### 5.2.2. Динамика фиксированной группы и индивидуальных изменений

#### 5.2.2.1 Динамика снижения активности 

#### Подготовка фиксированной группы
Сохраняем всех сопоставимых клиентов(даже без оплат) - фиксированная группа

In [ ]:
# Задаём полный календарь
months4 = pd.period_range('2024-01', '2025-06', freq='M')
required4 = ['client_id', 'activity_comparable', 'payments_base', 'payments_current',
             'activity_base', 'activity_current', 'activity_change',
             'activity_change_pct', 'activity_state', 'activity_declined']
missing4 = [name for name in required4 if name not in client_activity.columns]

# Фиксируем состав группы: клиенты без оплат также остаются.
activity_dynamics = client_activity.loc[
    client_activity['activity_comparable'].fillna(False)
].copy()


# Формируем общую матрицу клиент × месяц без изменения исходной.
monthly_fixed4 = counts_wide.loc[activity_dynamics['client_id'], months4].copy()


# Сверяем суммы периодов с уже рассчитанными клиентскими метриками.
base4 = monthly_fixed4.loc[:, '2024-01':'2024-12']
current4 = monthly_fixed4.loc[:, '2025-01':'2025-06']
metric_check4 = activity_dynamics.set_index('client_id')

n_fixed4 = len(activity_dynamics)
positive_base4 = activity_dynamics['payments_base'].gt(0)
n_positive4 = int(positive_base4.sum())

def number4(value, digits=2):
    return f'{value:,.{digits}f}' if pd.notna(value) else 'не определено'



вывод:
- фиксированная группа составляет 5'542 клиента
- у 5240(94,55%) есть хотя бы одна оплата в 2024 году. 
- у 302 не было оплат за 2024

#### 5.2.2.2 Общая помесячная динамика
Поcчитаем:
- охват - доля клиентов с хотя бы одной успешной оплатой за месяц(активные)
- интенсивность - среднее число оплат, которые совершили эти клиенты
Среднее число оплат на всех = доля активных * интенсивность

In [ ]:
# Отдельно считаем объём, охват и интенсивность: их нельзя смешивать.
monthly_overview4 = pd.DataFrame({
    'Успешных оплат': monthly_fixed4.sum(axis=0),
    'Активных клиентов': monthly_fixed4.gt(0).sum(axis=0),
    'Медиана оплат на клиента': monthly_fixed4.median(axis=0),
})
monthly_overview4['Клиентов в группе'] = n_fixed4
monthly_overview4['Доля активных, %'] = monthly_overview4['Активных клиентов'].div(n_fixed4).mul(100)
monthly_overview4['Оплат на клиента'] = monthly_overview4['Успешных оплат'].div(n_fixed4)
monthly_overview4['Оплат на активного'] = monthly_overview4['Успешных оплат'].div(
    monthly_overview4['Активных клиентов'].replace(0, np.nan)
)
monthly_overview4.index.name = 'Месяц'
display(monthly_overview4.round(2))

# Показываем составляющие частоты на двух панелях.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
x4 = months4.to_timestamp()
for ax, column, title, ylabel in [
    (axes[0], 'Доля активных, %', 'Охват: активные клиенты', 'Доля фиксированной группы, %'),
    (axes[1], 'Оплат на активного', 'Интенсивность среди активных', 'Успешных оплат на активного клиента'),
]:
    ax.plot(x4, monthly_overview4[column], marker='o')
    ax.axvline(pd.Timestamp('2025-01-01'), color='gray', linestyle='--')
    ax.set(title=title, xlabel='Месяц', ylabel=ylabel)
    ax.set_ylim(bottom=0)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.2)
    sns.despine(ax=ax)
fig.suptitle(f'Фиксированная группа: {n_fixed4:,} клиентов')
plt.tight_layout()
plt.show()

# Взвешиваем интенсивность числом активных клиент-месяцев.
period_rows4 = []
for label, cols in [('2024 год', base4.columns), ('Январь–июнь 2025', current4.columns)]:
    part = monthly_overview4.loc[cols]
    active_client_months = part['Активных клиентов'].sum()
    period_rows4.append({
        'Период': label,
        'Оплат на клиента в месяц': part['Успешных оплат'].sum() / (n_fixed4 * len(cols)),
        'Средняя месячная доля активных, %': part['Доля активных, %'].mean(),
        'Оплат на активный клиент-месяц': (
            part['Успешных оплат'].sum() / active_client_months if active_client_months else np.nan
        ),
    })
period_overview4 = pd.DataFrame(period_rows4).set_index('Период')
display(period_overview4.round(2))
mean_base4, mean_current4 = period_overview4['Оплат на клиента в месяц'].tolist()
aggregate_change4 = (mean_current4 / mean_base4 - 1) * 100 if mean_base4 else np.nan
peak4 = monthly_overview4['Оплат на клиента'].idxmax()
low4 = monthly_overview4['Оплат на клиента'].idxmin()


вывод:
средняя месячная частота снизилась, но при этом средняя месячная доля активных выросла, то есть снижение активности связано с меньшим количеством оплат среди активных, а не с уменьшением среднего охвата.
Динамика - не монотонное падение
Решение: нужно проверить на сезонность, то есть сопоставить одинаковые месяцы двух лет.

#### 5.2.2.3 Сравнение одинаковых календарных месяцев
Сравним первые полугодия 2024 и 2025 по месяцам на одной группе


In [ ]:
# Сопоставляем одинаковые месяцы, сохраняя фиксированный знаменатель.
h1_base4 = monthly_fixed4.loc[:, '2024-01':'2024-06']
h1_current4 = current4
same_month4 = pd.DataFrame({
    '2024': h1_base4.mean(axis=0).to_numpy(),
    '2025': h1_current4.mean(axis=0).to_numpy(),
}, index=pd.Index(['Январь', 'Февраль', 'Март', 'Апрель', 'Май', 'Июнь'], name='Месяц'))
same_month4['Изменение, оплат на клиента'] = same_month4['2025'] - same_month4['2024']
same_month4['Изменение, %'] = same_month4['2025'].div(same_month4['2024'].replace(0, np.nan)).sub(1).mul(100)
display(same_month4.round(2))

fig, ax = plt.subplots(figsize=(9, 4.5))
for year in ['2024', '2025']:
    ax.plot(same_month4.index, same_month4[year], marker='o', label=year)
ax.set(title=f'Одинаковые календарные месяцы; n = {n_fixed4:,}',
       xlabel='Месяц', ylabel='Успешных оплат на клиента за месяц')
ax.set_ylim(bottom=0)
ax.legend(title='Год')
ax.grid(axis='y', alpha=0.2)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

# Отношение средних уровней не равно среднему индивидуальных процентов.
h1_mean_base4 = h1_base4.to_numpy().mean()
h1_mean_current4 = h1_current4.to_numpy().mean()
h1_change4 = (h1_mean_current4 / h1_mean_base4 - 1) * 100 if h1_mean_base4 else np.nan
activity_dynamics['activity_h1_2024'] = activity_dynamics['client_id'].map(h1_base4.mean(axis=1))
# Полугодия равны по длительности, поэтому сравниваем целые счётчики
activity_dynamics['change_same_h1'] = activity_dynamics['activity_current'] - activity_dynamics['activity_h1_2024']
activity_dynamics['declined_same_h1'] = activity_dynamics['client_id'].map(
    h1_current4.sum(axis=1).lt(h1_base4.sum(axis=1))
)

# Для проверки согласованности оставляем клиентов с положительной базой в обоих вариантах сравнения.
both_bases4 = positive_base4 & activity_dynamics['activity_h1_2024'].gt(0)
direction_check4 = pd.crosstab(
    activity_dynamics.loc[both_bases4, 'activity_declined'],
    activity_dynamics.loc[both_bases4, 'declined_same_h1'],
).reindex(index=[False, True], columns=[False, True], fill_value=0)
direction_check4.index = ['Нет снижения по основной метрике', 'Есть снижение по основной метрике']
direction_check4.columns = ['Нет снижения при сравнении полугодий', 'Есть снижение при сравнении полугодий']
fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(direction_check4, annot=True, fmt='d', cmap='Blues', cbar=False, linewidths=1, linecolor='white', annot_kws={'fontsize': 13}, ax=ax)

ax.set( title=('Согласованность двух определений снижения\n'
               f'Клиентов с положительной базой: {int(both_bases4.sum()):,}'),
    xlabel='База: январь–июнь 2024 года', ylabel='База: весь 2024 год',)
ax.set_xticklabels(['Нет снижения', 'Есть снижение'], rotation=0)
ax.set_yticklabels(['Нет снижения', 'Есть снижение'], rotation=0)

ax.grid(False)
plt.tight_layout()
plt.show()

# Сохраняем показатель для итоговой сводки главы.
months_lower4 = int( same_month4['Изменение, оплат на клиента'].lt(0).sum())

months_lower4 = int(same_month4['Изменение, оплат на клиента'].lt(0).sum())


вывод:
все точки 2025 года ниже 2024, но при этом форму линии тоже примерно повторяют одну. Но говорить о сезонности нельзя по 2 полугодиям.
Для некоторых клиентов способ оценки снижения влияет на результат:
- 1257 — снижения нет по обоим способам
- 3043 — снижение есть по обоим способам
- 502 — снижение только по основной метрике (по 2024 платят реже, а по I полугодию 2024 не реже) - значит во II полугодии 2024 средняя частота была выше, чем в I
- 65 — снижение только при сравнении одинаковых полугодий (по 2024 году нет снижения, по I полугодию есть) - значит во II полугодии 2024 средняя частота была ниже, чем в I

#### 5.2.2.4 Распределение индивидуальных изменений

In [ ]:
# Для анализа индивидуальных изменений выбираем клиентов с положительной базовой частотой.
change_sample4 = activity_dynamics.loc[positive_base4].copy()
if len(change_sample4):
    change_stats4 = change_sample4[['activity_base', 'activity_current', 'activity_change', 'activity_change_pct']].describe()
    display(change_stats4.round(2))
    fig, ax = plt.subplots(figsize=(9, 4.5))
    sns.histplot(data=change_sample4, x='activity_change', bins=45, ax=ax)
    ax.axvline(0, color='gray', linestyle='--', label='Без изменения')
     # Медиана характеризует типичное индивидуальное изменение.
    median_change4 = change_sample4['activity_change'].median()

    ax.axvline(median_change4, color='darkorange', label=f'Медиана: {median_change4:.2f}')
    ax.set(title='Распределение индивидуального изменения частоты оплат', xlabel='Текущая − базовая частота, оплат в месяц', ylabel='Количество клиентов')

    ax.legend(fontsize=8)
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

    # Считаем строгое снижение по ранее созданному признаку.
    n_decline4 = int(change_sample4['activity_declined'].sum())
    stopped4 = int(change_sample4['payments_current'].eq(0).sum())
    
else:
    n_decline4 = stopped4 = 0
    

Вывод: преобладает снижение над ростом

#### 5.2.2.5 Характер снижения к концу периода
Среди клиентов со снижением выделим профили нескольких групп по отсутствии оплат в разные периоды

In [ ]:
# Сравниваем одинаковые по длительности кварталы 2025 года.
q1_counts4 = monthly_fixed4.loc[:, '2025-01':'2025-03'].sum(axis=1)
q2_counts4 = monthly_fixed4.loc[:, '2025-04':'2025-06'].sum(axis=1)
activity_dynamics['payments_q1_2025'] = activity_dynamics['client_id'].map(q1_counts4)
activity_dynamics['payments_q2_2025'] = activity_dynamics['client_id'].map(q2_counts4)
activity_dynamics['activity_q1_2025'] = activity_dynamics['payments_q1_2025'].div(3)
activity_dynamics['activity_q2_2025'] = activity_dynamics['payments_q2_2025'].div(3)

# Профиль назначается только клиентам со снижением основной метрики.
declined4 = activity_dynamics.loc[activity_dynamics['activity_declined']].copy()
profile_order4 = [
    'Нет оплат всё полугодие',
    'Были оплаты в I квартале; нет во II',
    'Во II квартале оплаты есть; частота ниже базы',
    'Во II квартале частота не ниже базы',
]
# Порядок условий делает четыре группы взаимоисключающими.
declined4['decline_profile'] = np.select(
    [declined4['payments_current'].eq(0),
     declined4['payments_q2_2025'].eq(0),
     declined4['payments_q2_2025'].mul(4).lt(declined4['payments_base'])],
    profile_order4[:3], default=profile_order4[3],
)
# Для остальных профиль снижения не назначается
activity_dynamics['decline_profile'] = activity_dynamics['client_id'].map(
    declined4.set_index('client_id')['decline_profile']
).astype('string')
profile_summary4 = pd.DataFrame(index=pd.Index(profile_order4, name='Профиль снижения'))
profile_summary4['Клиентов'] = declined4['decline_profile'].value_counts().reindex(profile_order4, fill_value=0)
profile_summary4['Доля снизивших, %'] = profile_summary4['Клиентов'].div(len(declined4) if len(declined4) else np.nan).mul(100)
profile_summary4['Доля клиентов с положительной базой, %'] = profile_summary4['Клиентов'].div(n_positive4 if n_positive4 else np.nan).mul(100)
display(profile_summary4.round(2))

# Кривые описывают группы, определённые по этим же данным.
if len(declined4):
    profile_means4 = pd.DataFrame(index=profile_order4, columns=months4, dtype=float)
    for profile in profile_order4:
        ids = declined4.loc[declined4['decline_profile'].eq(profile), 'client_id']
        if len(ids):
            profile_means4.loc[profile] = monthly_fixed4.loc[ids].mean(axis=0)
    fig, ax = plt.subplots(figsize=(12, 5))
    for profile in profile_order4:
        n_profile = int(profile_summary4.loc[profile, 'Клиентов'])
        if n_profile:
            ax.plot(months4.to_timestamp(), profile_means4.loc[profile], marker='o',
                    markersize=3, label=f'{profile}; n={n_profile:,}')
    ax.axvline(pd.Timestamp('2025-01-01'), color='gray', linestyle='--')
    ax.axvline(pd.Timestamp('2025-04-01'), color='gray', linestyle=':')
    ax.set(title='Помесячная динамика диагностических профилей снижения',
           xlabel='Месяц', ylabel='Оплат на клиента профиля за месяц')
    ax.set_ylim(bottom=0)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1))
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()
    late_stop4 = int(profile_summary4.loc[profile_order4[1], 'Клиентов'])
    recovered4 = int(profile_summary4.loc[profile_order4[3], 'Клиентов'])
    
else:
    late_stop4 = recovered4 = 0
    

вывод:
после январского провала 2946 клиентов продолжают платить, но на меньшем уровне, а 544 клиента восстанавливают средний уровень до провала

In [ ]:
# Собираем итог без новых удалений, отборов или изменения исходных сумм.
chapter4_summary = pd.Series({
    'Сопоставимых клиентов': n_fixed4,
    'Клиентов с положительной базой': n_positive4,
    'Клиентов со снижением': n_decline4,
    'Доля снизивших среди положительной базы, %': 100 * n_decline4 / n_positive4 if n_positive4 else np.nan,
    'Средняя частота 2024, оплат/месяц': mean_base4,
    'Средняя частота I полугодия 2025, оплат/месяц': mean_current4,
    'Изменение среднего уровня, %': aggregate_change4,
    'Изменение при одинаковых полугодиях, %': h1_change4,
    'Месяцев со снижением при календарном сравнении': months_lower4,
    'Нет оплат всё полугодие среди снизивших': stopped4,
    'Были оплаты в I квартале, нет во II': late_stop4,
    'Частота II квартала не ниже базы среди снизивших': recovered4,
}, name='Значение')
display(chapter4_summary.to_frame().round(2))


# Журналируем создание признаков одной записью.
add_log(
    key='clients_activity_dynamics_profiles',
    field='activity_h1_2024; change_same_h1; declined_same_h1; payments_q1_2025; payments_q2_2025; activity_q1_2025; activity_q2_2025; decline_profile',
    issue='Нужно отделить масштаб снижения от календарного сравнения и активности в конце периода',
    scope='Сопоставимые клиенты в activity_dynamics; профиль назначается только снизившим',
    n_rows=len(activity_dynamics),
    value_before='Дополнительные признаки динамики и профили отсутствовали',
    action='Созданы производные признаки; исходные таблицы сохранены',
    value_after=(f'{len(activity_dynamics)} клиентов; профиль назначен {len(declined4)}; '
                     f'нет оплат во II квартале после оплат в I: {late_stop4}'),
    rationale=('Одинаковые полугодия — дополнительная проверка, не замена основного сравнения. '
                'Конец периода — апрель–июнь 2025. Профили не являются группами кампании '
                'и не доказывают причины или уход. Операции и анкеты не удалялись.'),
)


вывод:
в доступной выгрузке и к выбранной группе не устанавливается изменение оборота, причины поведения или фактический уход из банка.


#### 5.2.3. Динамика расхождений категории и MCC

#### 5.2.3.1 Расхождения по месяцам
Проверим, сосредоточены ли расхождения в отдельных месяцах.

In [ ]:
# Оставляем операции с однозначно определённой основной категорией
valid_mcc_data = category_check.loc[assessed].copy()

# Рассчитываем общую долю отклонений за весь период
overall_mismatch_rate = valid_mcc_data['category_differs'].mean() * 100

# Группировка по месяцам
monthly_category = (
    valid_mcc_data
    .groupby(valid_mcc_data['txn_datetime'].dt.to_period('M'))['category_differs']
    .agg(total='size', differing='sum')
    .reindex(pd.period_range('2024-01', '2025-06', freq='M'))
)

monthly_category[['total', 'differing']] = (
    monthly_category[['total', 'differing']]
    .fillna(0)
    .astype(int)
)

monthly_category['share_pct'] = (
    monthly_category['differing']
    .div(monthly_category['total'].replace(0, float('nan')))
    .mul(100)
)

# Построение графика
fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(
    monthly_category.index.astype(str),
    monthly_category['share_pct'],
    marker='o',
    label='Доля за месяц',
)

ax.axhline(
    overall_mismatch_rate,  # Используем рассчитанную общую долю
    color='darkorange',
    linestyle='--',
    label=f'Весь период: {overall_mismatch_rate:.2f}%',
)

# Нулевое начало оси не преувеличивает небольшие колебания.
ax.set_ylim(0, max(monthly_category['share_pct'].max(), overall_mismatch_rate) * 1.2,)

ax.set(
    title='Расхождения категории с основным соответствием MCC',
    xlabel='Месяц',
    ylabel='Доля отклонений, %',
)

ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.2)
ax.legend()

sns.despine()
plt.tight_layout()
plt.show()

вывод:
резких скачков доли расхождения по месяцам не наблюдается, наблюдаётся очень маленькое уменьшение доли отклонений. Проблема не ограничивается
одним отдельным месяцем.

Решение: операции не удаляем и не меняем

#### 5.2.3.1 Описательный анализ доли по месяцам


In [ ]:
from scipy import stats

# Сохраняем реальное расстояние между месяцами,
# даже если в каком-то месяце нет операций.
trend_data = monthly_category.sort_index().copy()

trend_data['month_number'] = (
    trend_data.index.year * 12 + trend_data.index.month
)
trend_data['month_number'] -= trend_data['month_number'].min()

trend_data = trend_data.loc[
    trend_data['total'].gt(0)
    & trend_data['share_pct'].notna()
].copy()

x = trend_data['month_number'].to_numpy(dtype=float)
y = trend_data['share_pct'].to_numpy(dtype=float)

# Двусторонняя проверка: не выбираем направление теста
# только потому, что уже увидели снижение на графике.
reg = stats.linregress(x, y)

# 95%-й доверительный интервал наклона.
critical_t = stats.t.ppf(0.975, df=len(trend_data) - 2)

slope_low = reg.slope - critical_t * reg.stderr
slope_high = reg.slope + critical_t * reg.stderr

trend_data['trend_pct'] = reg.intercept + reg.slope * x

# Изменение по линии между первым и последним доступными месяцами.
fitted_change = reg.slope * (x.max() - x.min())

trend_result = pd.Series({
    'Месяцев в анализе': len(trend_data),
    'Наклон, п.п. в месяц': reg.slope,
    'Нижняя граница 95% ДИ наклона': slope_low,
    'Верхняя граница 95% ДИ наклона': slope_high,
    'Изменение по линии за период, п.п.': fitted_change,
    'R²': reg.rvalue ** 2,
    'p-value': reg.pvalue,
}, name='Результат')

display(trend_result.to_frame().round(6))

fig, ax = plt.subplots(figsize=(11, 4))

labels = trend_data.index.astype(str)

ax.plot(
    labels,
    trend_data['share_pct'],
    marker='o',
    label='Фактическая доля',
)

ax.plot(
    labels,
    trend_data['trend_pct'],
    color='darkorange',
    linestyle='--',
    label=f'Линейный тренд: {reg.slope:+.4f} п.п./месяц',
)

ax.set(
    title='Временной тренд расхождений категории и MCC',
    xlabel='Месяц',
    ylabel='Доля отклонений, %',
)

ax.set_ylim(0, max( trend_data['share_pct'].max(), trend_data['trend_pct'].max(),) * 1.2,)

ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.2)
ax.legend()

sns.despine()
plt.tight_layout()
plt.show()

вывод:
регрессия описала небольшие снижения всего доли отклонений.

### 5.3. Частотные характеристики состояний активности

После подготовки метрики показываем частоты состояний среди сопоставимых клиентов с положительной базой.


In [ ]:
# Распространённость состояний среди сопоставимых клиентов с положительной базой
state_plot_sample = client_activity.loc[
    client_activity['activity_comparable']
    & client_activity['payments_base'].gt(0)
].copy()

state_order = [
    'Снижение',
    'Прекратились оплаты',
    'Без изменения',
    'Рост',
]

state_labels = {
    'Снижение': 'Снижение; оплаты продолжаются',
    'Прекратились оплаты': 'Нет оплат за январь–июнь 2025',
    'Без изменения': 'Частота не изменилась',
    'Рост': 'Частота выросла',
}

state_plot_table = (
    state_plot_sample['activity_state']
    .value_counts()
    .reindex(state_order, fill_value=0)
    .rename_axis('Состояние')
    .to_frame('Клиентов')
)

n_state_plot = len(state_plot_sample)

state_plot_table['Доля, %'] = (
    state_plot_table['Клиентов']
    .div(n_state_plot if n_state_plot else np.nan)
    .mul(100)
)

if n_state_plot:
    fig, ax = plt.subplots(figsize=(10, 4.5))

    labels = [
        state_labels[state]
        for state in state_plot_table.index
    ]

    bars = ax.barh(
        labels,
        state_plot_table['Доля, %'],
        color=[
            'steelblue',
            'crimson',
            'gray',
            'darkorange',
        ],
    )

    bar_labels = [
        f'{int(row["Клиентов"]):,} клиентов — {row["Доля, %"]:.2f}%'
        for _, row in state_plot_table.iterrows()
    ]

    ax.bar_label(
        bars,
        labels=bar_labels,
        padding=5,
        fontsize=9,
    )

    ax.set(
        title=(
            'Изменение месячной частоты успешных оплат\n'
            f'Сопоставимые клиенты с положительной базой: '
            f'n = {n_state_plot:,}'
        ),
        xlabel='Доля клиентов с положительной базой, %',
        ylabel='',
        xlim=(0, 100),
    )

    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)

    sns.despine(ax=ax)
    plt.tight_layout()
    plt.show()

else:
    pass

### 5.4. Разрезы и связи между признаками

#### 5.4.1. Кто снизил активность: клиентские разрезы


In [ ]:
# Используем клиентскую таблицу главы 4, не меняя исходные данные.
required5 = ['client_id', 'activity_comparable', 'payments_base',
             'activity_base', 'activity_current', 'activity_change',
             'activity_change_pct', 'activity_declined', 'segment',
             'loyalty_tier', 'card_type', 'city', 'has_mobile_app',
             'age', 'monthly_income', 'satisfaction_score', 'marketing_opt_in']
missing5 = sorted(set(required5) - set(activity_dynamics.columns))

# Положительная база нужна для определённого индивидуального процента.
profile_clients5 = activity_dynamics.loc[
    activity_dynamics['activity_comparable'].fillna(False)
    & activity_dynamics['payments_base'].gt(0)
].copy()

# Сверяем направление с целыми счётчиками: текущий период вдвое короче.
expected5 = profile_clients5['payments_current'].mul(2).lt(profile_clients5['payments_base'])
profile_clients5['activity_declined'] = expected5.astype(bool)
n5 = len(profile_clients5)
d5 = int(expected5.sum())
rate5 = 100 * expected5.mean()


#### 5.4.2. Сегмент, лояльность, тип карты и приложение

Анализируем численность и долю снизивших внутри каждой группы.


In [ ]:
# Одна функция заменяет повторяющийся код для категориальных разрезов.
# Пропуски выделяем в отдельную группу, а не смешиваем с известными ответами.
profile_labels5 = {
    'n': 'Всего клиентов',
    'declined': 'Снизили активность, чел.',
    'base_median': 'Частота 2024: медиана',
    'current_median': 'Частота 2025: медиана',
    'change_median': 'Изменение: медиана, оплат/мес.',
    'change_pct_median': 'Изменение: медиана, %',
    'decline_pct': 'Снизили внутри группы, %',
    'difference_pp': 'Отличие от общей доли, п.п.',
    'share_of_declined_pct': 'Среди всех снизивших, %',
    'small_group': 'Менее 150 клиентов',
}

def make_profile5(data, column):
    working = data.copy()
    working['_group5'] = working[column].astype('string').fillna('Нет сведений')
    table = working.groupby('_group5', observed=True, sort=False).agg(
        n=('client_id', 'size'),
        declined=('activity_declined', 'sum'),
        base_median=('activity_base', 'median'),
        current_median=('activity_current', 'median'),
        change_median=('activity_change', 'median'),
        change_pct_median=('activity_change_pct', 'median'),
    )
    table['decline_pct'] = table['declined'].div(table['n']).mul(100)
    table['difference_pp'] = table['decline_pct'] - rate5
    table['share_of_declined_pct'] = table['declined'].div(d5 if d5 else np.nan).mul(100)
    table['small_group'] = table['n'].lt(150)
    table.index.name = column
    return table.sort_values('decline_pct', ascending=False)


# Формируем таблицу и предметный мини-вывод после каждого разреза.
profile_tables5 = {}
main_features5 = {
    'segment': 'Сегмент', 'loyalty_tier': 'Уровень лояльности',
    'card_type': 'Тип карты', 'has_mobile_app': 'Наличие приложения',
}
for column, title in main_features5.items():
    table = make_profile5(profile_clients5, column)
    profile_tables5[column] = table
    display((f'{title}'))
    display(table.rename(columns=profile_labels5).round(2))
    known = table.loc[table.index != 'Нет сведений']
    if not known.empty:
        highest = known['decline_pct'].idxmax()
        largest = known['declined'].idxmax()
        a, b = known.loc[highest], known.loc[largest]
        

# Одинаковая процентная шкала для четырёх разрезов.
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (column, title) in zip(axes.flat, main_features5.items()):
    table = profile_tables5[column].sort_values('decline_pct')
    labels = [f'{label} (n={int(row.n)})' for label, row in table.iterrows()]
    bars = ax.barh(labels, table['decline_pct'])
    ax.bar_label(bars, labels=[f'{x:.1f}%' for x in table['decline_pct']], padding=3)
    ax.axvline(rate5, color='darkorange', linestyle='--', label=f'Вся группа: {rate5:.1f}%')
    ax.set(title=title, xlabel='Клиентов со снижением внутри группы, %', xlim=(0, 110))
    ax.legend(fontsize=8)
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()


вывод:
массовый сегмент и отсутствие приложения имеют побольше снижений активности относительно остальных, поэтому они будут представлять дальнейший интерес при исследовании. 

Решение: проверим пересечение массового сегмента и отсутствия приложения, возможно ещё базовую лояльность.

#### 5.4.3. Возраст, доход и оценка обслуживания


In [ ]:
# Описательные интервалы, не критерии ошибки или кампании.
profile_clients5['age_group5'] = pd.cut(
    profile_clients5['age'], bins=[-np.inf, 34, 44, 54, 64, np.inf],
    labels=['До 35', '35–44', '45–54', '55–64', '65+'], right=True,
)
profile_clients5['income_group5'] = pd.cut(
    profile_clients5['monthly_income'],
    bins=[0, 10000, 50000, 100000, 200000, np.inf],
    labels=['До 10 тыс.', '10–50 тыс.', '50–100 тыс.', '100–200 тыс.', '200 тыс. и выше'],
    right=False,
)
# Нули и отрицательные доходы должны быть разрешены до этой главы.
if profile_clients5['monthly_income'].dropna().le(0).any():
    raise ValueError('Сначала проверьте неположительные доходы в главе 1.')

for column in ['age_group5', 'income_group5', 'satisfaction_score']:
    table = make_profile5(profile_clients5, column)
    profile_tables5[column] = table
    display((f'Разрез: {column}'))
    display(table.rename(columns=profile_labels5).round(2))
    nonmissing = table.loc[table.index != 'Нет сведений']
    if not nonmissing.empty:
        high = nonmissing['decline_pct'].idxmax()
        low = nonmissing['decline_pct'].idxmin()
        

# Числовые показатели показываем отдельно для двух направлений изменения.
# Доход описываем медианой и квартилями, а не одним средним.
numeric_summary5 = profile_clients5.groupby('activity_declined').agg(
    n=('client_id', 'size'), 
    age_known=('age', 'count'),
    age_median=('age', 'median'), 
    income_known=('monthly_income', 'count'),
    income_q25=('monthly_income', lambda x: x.quantile(.25)),
    income_median=('monthly_income', 'median'),
    income_q75=('monthly_income', lambda x: x.quantile(.75)),
)
numeric_summary5.index.name = 'Есть снижение'
display(numeric_summary5.rename(
    columns={'n': 'Всего клиентов', 'age_known': 'Возраст известен, чел.',
             'age_median': 'Возраст: медиана, лет', 'income_known': 'Доход известен, чел.',
             'income_q25': 'Доход: Q1, ₽', 'income_median': 'Доход: медиана, ₽',
             'income_q75': 'Доход: Q3, ₽'},
    index={False: 'Нет снижения', True: 'Есть снижение'}
).rename_axis('Изменение активности').round(2))


# Диаграммы показывают доли из готовых таблиц без пересчёта правил.
plot_settings5 = {
    'age_group5': ('Возраст', ['До 35', '35–44', '45–54', '55–64', '65+', 'Нет сведений']),
    'income_group5': ('Доход, ₽', ['До 10 тыс.', '10–50 тыс.', '50–100 тыс.', '100–200 тыс.', '200 тыс. и выше', 'Нет сведений']),
    'satisfaction_score': ('Оценка обслуживания', ['1', '2', '3', '4', '5', 'Нет сведений']),
}
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (column, (title, order)) in zip(axes, plot_settings5.items()):
    view = profile_tables5[column].copy()
    view.index = view.index.astype(str)
    extra = set(view.index) - set(order)
    if extra:
        raise ValueError(f'Неожиданные группы в {column}: {extra}')
    view = view.loc[[label for label in order if label in view.index]]
    labels = [f'{label} (n={int(row.n):,})' + (' *' if row.n < 150 else '')
              for label, row in view.iterrows()]
    bars = ax.barh(labels, view['decline_pct'],
                   color=['gray' if label == 'Нет сведений' else 'steelblue' for label in view.index])
    ax.bar_label(bars, labels=[f'{x:.2f}%' for x in view['decline_pct']], padding=3)
    ax.axvline(rate5, color='darkorange', linestyle='--', label=f'Общая доля: {rate5:.2f}%')
    ax.invert_yaxis()
    ax.set(title=title, xlabel='Снизили внутри группы, %', xlim=(0, 110))
    ax.set_xticks([0, 20, 40, 60, 80, 100])
    ax.legend(fontsize=8, loc='lower right')
    sns.despine(ax=ax)
fig.suptitle(f'Доля снижения по клиентским признакам; n={n5:,}')
fig.text(.01, .01, '* Менее 150 клиентов. Серый цвет — нет сведений.', fontsize=9)
plt.tight_layout(rect=(0, .05, 1, .93))
plt.show()

# Отдельно показываем медианы и квартильный диапазон дохода.
view = numeric_summary5.reindex([False, True])
labels = ['Нет снижения', 'Есть снижение']
positions = np.arange(2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(view['age_median'], positions, s=70)
for pos, (_, row) in enumerate(view.iterrows()):
    axes[0].annotate(f'{row.age_median:.0f} лет; n={int(row.age_known)}',
                     (row.age_median, pos), xytext=(8, 8), textcoords='offset points')
axes[0].set(yticks=positions, yticklabels=labels, xlabel='Возраст, лет',
            title='Медианный возраст', xlim=(0, max(65, view.age_median.max()+12)))
axes[1].hlines(positions, view['income_q25'], view['income_q75'], linewidth=4)
axes[1].scatter(view['income_median'], positions, s=70, color='darkorange', zorder=3)
for pos, (_, row) in enumerate(view.iterrows()):
    axes[1].annotate(f'Медиана {row.income_median:,.0f} ₽; n={int(row.income_known)}',
                     (row.income_median, pos), xytext=(0, 12), textcoords='offset points', ha='center')
axes[1].set(yticks=positions, yticklabels=labels, xlabel='Ежемесячный доход, ₽',
            title='Доход: медиана и центральные 50%', xlim=(0, view.income_q75.max()*1.2))
for ax in axes:
    ax.set_ylim(-.5, 1.5)
    ax.invert_yaxis()
    sns.despine(ax=ax)
plt.tight_layout()
plt.show()


Вывод:
- по возрасту: наибольшая доля снижения у клиентов 65+, возможно из-за старения клиентов снижается активность 
- по доходу: постепенное уменьшение доли снижения с ростом дохода
- по оценке обслуживания: доля снижения последовательно уменьшается(монотонно) с повышением оценки. Но по ~60% клиентов у нас нет данных, поэтому судить по этому признаку не очень объективно.
Решение: далее изучим детальнее данные признаки

#### 5.4.4. Изменение возраста
Подготовим возрастные группы и выборки успешных оплат

In [ ]:
# Отдельные рабочие таблицы; кампания и исходные данные не меняются.
age_h = df_clients.set_index('client_id').copy()
AGE_LABELS_H = ['До 35', '35–44', '45–54', '55–64', '65+', 'Возраст неизвестен']
PERIODS_H = {
    'I пол. 2024': ('2024-01-01', '2024-07-01'),
    'II пол. 2024': ('2024-07-01', '2025-01-01'),
    'I пол. 2025': ('2025-01-01', '2025-07-01'),
}

def completed_age_h(birth, date):
    dates = pd.Series(date, index=birth.index) if isinstance(date, pd.Timestamp) else date.reindex(birth.index)
    pending = (dates.dt.month < birth.dt.month) | ((dates.dt.month == birth.dt.month) & (dates.dt.day < birth.dt.day))
    return (dates.dt.year - birth.dt.year - pending.astype(int)).astype('Int64').where(
        birth.notna() & dates.notna() & dates.ge(birth))

def age_group_h(age):
    return pd.cut(age.astype(float), [0, 35, 45, 55, 65, np.inf],
                  labels=AGE_LABELS_H[:-1], right=False).astype('string').fillna(AGE_LABELS_H[-1])

for column, date in [('age_start_h', '2024-01-01'), ('age_end_h', '2025-06-30'),
                     ('age_dec_projection_h', '2025-12-31')]:
    age_h[column] = completed_age_h(age_h.birth_date, pd.Timestamp(date))
age_h['age_registration_h'] = completed_age_h(age_h.birth_date, age_h.registration_date)
age_h['registration_group_h'] = age_group_h(age_h.age_registration_h)
age_h['birthday65_h'] = age_h.birth_date.map(
    lambda d: d + pd.DateOffset(years=65) if pd.notna(d) else pd.NaT)

metric_h = client_activity.set_index('client_id')
fixed_age_h = age_h.reindex(metric_h.index[metric_h.activity_comparable.fillna(False)]).copy()
# Метрики берём только из client_activity, чтобы не дублировать одноимённые поля.
for column in ['activity_base', 'activity_current', 'activity_change', 'activity_declined', 'payments_base']:
    fixed_age_h[column] = metric_h[column].reindex(fixed_age_h.index)
fixed_age_h['baseline_group_h'] = age_group_h(fixed_age_h.age_start_h)
fixed_age_h['age_cohort_h'] = np.select([
    fixed_age_h.age_start_h.ge(65).fillna(False).to_numpy(dtype=bool),
    (fixed_age_h.age_start_h.lt(65) & fixed_age_h.age_end_h.ge(65)).fillna(False).to_numpy(dtype=bool),
    fixed_age_h.age_end_h.lt(65).fillna(False).to_numpy(dtype=bool),
], ['Уже 65+ в начале 2024', 'Исполнилось 65 за период',
    'Младше 65 до конца наблюдения'], default='Возраст неизвестен')

payments_h = transactions_period.loc[
    transactions_period.status.astype('string').str.strip().str.casefold().eq('успешно').fillna(False)
    & transactions_period.amount.gt(0) & np.isfinite(transactions_period.amount),
    ['client_id', 'txn_datetime'],
].copy()
payments_h['period_h'] = pd.cut(payments_h.txn_datetime,
    pd.to_datetime(['2024-01-01', '2024-07-01', '2025-01-01', '2025-07-01']),
    labels=list(PERIODS_H), right=False)
# Таблица включает нулевые оплаты всех клиентов справочника.
counts_h = (payments_h.groupby(['client_id', 'period_h'], observed=True).size().unstack(fill_value=0)
            .reindex(index=age_h.index, columns=list(PERIODS_H), fill_value=0).fillna(0).astype(int))

#### 5.4.4.1 Возрастной состав и частота всех возрастных групп

In [ ]:
# Состав одной фиксированной группы на три даты.
age_composition_h = pd.DataFrame({
    label: age_group_h(fixed_age_h[column]).value_counts().reindex(AGE_LABELS_H, fill_value=0)
    for column, label in [('age_start_h', '01.01.2024'), ('age_end_h', '30.06.2025'),
                         ('age_dec_projection_h', '31.12.2025: расчёт')]
})

# Для изменения частоты возрастная группа закреплена на 01.01.2024.
def summarize_age_h(frame, group):
    summary = frame.groupby(group, observed=True).agg(
        n=('activity_base', 'size'), base_mean=('activity_base', 'mean'),
        current_mean=('activity_current', 'mean'), mean_change=('activity_change', 'mean'))
    positive = frame.loc[frame.payments_base.gt(0)].groupby(group, observed=True).agg(
        n_positive=('activity_declined', 'size'), declined=('activity_declined', 'sum'))
    summary = summary.join(positive).fillna({'n_positive': 0, 'declined': 0})
    summary['decline_pct_positive'] = 100 * summary.declined / summary.n_positive.replace(0, np.nan)
    summary['contribution'] = summary.mean_change * summary.n / (len(frame) or np.nan)
    return summary

age_frequency_h = summarize_age_h(fixed_age_h, 'baseline_group_h').reindex(AGE_LABELS_H)
# Сохранено прежнее имя для совместимости с последующими возрастными проверками.
cohort_summary_h = summarize_age_h(fixed_age_h, 'age_cohort_h')
crossed_h = int(fixed_age_h.age_cohort_h.eq('Исполнилось 65 за период').sum())

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
age_composition_h.plot.bar(ax=axes[0], rot=0)
axes[0].set(title=f'Возраст фиксированной группы, n={len(fixed_age_h)}', xlabel='', ylabel='Клиентов')
age_frequency_h[['base_mean', 'current_mean']].rename(
    columns={'base_mean': '2024', 'current_mean': 'I пол. 2025'}).plot.bar(ax=axes[1], rot=0)
axes[1].set(title='Частота по возрасту на 01.01.2024', xlabel='', ylabel='Оплат / клиента / месяц')
for ax in axes:
    ax.tick_params(axis='x', labelsize=8)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
display(age_frequency_h.rename(columns={
    'n': 'Клиентов', 'base_mean': 'Частота 2024', 'current_mean': 'Частота 2025',
    'mean_change': 'Изменение частоты', 'n_positive': 'Положительная база',
    'declined': 'Снизили', 'decline_pct_positive': 'Снизили / положительная база, %',
    'contribution': 'Вклад в общее изменение'}).round(3))

Вывод: клиенты страреют, но частота оплат сокращается во всех возрастных группах. Нужно изучить подробнее.

#### 5.4.4.2 Регистрации всех возрастов и переходы через 65

In [ ]:
# H_AGE: возраст новых клиентов фиксируем на дату регистрации.
registrations_h = age_h.loc[age_h.registration_date.ge('2024-01-01')
                            & age_h.registration_date.lt('2025-07-01')].copy()
registrations_h['month_h'] = registrations_h.registration_date.dt.to_period('M')
monthly_registrations_age5 = (registrations_h.groupby(['month_h', 'registration_group_h'], observed=True)
    .size().unstack(fill_value=0).reindex(index=pd.period_range('2024-01', '2025-06', freq='M'),
    columns=AGE_LABELS_H, fill_value=0).fillna(0).astype(int))
monthly_registrations_age5.index.name = 'Месяц'
monthly_view_h = monthly_registrations_age5.copy()
monthly_view_h.index = monthly_view_h.index.astype(str)
ax = monthly_view_h.plot.bar(stacked=True, figsize=(13, 4.5), rot=45, colormap='tab20')
ax.set(title='Новые регистрации: все возраста на дату прихода', xlabel='Месяц регистрации', ylabel='Клиентов')
ax.legend(title='Возраст', fontsize=8, ncol=3)
plt.tight_layout(); plt.show()

flow_rows_h = []
for label, (start, end) in PERIODS_H.items():
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    registered = age_h.registration_date.ge(start) & age_h.registration_date.lt(end)
    before = completed_age_h(age_h.birth_date, start - pd.Timedelta(days=1))
    after = completed_age_h(age_h.birth_date, end - pd.Timedelta(days=1))
    crossed = (age_h.registration_date.lt(start) & before.lt(65) & after.ge(65)).fillna(False)
    row = age_h.loc[registered, 'registration_group_h'].value_counts().reindex(AGE_LABELS_H, fill_value=0).to_dict()
    row.update({'Период': label, 'Все регистрации': int(registered.sum()),
                'Достигли 65 среди прежних клиентов': int(crossed.sum())})
    flow_rows_h.append(row)
age_flow_comparison5 = pd.DataFrame(flow_rows_h).set_index('Период')
registration_summary_h = age_flow_comparison5[AGE_LABELS_H + ['Все регистрации']].copy()
display(age_flow_comparison5)

Вывод: прирост клиентов также сокращается по всем возрастным группам, а не только среди людей помоложе

#### 5.4.4.3 Оплаты новых клиентов всех возрастов и общий баланс

In [ ]:
# Непересекающееся разбиение ВСЕХ анкет.
old_h = age_h.registration_date.le(pd.Timestamp('2024-01-01'))
transition_h = (old_h & age_h.birthday65_h.ge('2024-07-01') & age_h.birthday65_h.lt('2025-01-01'))
age_h['accounting_group_h'] = 'Прочие / неизвестна дата регистрации'
age_h.loc[old_h, 'accounting_group_h'] = 'Прежние: остальные'
age_h.loc[transition_h, 'accounting_group_h'] = 'Прежние: достигли 65 во II пол. 2024'
old_label_h = 'Прежние: остальные'
senior_label_h = 'Прежние: достигли 65 во II пол. 2024'
new_labels_h = []
for period, (start, end) in PERIODS_H.items():
    # 01.01.2024 уже включены в прежних, как в activity_comparable.
    new = (~old_h & age_h.registration_date.ge(start) & age_h.registration_date.lt(end))
    for age in AGE_LABELS_H:
        label = f'Новые {period}: {age}'
        new_labels_h.append(label)
        age_h.loc[new & age_h.registration_group_h.eq(age), 'accounting_group_h'] = label

order_h = [senior_label_h, old_label_h] + new_labels_h + ['Прочие / неизвестна дата регистрации']
accounting_h = counts_h.groupby(age_h.accounting_group_h).sum().reindex(order_h, fill_value=0)
accounting_h.insert(0, 'Клиентов', age_h.accounting_group_h.value_counts().reindex(order_h, fill_value=0))
accounting_h['Изменение I пол. 2025 − I пол. 2024'] = accounting_h['I пол. 2025'] - accounting_h['I пол. 2024']
# В таблице нет скрытых возрастов: оставляем даже пустые когорты.
accounting_h.loc['Итого по всем анкетам'] = accounting_h.sum()
# Операции без анкеты учитываем отдельно, возраст им не приписываем.
unlinked_h = (payments_h.loc[~payments_h.client_id.isin(age_h.index)]
              .groupby('period_h', observed=True).size().reindex(list(PERIODS_H), fill_value=0))
accounting_h.loc['Без анкеты: возраст неизвестен', 'Клиентов'] = pd.NA
for period in PERIODS_H:
    accounting_h.loc['Без анкеты: возраст неизвестен', period] = int(unlinked_h[period])
accounting_h.loc['Без анкеты: возраст неизвестен', 'Изменение I пол. 2025 − I пол. 2024'] = int(unlinked_h.iloc[2] - unlinked_h.iloc[0])
accounting_h.loc['Итого все успешные оплаты', list(PERIODS_H)] = (
    accounting_h.loc['Итого по всем анкетам', list(PERIODS_H)] + unlinked_h)
accounting_h.loc['Итого все успешные оплаты', 'Клиентов'] = pd.NA
accounting_h.loc['Итого все успешные оплаты', 'Изменение I пол. 2025 − I пол. 2024'] = (
    accounting_h.loc['Итого все успешные оплаты', 'I пол. 2025'] - accounting_h.loc['Итого все успешные оплаты', 'I пол. 2024'])
display(accounting_h)

# Один рисунок без повторения общей таблицы:
# слева — фиксированная переходная группа;
# справа — вклад ВСЕХ возрастов, пришедших в двух последних полугодиях.
last_two_h = list(PERIODS_H)[1:]
new_operations_h = pd.DataFrame(0, index=AGE_LABELS_H, columns=last_two_h)
for period in last_two_h:
    for age in AGE_LABELS_H:
        new_operations_h.loc[age, period] = accounting_h.loc[f'Новые {period}: {age}', 'I пол. 2025']
new_client_counts_h = pd.DataFrame(0, index=AGE_LABELS_H, columns=last_two_h)
for period in last_two_h:
    for age in AGE_LABELS_H:
        new_client_counts_h.loc[age, period] = accounting_h.loc[f'Новые {period}: {age}', 'Клиентов']
new_cohort_summary_h = pd.concat({'Клиентов': new_client_counts_h,
                                 'Оплат в I пол. 2025': new_operations_h}, axis=1)
display(new_cohort_summary_h)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(list(PERIODS_H), accounting_h.loc[senior_label_h, list(PERIODS_H)].astype(float))
axes[0].set(title=f'Достигли 65 во II пол. 2024: n={int(transition_h.sum())}', ylabel='Оплат за полугодие')
new_operations_h.plot.bar(ax=axes[1], rot=0)
axes[1].set(title='Новые всех возрастов: оплаты в I пол. 2025', xlabel='Возраст при регистрации', ylabel='Успешных оплат')
axes[1].legend(title='Когда зарегистрировались', fontsize=8)
axes[1].tick_params(axis='x', labelsize=8)
plt.tight_layout(); plt.show()

# Частичный баланс не выдаём за общебанковский.
new_total_h = int(new_operations_h.to_numpy().sum())
senior_change_h = int(accounting_h.loc[senior_label_h, 'Изменение I пол. 2025 − I пол. 2024'])
balance_summary_age5 = pd.Series({
    'Изменение оплат достигших 65': senior_change_h,
    'Оплаты новых всех возрастов из двух последних полугодий': new_total_h,
    'Частичный итог: переходная группа + эти новые клиенты': senior_change_h + new_total_h,
    'Общее изменение оплат всех клиентов с анкетами': int(accounting_h.loc['Итого по всем анкетам', 'Изменение I пол. 2025 − I пол. 2024']),
    'Общее изменение всех успешных оплат, включая без анкеты': int(accounting_h.loc['Итого все успешные оплаты', 'Изменение I пол. 2025 − I пол. 2024']),
}, name='Изменение / вклад, оплат')
display(balance_summary_age5.to_frame())

### Итог по возрасту
Гипотеза о старении клиентов и недостаточном притоке новых более молодых клиентов как объяснение нынешнего снижения активности не подходит.
Старение фиксированной базы и уменьшение наблюдаемого притока подтверждаются описательными данными, но сейчас частота падает во всех возрастных группах.

In [ ]:
# Срезы: пол, город проживания и отделение обслуживания.
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, column, title in zip(
    axes,
    ['gender', 'city', 'branch_code'],
    ['Пол', 'Город проживания', 'Отделение обслуживания'],
):
    table = (
        profile_clients5.assign(
            group=profile_clients5[column].astype('string').fillna('Нет данных')
        )
        .groupby('group')
        .agg(
            n=('client_id', 'size'),
            base=('activity_base', 'mean'),
            current=('activity_current', 'mean'),
        )
    )
    table['change_pct'] = 100 * (table.current / table.base - 1)
    display(table.rename(columns={
        'n': 'Клиентов', 'base': 'Оплат/мес. 2024',
        'current': 'Оплат/мес. 2025', 'change_pct': 'Изменение, %',
    }).round(2))

    # Для отделений показываем 15 крупнейших по числу клиентов,
    # а не 15 с самым сильным падением.
    view = table.nlargest(15, 'n').sort_values('change_pct')
    bars = ax.barh(
        [f'{group} (n={int(row.n)})' for group, row in view.iterrows()],
        view.change_pct,
        color=np.where(view.change_pct < 0, 'indianred', 'seagreen'),
    )
    ax.bar_label(bars, fmt='%+.1f%%', padding=3)
    ax.axvline(0, color='gray', linewidth=1)
    ax.set(title=title, xlabel='Изменение средней частоты, %')
    ax.margins(x=0.25)

plt.tight_layout()
plt.show()

Вывод: снижение активности очень разных городах и разных отделелиях обслуживания. Возможно это связано с пролохим обслуживанием в этих местах. То есть надо проверить связь снижения активности с оценками качества обслуживания и георафическое расположение клиентов.

In [ ]:
# Географическая гипотеза: одинаковые первые полугодия и состав клиентов.
geo5 = profile_clients5.set_index('client_id').copy()
for col in ['city', 'branch_code', 'segment', 'loyalty_tier']:
    geo5[col] = geo5[col].astype('string').fillna('Нет данных')

months5 = counts_wide.reindex(geo5.index)
geo5['base'] = months5.loc[:, '2024-01':'2024-06'].sum(axis=1) / 6
geo5['current'] = months5.loc[:, '2025-01':'2025-06'].sum(axis=1) / 6

# Карта 1: связь города проживания с 15 крупнейшими отделениями.
links5 = pd.crosstab(geo5.city, geo5.branch_code)
links5 = links5.loc[:, links5.sum().nlargest(15).index]
links5.columns = [f'{col}\nn={links5[col].sum()}' for col in links5]

# Карта 2: одинаковые сегмент и лояльность, без неизвестного города.
sample5 = geo5.loc[geo5.city.ne('Нет данных')].copy()
sample5['region'] = np.where(
    sample5.city.eq('Екатеринбург'), 'Екатеринбург', 'Остальные города'
)
table5 = sample5.groupby(
    ['segment', 'loyalty_tier', 'region'], observed=True
).agg(n=('base', 'size'), base=('base', 'mean'), current=('current', 'mean'))
table5['change'] = 100 * (
    table5.current / table5.base.replace(0, np.nan) - 1
)

regions5 = ['Екатеринбург', 'Остальные города']
changes5 = table5.change.unstack().reindex(columns=regions5)
sizes5 = table5.n.unstack().reindex_like(changes5)
keep5 = sizes5.ge(30).all(axis=1) & changes5.notna().all(axis=1)
changes5, sizes5 = changes5.loc[keep5], sizes5.loc[keep5]

labels5 = changes5.copy().astype(object)
for row in changes5.index:
    for col in regions5:
        labels5.loc[row, col] = (
            f'{changes5.loc[row, col]:+.1f}%\nn={int(sizes5.loc[row, col])}'
        )

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
sns.heatmap(links5, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            cbar_kws={'label': 'Клиентов'})
axes[0].set(title='Город проживания × отделение обслуживания',
            xlabel='15 крупнейших отделений', ylabel='')

if not changes5.empty:
    values5 = changes5.copy()
    values5.index = [f'{segment} / {loyalty}' for segment, loyalty in values5.index]
    labels5.index = values5.index
    sns.heatmap(values5, annot=labels5, fmt='', cmap='RdYlGn',
                center=0, vmin=-40, vmax=40, ax=axes[1],
                cbar_kws={'label': 'Изменение частоты, %'})
else:
    axes[1].text(.5, .5, 'Нет сочетаний с n ≥ 30 в обеих группах',
                 ha='center', va='center', transform=axes[1].transAxes)

axes[1].set(title='I пол. 2025 к I пол. 2024: одинаковый состав',
            xlabel='', ylabel='Сегмент / лояльность')
plt.tight_layout()
plt.show()

Вывод: В одном отделении обслуживаются люди с разных городов, никаких закономерностей не обнаружено. Екатеренбург больший процент по снижению активности, и основная часть находится в массовом сегменте. 

In [ ]:
# Связь оценки обслуживания с динамикой оплат внутри городов.
service5 = profile_clients5.set_index('client_id').copy()
service5['city_group'] = service5.city.astype('string').fillna('Нет данных')
service5['score_group'] = pd.cut(
    pd.to_numeric(service5.satisfaction_score, errors='coerce'),
    [0, 2, 3, 5], labels=['1–2', '3', '4–5'],
).astype('string').fillna('Нет оценки')

months5 = counts_wide.reindex(service5.index)
service5['base'] = months5.loc[:, '2024-01':'2024-06'].sum(axis=1) / 6
service5['current'] = months5.loc[:, '2025-01':'2025-06'].sum(axis=1) / 6

table5 = service5.groupby(['city_group', 'score_group'], observed=True).agg(
    n=('base', 'size'), base=('base', 'mean'), current=('current', 'mean')
)
table5['change'] = 100 * (
    table5.current / table5.base.replace(0, np.nan) - 1
)

values5 = table5.change.unstack().reindex(
    columns=['1–2', '3', '4–5', 'Нет оценки']
)
sizes5 = table5.n.unstack().reindex_like(values5)
visible5 = sizes5.ge(30) & values5.notna()
labels5 = pd.DataFrame('', index=values5.index, columns=values5.columns)

for city in values5.index:
    for score in values5.columns:
        if visible5.loc[city, score]:
            labels5.loc[city, score] = (
                f'{values5.loc[city, score]:+.1f}%\n'
                f'n={int(sizes5.loc[city, score])}'
            )

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_facecolor('#e5e7eb')

if visible5.any().any():
    sns.heatmap(
        values5, mask=~visible5, annot=labels5, fmt='',
        cmap='RdBu', center=0, vmin=-50, vmax=50,
        linewidths=.5, linecolor='#d1d5db', ax=ax,
        cbar_kws={'label': 'Изменение частоты, %', 'extend': 'both'},
    )

ax.set(
    title='Город × оценка: I полугодие 2025 к I полугодию 2024',
    xlabel='Оценка обслуживания; показаны ячейки n ≥ 30',
    ylabel='Город проживания',
)
plt.tight_layout()
plt.show()

Вывод: нет явной зависимости плохой оценкии снижения активности, и противоречие, что при оценке 3 процент снижения выше, чем при оценке 1-2, не позволяет взять это за правило. Ещё если брать отдельно Екатеренбург, то после учтения остальных признаков может оказаться в группе меньше 150 человек.

#### 5.4.5. Каналы оплат

In [ ]:
# Готовим единую совокупность успешных оплат для разложения.
# Все исходные таблицы сохраняем без изменений.
needed5 = {'client_id', 'txn_datetime', 'status', 'amount', 'channel', 'category'}
missing5 = needed5 - set(activity_payments.columns)

payments_structure5 = activity_payments.loc[
    activity_payments['client_id'].isin(profile_clients5['client_id'])
    & activity_payments['txn_datetime'].ge('2024-01-01')
    & activity_payments['txn_datetime'].lt('2025-07-01')
].copy()

payments_structure5['period5'] = np.where(
    payments_structure5['txn_datetime'].lt('2025-01-01'), 'base', 'current'
)

# Контролируем согласованность с клиентскими счётчиками главы 3.
expected_counts5 = profile_clients5.set_index('client_id')[['payments_base', 'payments_current']]
actual_counts5 = payments_structure5.groupby(['client_id', 'period5']).size().unstack(fill_value=0)
actual_counts5 = actual_counts5.reindex(index=expected_counts5.index,
                                        columns=['base', 'current'], fill_value=0)

# Функция работает одинаково для канала и категории.
def decompose_frequency5(payments, client_ids, feature):
    """  Decompose the change in payment frequency by a categorical feature (channel or category)."""
    ids = pd.Index(client_ids).drop_duplicates()
    if len(ids) == 0:
        raise ValueError('Пустая клиентская группа для разложения.')
    part = payments.loc[payments['client_id'].isin(ids)].copy()
    part['_feature5'] = part[feature].astype('string').fillna('Нет сведений')
    counts = part.groupby(['_feature5', 'period5'], observed=True).size().unstack(fill_value=0)
    counts = counts.reindex(columns=['base', 'current'], fill_value=0)
    if counts.empty:
        raise ValueError('Нет оплат в обоих периодах выбранной группы.')
    table = pd.DataFrame(index=counts.index)
    table['payments_base'] = counts['base']
    table['payments_current'] = counts['current']
    # Один знаменатель для всех строк: ВСЕ клиенты выбранной группы.
    table['frequency_base'] = counts['base'] / (12 * len(ids))
    table['frequency_current'] = counts['current'] / (6 * len(ids))
    table['change'] = table['frequency_current'] - table['frequency_base']
    table['change_pct'] = table['change'].div(table['frequency_base'].replace(0, np.nan)).mul(100)
    table['share_base_pct'] = counts['base'].div(counts['base'].sum()).mul(100)
    table['share_current_pct'] = counts['current'].div(counts['current'].sum()).mul(100)
    table['share_change_pp'] = table['share_current_pct'] - table['share_base_pct']
    gross_loss = -table['change'].clip(upper=0).sum()
    # Доли потерь имеют знаменателем только сумму отрицательных вкладов.
    table['share_of_gross_loss_pct'] = (-table['change'].clip(upper=0)).div(
        gross_loss if gross_loss else np.nan
    ).mul(100)
    table.index.name = feature
    expected_delta = (part['period5'].eq('current').sum() / (6 * len(ids))
                      - part['period5'].eq('base').sum() / (12 * len(ids)))
    return table.sort_values('change')


def structure_result5(table, title):
    """Print a textual summary of the decomposition produced by  `decompose_frequency5`."""
    losses = -table['change'].clip(upper=0).sum()
    growth = table['change'].clip(lower=0).sum()
    compensation = 100 * growth / losses if losses > 0 else np.nan
    worst = table['change'].idxmin()
    display((
        f'Общее изменение частоты: {table.change.sum():+.3f} оплаты на клиента в месяц.'
        f' Сумма отрицательных вкладов: {losses:.3f}; положительных: {growth:.3f}.'
        + (f'Рост компенсирует {compensation:.2f}% отрицательных вкладов. ' if losses > 0
           else 'Отрицательных вкладов нет. ')
        + (f'Наибольший отрицательный вклад — «{worst}»: {table.loc[worst, "change"]:.3f}. '
           if table.loc[worst, 'change'] < 0 else '')
        + 'Это разложение наблюдаемого изменения, не доказанная причина поведения.'
    ))
    shift = table.loc[table['change'].lt(0) & table['share_change_pp'].gt(0)]
    if not shift.empty:
        display(('Доля выросла при снижении частоты: ' + ', '.join(map(str, shift.index))
                         + '. Это относительное перераспределение, а не рост использования.'))

def draw_structure5(table, title):
    """Visualise the decomposition produced by `decompose_frequency5`"""
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4.5, len(table)*.45)))
    colors = ['indianred' if x < 0 else 'steelblue' for x in table['change']]
    bars = axes[0].barh(table.index.astype(str), table['change'], color=colors)
    axes[0].bar_label(bars, labels=[f'{x:+.3f}' for x in table['change']], padding=3)
    axes[0].axvline(0, color='gray', linewidth=1)
    axes[0].set(title='Вклад в общее изменение', xlabel='Оплат на клиента в месяц')
    positions = np.arange(len(table))
    axes[1].barh(positions-.18, table['share_base_pct'], height=.35, label='2024')
    axes[1].barh(positions+.18, table['share_current_pct'], height=.35, label='I полугодие 2025')
    axes[1].set(yticks=positions, yticklabels=table.index.astype(str),
                title='Структура количества оплат', xlabel='Доля всех оплат группы, %', xlim=(0,100))
    axes[1].legend()
    for ax in axes:
        ax.invert_yaxis()
        sns.despine(ax=ax)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

#### 5.4.6. Вклад каналов и структура количества оплат

In [ ]:
structure_labels5 = {
    'payments_base': 'Оплат за 2024', 'payments_current': 'Оплат за I полугодие 2025',
    'frequency_base': 'Частота 2024, оплат/мес.',
    'frequency_current': 'Частота 2025, оплат/мес.',
    'change': 'Вклад в изменение, оплат/мес.', 'change_pct': 'Изменение частоты, %',
    'share_base_pct': 'Доля оплат 2024, %', 'share_current_pct': 'Доля оплат 2025, %',
    'share_change_pp': 'Изменение доли, п.п.',
    'share_of_gross_loss_pct': 'Доля отрицательных вкладов, %',
}

channel_structure5 = decompose_frequency5(payments_structure5, profile_clients5.client_id, 'channel')
display(channel_structure5.rename(columns=structure_labels5).round(3))
draw_structure5(channel_structure5, f'Каналы: фиксированная группа, n={len(profile_clients5):,}')
structure_result5(channel_structure5, 'каналы')


вывод:
больше всего снижений количества оплат по карте, а токен наоборот растёт и компенсирует частично снижение по остальным каналам, но не достаточно, чтобы сказать, что люди просто меняют канал оплаты. 
Решение: нужно проверить не только успешные операции, но и отклонённые, может их количество возрасло и люди перестали платить картой. Также нужно проверить является ли рост токенов при падении по остальным каналам просто заменой одного канала другим.

In [ ]:
# Проверяем расло ли количество отказов по каналам.
# Используем таблицу всех статусов после очистки, а не только успешные оплаты.
attempts_source5 = transactions_period.loc[
    transactions_period['client_id'].isin(profile_clients5['client_id'])
    & transactions_period['txn_datetime'].ge('2024-01-01')
    & transactions_period['txn_datetime'].lt('2025-07-01')
].copy()
# Нормализуем статус только в рабочей копии для проверки.
attempt_status5 = attempts_source5['status'].astype('string').str.strip().str.casefold()
# Неизвестные статусы нельзя незаметно пропускать.
unknown_status5 = attempt_status5.isna() | ~attempt_status5.isin(['успешно', 'отклонено', 'возврат'])
if unknown_status5.any():
    raise ValueError(
        'В проверке отказов есть неизвестные статусы. '
        'Сначала уточните их в главе очистки.'
    )
# Возвраты исключаем; сумму отказа не используем как условие включения.
# Иначе можно потерять попытки, у которых сумма записана иначе.
attempts5 = attempts_source5.loc[attempt_status5.isin(['успешно', 'отклонено'])].copy()
attempts5['status_view5'] = attempt_status5.loc[attempts5.index]
if attempts5.empty:
    raise ValueError('Нет успешных или отклонённых операций для анализа.')
if attempts5['txn_id'].isna().any() or attempts5['txn_id'].duplicated().any():
    raise ValueError('Проверьте заполненность и уникальность txn_id.')
# Сохраняем неизвестный канал отдельной группой.
attempts5['channel_view5'] = attempts5['channel'].astype('string').fillna('Нет сведений')
attempts5['period5'] = np.where(
    attempts5['txn_datetime'].lt('2025-01-01'),
    '2024',
    '2025: январь–июнь',
)
# Подсчитываем строки-операции, а не денежные суммы.
attempt_counts5 = attempts5.groupby(
    ['channel_view5', 'period5', 'status_view5'], observed=True,
).size()
# Добавляем отсутствующие сочетания с нулевыми количествами.
channel_order5 = [
    channel for channel in ['Карта', 'Токен', 'СБП', 'Онлайн-банк']
    if channel in attempts5['channel_view5'].unique()
]
channel_order5 += sorted(set(attempts5['channel_view5']) - set(channel_order5))
period_order5 = ['2024', '2025: январь–июнь']
complete_index5 = pd.MultiIndex.from_product(
    [channel_order5, period_order5, ['успешно', 'отклонено']],
    names=['channel_view5', 'period5', 'status_view5'],
)
attempt_summary5 = (
    attempt_counts5
    .reindex(complete_index5, fill_value=0)
    .unstack('status_view5')
    .rename(columns={'успешно': 'successful', 'отклонено': 'declined'})
)
attempt_summary5.columns.name = None
# Считаем общие попытки и доли отказов/успехов.
# Если попыток нет, доля не определена, а не равна нулю.
attempt_summary5['attempts'] = attempt_summary5['successful'] + attempt_summary5['declined']
attempt_summary5['decline_rate_pct'] = (
    attempt_summary5['declined']
    .div(attempt_summary5['attempts'].replace(0, np.nan))
    .mul(100)
)
attempt_summary5['success_rate_pct'] = (
    attempt_summary5['successful']
    .div(attempt_summary5['attempts'].replace(0, np.nan))
    .mul(100)
)
# Окна разной длины: частоты приводим к месяцу и одному клиенту.
period_months5 = (
    attempt_summary5.index
    .get_level_values('period5')
    .map({'2024': 12, '2025: январь–июнь': 6})
    .to_numpy()
)
n_attempt_clients5 = len(profile_clients5)
attempt_summary5['attempt_frequency'] = attempt_summary5['attempts'] / period_months5 / n_attempt_clients5
attempt_summary5['successful_frequency'] = attempt_summary5['successful'] / period_months5 / n_attempt_clients5
attempt_summary5['declined_frequency'] = attempt_summary5['declined'] / period_months5 / n_attempt_clients5
# Проверяем, что успешные операции соответствуют предыдущему разложению.
# Если проверка не проходит, нельзя молча сравнивать разные определения успеха.
successful_check5 = attempt_summary5['successful'].unstack('period5').reindex(channel_structure5.index)

attempt_labels5 = {
    'successful': 'Успешных операций', 'declined': 'Отклонённых операций',
    'attempts': 'Всего попыток', 'decline_rate_pct': 'Доля отказов, %',
    'success_rate_pct': 'Доля успешных, %',
    'attempt_frequency': 'Попыток на клиента в месяц',
    'successful_frequency': 'Успешных на клиента в месяц',
    'declined_frequency': 'Отказов на клиента в месяц',
}
display(
    attempt_summary5
    .rename(columns=attempt_labels5)
    .rename_axis(index=['Канал', 'Период'])
    .round(3)
)
# Строим один рисунок:
# слева — доля отказов, справа — частота попыток и успешных оплат.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
positions5 = np.arange(len(channel_order5))
width5 = 0.36
colors5 = ['steelblue', 'darkorange']
for i, period in enumerate(period_order5):
    view5 = attempt_summary5.xs(period, level='period5').reindex(channel_order5)
    # Числа попыток подписываем, чтобы был виден знаменатель процентов.
    bars5 = axes[0].bar(
        positions5 + (i - 0.5) * width5, view5['decline_rate_pct'],
        width=width5, color=colors5[i], label=period,
    )
    axes[0].bar_label(
        bars5,
        labels=[
            f'{rate:.2f}%\nn={int(count):,}' if pd.notna(rate) else 'Нет попыток'
            for rate, count in zip(view5['decline_rate_pct'], view5['attempts'])
        ],
        padding=3, fontsize=8,
    )
    axes[1].plot(
        positions5, view5['attempt_frequency'],
        marker='o', linestyle='--', color=colors5[i],
        label=f'{period}: все попытки',
    )
    axes[1].plot(
        positions5, view5['successful_frequency'],
        marker='o', linestyle='-', color=colors5[i],
        label=f'{period}: успешные',
    )
axes[0].set(
    title='Доля отклонённых среди попыток',
    xlabel='Канал',
    ylabel='Отклонённые / (успешные + отклонённые), %',
)
# Оставляем место для подписей над столбцами.
max_decline_rate5 = attempt_summary5['decline_rate_pct'].max()
axes[0].set_ylim(0, max(5, max_decline_rate5 * 1.4))
axes[0].legend(title='Период')
axes[1].set(
    title='Частота попыток и успешных оплат',
    xlabel='Канал',
    ylabel='Операций на клиента в месяц',
)
axes[1].set_ylim(bottom=0)
axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xticks(positions5)
    ax.set_xticklabels(channel_order5)
    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)
fig.suptitle(
    f'Отказы и использование каналов; '
    f'фиксированная группа n={n_attempt_clients5:,}'
)
plt.tight_layout()
plt.show()
# Печатаем фактическое изменение для каждого канала.
base_attempts5 = attempt_summary5.xs('2024', level='period5')
current_attempts5 = attempt_summary5.xs('2025: январь–июнь', level='period5')
for channel in channel_order5:
    before5 = base_attempts5.loc[channel]
    after5 = current_attempts5.loc[channel]
    rate_change5 = after5['decline_rate_pct'] - before5['decline_rate_pct']
    

вывод:
количество отклонений не растёт, поэтому причина снижения количества по карте не установлена.

In [ ]:
# Проверяем совместные изменения каналов на уровне клиента.
# Не называем их подтверждённым переносом оплат: назначения операций неизвестны.
ids5 = profile_clients5['client_id']
channels5 = channel_structure5.index
channel_client_counts5 = payments_structure5.assign(
    channel_view5=payments_structure5['channel'].astype('string').fillna('Нет сведений')
).groupby(['client_id', 'channel_view5', 'period5']).size()
full_index5 = pd.MultiIndex.from_product(
    [ids5, channels5, ['base', 'current']], names=['client_id', 'channel_view5', 'period5']
)
channel_counts_complete5 = channel_client_counts5.reindex(full_index5, fill_value=0).unstack('period5')
client_channel_change5 = (channel_counts_complete5['current']/6
                           - channel_counts_complete5['base']/12).unstack('channel_view5')
cochange_rows5 = []
for source in channels5:
    reduced = client_channel_change5[source].lt(0)
    for destination in channels5:
        if source == destination:
            continue
        increased = client_channel_change5[destination].gt(0)
        cochange_rows5.append({
            'Канал снижения': source, 'Канал роста': destination,
            'Снижение исходного, клиентов': int(reduced.sum()),
            'Одновременно снижение и рост': int((reduced & increased).sum()),
            'Среди снизивших исходный канал, %': 100*(reduced & increased).sum()/reduced.sum()
                if reduced.any() else np.nan,
        })
channel_cochange5 = pd.DataFrame(cochange_rows5)
display(channel_cochange5.round(2))



вывод:
одновременное снижение одного и рост другого канала считаем кандидатом на изменение структуры использования.
Решение: используем совместные изменения как признак возможной перестройки использования каналов, не как доказанную миграцию.

#### 5.4.7. Категории покупок

In [ ]:
# Сохраняем исходное разложение числа оплат по всем валютам.
category_structure5 = decompose_frequency5(
    payments_structure5, profile_clients5.client_id, 'category')
display(category_structure5.rename(columns=structure_labels5).round(3))
draw_structure5(category_structure5,
                f'Категории: фиксированная группа, n={len(profile_clients5):,}')
structure_result5(category_structure5, 'категории')

# денежное сравнение только внутри одной валюты.
CURRENCY5 = 'RUB'
ids_money5 = pd.Index(profile_clients5['client_id'])
n_money5 = len(ids_money5)
if n_money5 == 0:
    raise ValueError('Пустая клиентская группа.')
required_money5 = {'client_id', 'category', 'currency', 'amount', 'txn_datetime', 'status', 'txn_id'}
missing_money5 = required_money5 - set(payments_structure5.columns)
if missing_money5:
    raise ValueError(f'Нет полей: {sorted(missing_money5)}')

# Показываем валютный состав до отбора, не маскируя исключённые валюты.
display(payments_structure5['currency'].value_counts(dropna=False).to_frame('Успешных оплат'))
money_payments5 = payments_structure5.loc[
    payments_structure5['client_id'].isin(ids_money5)
    & payments_structure5['currency'].astype('string').str.strip().str.upper().eq(CURRENCY5).fillna(False)
    & payments_structure5['txn_datetime'].ge('2024-01-01')
    & payments_structure5['txn_datetime'].lt('2025-07-01')
].copy()

money_payments5['category_view5'] = money_payments5['category'].astype('string').fillna('Нет сведений')
money_payments5['period_money5'] = np.where(
    money_payments5['txn_datetime'].lt('2025-01-01'), 'base', 'current')

# Количество, сумма и медианный чек по категории и периоду.
# Пропущенные сочетания означают отсутствие операций, не отсутствие данных о сумме.
agg_money5 = money_payments5.groupby(['category_view5', 'period_money5'], observed=True).agg(
    count=('amount', 'size'), total=('amount', 'sum'), median_ticket=('amount', 'median'))
categories_money5 = sorted(money_payments5['category_view5'].unique())
full_money5 = pd.MultiIndex.from_product(
    [categories_money5, ['base', 'current']], names=['category_view5', 'period_money5'])
agg_money5 = agg_money5.reindex(full_money5)
agg_money5[['count', 'total']] = agg_money5[['count', 'total']].fillna(0)
agg_money5['mean_ticket'] = agg_money5['total'].div(agg_money5['count'].replace(0, np.nan))
base_money5 = agg_money5.xs('base', level='period_money5')
current_money5 = agg_money5.xs('current', level='period_money5')
category_money5 = pd.DataFrame(index=base_money5.index)
for period, table, months in [('base', base_money5, 12), ('current', current_money5, 6)]:
    category_money5[f'frequency_{period}'] = table['count'] / (months*n_money5)
    category_money5[f'spending_{period}'] = table['total'] / (months*n_money5)
    category_money5[f'mean_ticket_{period}'] = table['mean_ticket']
    category_money5[f'median_ticket_{period}'] = table['median_ticket']
    # Проверяем тождество: месячная сумма = частота × средний чек.
    mask = table['count'].gt(0)
for metric in ['frequency', 'spending', 'mean_ticket']:
    category_money5[f'{metric}_change_pct'] = (
        category_money5[f'{metric}_current']
        .div(category_money5[f'{metric}_base'].replace(0, np.nan)).sub(1).mul(100))
labels_money5 = {
    'frequency_base':'Частота 2024, оплат/клиента/мес.',
    'frequency_current':'Частота 2025, оплат/клиента/мес.',
    'spending_base':f'Сумма 2024, {CURRENCY5}/клиента/мес.',
    'spending_current':f'Сумма 2025, {CURRENCY5}/клиента/мес.',
    'mean_ticket_base':f'Средний чек 2024, {CURRENCY5}',
    'mean_ticket_current':f'Средний чек 2025, {CURRENCY5}',
    'median_ticket_base':f'Медианный чек 2024, {CURRENCY5}',
    'median_ticket_current':f'Медианный чек 2025, {CURRENCY5}',
    'frequency_change_pct':'Изменение частоты, %',
    'spending_change_pct':'Изменение месячной суммы, %',
    'mean_ticket_change_pct':'Изменение среднего чека, %',
}
display(category_money5.rename(columns=labels_money5).round(2))

# Три согласованные панели: сравниваем изменения, а не суммы за 12 и 6 месяцев.
fig, axes = plt.subplots(1, 3, figsize=(18, max(5, len(category_money5)*.5)))
for ax, metric, title in zip(axes, ['frequency','spending','mean_ticket'],
                            ['Частота оплат','Месячная сумма успешных оплат','Средний чек']):
    values = category_money5[f'{metric}_change_pct']
    bars = ax.barh(category_money5.index, values,
                   color=['indianred' if x < 0 else 'steelblue' for x in values])
    ax.bar_label(bars, labels=[f'{x:+.1f}%' if pd.notna(x) else '' for x in values], padding=3)
    ax.axvline(0, color='gray', linewidth=1)
    ax.invert_yaxis()
    ax.set(title=title, xlabel='Изменение относительно 2024 года, %')
    sns.despine(ax=ax)
fig.suptitle(f'Категории, только {CURRENCY5}; фиксированная группа n={n_money5:,}')
plt.tight_layout()
plt.show()


# Печатаем результат каждой категории без заранее придуманных чисел.
for category, row in category_money5.iterrows():
    
    if row.frequency_current < row.frequency_base:
        if row.spending_current >= row.spending_base:
            pass
        elif row.mean_ticket_current > row.mean_ticket_base:
            pass
        else:
            pass

# Проверяем гипотезу у одних и тех же клиентов в каждой категории.
# Анализ только клиентов с оплатами категории в базе; новые пользователи отдельно.
client_money5 = money_payments5.groupby(
    ['client_id','category_view5','period_money5'], observed=True).agg(
        count=('amount','size'), total=('amount','sum'))
base_client5 = client_money5.xs('base', level='period_money5') if 'base' in client_money5.index.get_level_values('period_money5') else pd.DataFrame(columns=['count','total'])
current_client5 = client_money5.xs('current', level='period_money5') if 'current' in client_money5.index.get_level_values('period_money5') else pd.DataFrame(columns=['count','total'])
if base_client5.empty:
    raise ValueError('Нет базовых оплат для индивидуального сравнения.')
paired_money5 = base_client5.rename(columns={'count':'count_base','total':'total_base'}).join(
    current_client5.rename(columns={'count':'count_current','total':'total_current'}), how='left').fillna(0)
# Для количеств сравниваем целые счётчики: current/6 < base/12.
paired_money5['fewer_payments'] = paired_money5['count_current'].mul(2).lt(paired_money5['count_base'])
# Денежное равенство учитываем с малым численным допуском, не округляя до рублей.
paired_money5['spending_not_lower'] = (
    paired_money5['total_current'].mul(2).ge(paired_money5['total_base'])
    | np.isclose(paired_money5['total_current']*2, paired_money5['total_base'], rtol=1e-10, atol=1e-6))
paired_money5['fewer_but_spending_not_lower'] = paired_money5['fewer_payments'] & paired_money5['spending_not_lower']
client_hypothesis5 = paired_money5.groupby(level='category_view5').agg(
    base_clients=('count_base','size'),
    fewer_clients=('fewer_payments','sum'),
    fewer_spending_not_lower=('fewer_but_spending_not_lower','sum'))
client_hypothesis5['share_among_fewer_pct'] = (
    client_hypothesis5['fewer_spending_not_lower']
    .div(client_hypothesis5['fewer_clients'].replace(0,np.nan)).mul(100))
client_hypothesis5['new_category_clients'] = [
    len(set(current_client5.loc[current_client5.index.get_level_values('category_view5')==category].index.get_level_values('client_id'))
        - set(base_client5.loc[base_client5.index.get_level_values('category_view5')==category].index.get_level_values('client_id')))
    for category in client_hypothesis5.index]
display(client_hypothesis5.rename(columns={
    'base_clients':'Клиентов с категорией в базе',
    'fewer_clients':'Клиентов с меньшей частотой',
    'fewer_spending_not_lower':'Реже, но месячная сумма не ниже',
    'share_among_fewer_pct':'Среди снизивших частоту категории, %',
    'new_category_clients':'Новых клиентов категории в текущем периоде',
}).round(2))
for category,row in client_hypothesis5.iterrows():
    pass



вывод:
сумма покупок уменьшилась вместе с количеством оплат, а также уменьшилась даже сумма среднего чека, что свидетельсвует об отвержении гипотезы: реже стали покупать, но просто на более большую сумму(поход в магазин не 3 раза в неделю, а один на всю неделю)

#### 5.4.8. Целевые пересечения и структура потенциальных групп

In [ ]:
# Пересекающиеся разрезы строим только для выбранных вопросов.
cross_tables5 = {}
pairs5 = [
    ('segment', 'has_mobile_app', 'Сегмент × приложение'),
    ('age_group5', 'has_mobile_app', 'Возраст × приложение'),
    ('segment', 'loyalty_tier', 'Сегмент × лояльность'),
]
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for ax, (row_col, col_col, title) in zip(axes, pairs5):
    work = profile_clients5.assign(
        _row5=profile_clients5[row_col].astype('string').fillna('Нет сведений'),
        _col5=profile_clients5[col_col].astype('string').fillna('Нет сведений'))
    table = work.groupby(['_row5', '_col5'], observed=True).agg(
        n=('client_id', 'size'), declined=('activity_declined', 'sum'))
    table['decline_pct'] = 100*table['declined']/table['n']

    cross_tables5[title] = table
    rates, counts = table['decline_pct'].unstack(), table['n'].unstack()
    annotations = rates.copy().astype(object)
    for row in rates.index:
        for col in rates.columns:
            value, n = rates.loc[row,col], counts.loc[row,col]
            annotations.loc[row,col] = '' if pd.isna(value) else f'{value:.1f}%\nn={int(n)}' + (' *' if n<150 else '')
    sns.heatmap(rates, annot=annotations, fmt='', cmap='Blues', vmin=0, vmax=100,
                mask=rates.isna(), cbar=False, ax=ax)
    ax.set(title=title, xlabel=col_col, ylabel=row_col)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.suptitle('Доля снижения внутри сочетаний; * менее 150 клиентов (не критерий достоверности)')
plt.tight_layout()
plt.show()

# Потенциальные направления определяем явными правилами, не по максимальной ячейке.
segment_text5 = profile_clients5['segment'].astype('string')
app_text5 = profile_clients5['has_mobile_app'].astype('string')
decline_mask5 = profile_clients5['activity_declined']
structure_groups5 = {
    'Все со снижением': profile_clients5.loc[decline_mask5, 'client_id'],
    'Массовый без приложения, снижение': profile_clients5.loc[
        decline_mask5 & segment_text5.eq('Массовый').fillna(False)
        & app_text5.eq('Нет').fillna(False), 'client_id'],
    'Массовый с приложением, снижение': profile_clients5.loc[
        decline_mask5 & segment_text5.eq('Массовый').fillna(False)
        & app_text5.eq('Да').fillna(False), 'client_id'],
}
# Группы пересекаются с «Все со снижением»; их размеры не складываем.
group_structure5 = {}
for feature in ['channel', 'category']:
    rows = []
    for name, ids in structure_groups5.items():
        if len(ids) == 0:
            continue
        table = decompose_frequency5(payments_structure5, ids, feature)
        group_structure5[(name, feature)] = table
        rows.append(table['change'].rename(name))
        display((f' {name}; n={len(ids):,}; разложение по {feature}'))
        display(table.rename(columns=structure_labels5).round(3))
        structure_result5(table, name)
    if rows:
        matrix = pd.concat(rows, axis=1).fillna(0)
        limit = max(float(matrix.abs().max().max()), .001)
        fig, ax = plt.subplots(figsize=(11, max(4, len(matrix)*.5)))
        sns.heatmap(matrix, annot=True, fmt='.3f', cmap='RdBu', center=0,
                    vmin=-limit, vmax=limit, cbar_kws={'label':'Изменение, оплат/клиента/мес.'}, ax=ax)
        ax.set(title=f'Состав изменения по {feature}: сравнение аналитических групп',
               xlabel='Группа', ylabel=feature)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
        plt.tight_layout()
        plt.show()


вывод:
наиболее выраженное и широкое снижение активности наблюдается в массовои сегменте, особенно у клиентов без мобильного приложения. 
Решение: кандидат на дальнейшее рассмотрение и анализ массовый сегмент

#### 5.4.9. Согласие на коммунникацию

In [ ]:
# Сначала явно проверяем допустимые значения согласия.
consent_text5 = profile_clients5['marketing_opt_in'].astype('string').str.strip().str.casefold()
display(consent_text5.value_counts(dropna=False).to_frame('Клиентов'))
unknown_consent5 = consent_text5.notna() & ~consent_text5.isin(['да', 'нет'])

# Сохраняем признак только в рабочей таблице главы 5.
profile_clients5['contact_allowed5'] = consent_text5.eq('да').fillna(False)
declined_clients5 = profile_clients5.loc[profile_clients5['activity_declined']].copy()
n_contact5 = int(declined_clients5['contact_allowed5'].sum())
display((
    f'Среди {d5:,} клиентов со снижением '
    f'явное согласие есть у {n_contact5:,}; '
    f'без подтверждённого согласия — {d5-n_contact5:,}. '
    f'Даже до выбора групп верхняя граница размера кампании — '
    f'{min(n_contact5, 2000):,} клиентов. '
    'Это ограничение охвата, а не готовый список кампании.'
))

# Признак decline_profile создавался только в activity_dynamics.
# Не пытаемся искать его в client_activity или назначать заново.
if 'decline_profile' not in declined_clients5:
    raise ValueError('Сначала выполните профили снижения главы 4.')
if declined_clients5['decline_profile'].isna().any():
    raise ValueError('Не всем снизившим назначен диагностический профиль.')
contact_profile5 = declined_clients5.groupby('decline_profile', observed=True).agg(
    n=('client_id', 'size'), contact_allowed=('contact_allowed5', 'sum'),
)
contact_profile5['contact_pct'] = 100 * contact_profile5['contact_allowed'] / contact_profile5['n']
display(contact_profile5.round(2))


Решение: при дальнейшем выборе групп используем только явное согласие.

In [ ]:
# Журналируем только создание рабочих полей и аналитических таблиц одной записью.
if 'add_log' not in globals():
    raise RuntimeError('Выполните инициализацию журнала.')
add_log(
    key='clients_activity_profiles_chapter5',
    field='age_group5; income_group5; contact_allowed5; period5; channel/category decomposition',
    issue='Нужно описать клиентские различия и разложить изменение частоты',
    scope='Сопоставимые клиенты с payments_base > 0; связанные успешные оплаты 2024-01-01 .. 2025-06-30',
    n_rows=len(profile_clients5),
    value_before='Дополнительные интервалы и таблицы разложения отсутствовали',
    action='Созданы рабочие признаки и агрегаты; исходные таблицы сохранены',
    value_after=(f'{len(profile_clients5)} клиентов; {len(payments_structure5)} оплат; '
                 f'каналов {len(channel_structure5)}, категорий {len(category_structure5)}; '
                 f'согласие среди снизивших {n_contact5}; удалено 0 строк'),
    rationale=('Частоты: количество / 12 или 6 / фиксированное число клиентов, включая нули. '
               'Вклады суммируются в общее изменение; неизвестные категории сохранены. '
               'Проверены клиентские счётчики. Пересечения выбраны по гипотезам, не полным перебором. '
               'Признаки созданы в рабочих копиях. Причины и переходы каналов не доказаны; '
               'окончательная кампания не назначена.'),
)


#### 5.4.10. Итог 
- На 5240 клиентах чистое снижение −1,242 оплаты на клиента в месяц. Основной канал потерь — карта (−0,984), токен растёт (+0,267) и компенсирует 17,70% отрицательных канальных вкладов.
- Все 9 категорий трат падают. Больше всего продукты
- Наибольший процент клиентов со снижением операций в массовом сегменте, особенно без приложения.

## 6. Метрика активности

### 6.1. Что считаем активностью и почему

Активность — среднее число успешных положительных конечных оплат клиента в месяц. Одна строка успешной операции даёт одну оплату независимо от суммы. Считаем по всем каналам: карта, токен, СБП, онлайн-банк; количество операций разных валют объединяем, денежные суммы — нет.

Выбор отвечает задаче «стали реже пользоваться», а не «стали меньше тратить». Количество не зависит напрямую от стоимости покупки. Недостаток — одинаково учитывает маленькую и большую оплату, а одна потребность может оформляться несколькими операциями. Данные отражают только доступную выгрузку банка. Анализ идёт всегда за месяц не учитывает варьирование активности в течении месяца. 

Расчёт показателей уже выполнен в 5.3 перед их использованием в графиках. 

### 6.2. Периоды и формулы

Базовый период — [01.01.2024; 01.01.2025), 12 месяцев. Текущий период — [01.01.2025; 01.07.2025), 6 месяцев. Конец периода — апрель–июнь 2025, 3 месяца.

- activity_base = payments_base / 12.
- activity_current = payments_current / 6.
- activity_change = activity_current − activity_base.
- activity_change_pct = (activity_current / activity_base − 1) × 100 при положительной сопоставимой базе.
- Строгое снижение: 2 × payments_current < payments_base; равенство не считается снижением.
- Частота II квартала ниже базы: 4 × payments_q2_2025 < payments_base.


### 6.3. Клиенты и доступность истории

Сопоставимая история — известная дата регистрации не позднее 01.01.2024.

В сохранённых результатах 5647 анкет, из них 5542 сопоставимы; 105 зарегистрированы позднее начала базы. В фиксированной группе сохраняем клиентов, в которой список клиентов не меняется. Индивидуальный процент считаем только для 5240 с положительной базой; у 302 база нулевая. Они не должны автоматически попадать в «без снижения» при сравнении процентов.

Операции без найденной анкеты остаются в операционном анализе, но не участвуют в клиентской кампании: нет подтверждённого согласия.

### 6.4. Нулевая база и отсутствие текущих оплат

При положительной базе и нуле текущих оплат процент равен −100%. При нулевой базе процент не определён; такие клиенты имеют отдельное состояние.

Окончательное отсутствие оплат в квартале или полугодии — результат наблюдения(нельзя доказать уход из банка).

### 6.5. Проверка реализации без пересчёта исходных таблиц


In [ ]:
#  проверяем существующую метрику по счётчикам и помесячной таблице.
metric_audit6 = client_activity.set_index('client_id').copy()
months_metric6 = pd.period_range('2024-01','2025-06',freq='M')
monthly_metric6 = counts_wide.reindex(index=metric_audit6.index,columns=months_metric6)
comparable_metric6 = metric_audit6.activity_comparable.fillna(False)

base_count_metric6 = monthly_metric6.loc[:,'2024-01':'2024-12'].sum(axis=1)
current_count_metric6 = monthly_metric6.loc[:,'2025-01':'2025-06'].sum(axis=1)

positive_metric6 = comparable_metric6 & base_count_metric6.gt(0)

expected_decline6 = positive_metric6 & current_count_metric6.mul(2).lt(base_count_metric6)

metric_summary6 = pd.Series({
    'Анкет': len(metric_audit6), 'Сопоставимых': int(comparable_metric6.sum()),
    'Недостаточно истории': int((~comparable_metric6).sum()),
    'Сопоставимых с положительной базой': int(positive_metric6.sum()),
    'Сопоставимых с нулевой базой': int((comparable_metric6 & base_count_metric6.eq(0)).sum()),
    'Со снижением': int(expected_decline6.sum()),
    'Доля снизивших среди положительной базы, %':100*expected_decline6.sum()/positive_metric6.sum(),
    'Нет текущих оплат при положительной базе':int((positive_metric6 & current_count_metric6.eq(0)).sum()),
},name='Значение')
display(metric_summary6.to_frame().round(2))
display(metric_audit6.loc[comparable_metric6,'activity_state'].value_counts().to_frame('Клиентов'))


Вывод: по сохранённым результатам снижение есть у 3560 из 5240 клиентов — 67,94%. У пяти из них нет оплат всё текущее полугодие. Основная наблюдаемая картина — снижение частоты, а не массовое полное прекращение оплат.

### 6.6. Календарная проверка и устойчивость

В разделе 5 сравнивались январь–июнь 2024 и 2025 на том же составе 5542 клиентов. Снижение среднего уровня — 10,81%, все шесть соответствующих месяцев ниже.

У 4867 клиентов с положительной базой в обоих определениях формы совпадают в 4300 случаях (88,35%), расходятся в 567 (11,65%). Но не доказанная сезонная модель из-за малого периода исследования.

### 6.7. Что метрика не увидит

- Меньшую сумму при прежнем количестве оплат; отдельно исследованы номинальные RUB-суммы и чеки.
- Расходы вне банка, визиты в магазин и физический объём покупок.
- Причины отказов, если в данных нет кода причины, и покупки, которых не было в выгрузке.
- Изменение внутри месяцев при одинаковом месячном количестве.

### 6.8. Решение и журнал

Используем метрику совместно с концом периода, каналами, категориями, денежным сравнением и согласием. Определение и создание метрик уже отражены в clients_activity_metrics, дополнительные профили — в clients_activity_dynamics_profiles. Сам контроль формул и этот текст не меняют исходные данные, отдельная дублирующая запись журнала не нужна.


## 7. Поиск и описание групп

### 7.0. Подготовка диагностических признаков и правила групп
Общие условия: сопоставимая положительная база, снижение за полугодие, частота II квартала ниже базы. Восстановившихся во II квартале не включаем в эту реактивационную кампанию.

Правила: 
- G1 — массовый сегмент без приложения; 
- G2 — с приложением, карта даёт не менее 50% отрицательных канальных вкладов и не компенсирована ростом других каналов; 
- G3 — преобладающее сокращение повседневных категорий вместе с падением месячной RUB-суммы. Порог 50% задаёт преобладание компонента. 

In [ ]:
BUDGET6 = 2000
MIN_GROUP6 = 150
BASE_START6 = pd.Timestamp('2024-01-01')
CURRENT_START6 = pd.Timestamp('2025-01-01')
END6 = pd.Timestamp('2025-07-01')

# Половина отрицательных вкладов означает «преобладающую» составляющую.
# Это явное аналитическое определение, не подобранный по размеру кампании порог.
DOMINANT_SHARE6 = 0.50
EVERYDAY6 = ['Продукты', 'Транспорт', 'Аптеки', 'Связь и Интернет']


campaign_features6 = client_activity.copy()
ids6 = pd.Index(campaign_features6.client_id)
idx6 = campaign_features6.set_index('client_id')
comparable6 = idx6.activity_comparable.fillna(False)
positive6 = comparable6 & idx6.payments_base.gt(0)
months6 = pd.period_range('2024-01','2025-06',freq='M')
monthly6 = counts_wide.reindex(index=ids6,columns=months6)

q2_count6 = monthly6.loc[:,'2025-04':'2025-06'].sum(axis=1)
idx6['q2_frequency6'] = q2_count6.div(3)

# Целочисленное сравнение исключает погрешность деления.
idx6['persistent_decline6'] = positive6 & q2_count6.mul(4).lt(idx6.payments_base)
idx6['q2_restored6'] = positive6 & q2_count6.mul(4).ge(idx6.payments_base)
consent6 = idx6.marketing_opt_in.astype('string').str.strip().str.casefold()
idx6['contact_allowed6'] = consent6.eq('да').fillna(False).astype(bool)

# Актуальные успешные оплаты выбранных клиентов: количество по всем валютам.
pay6 = activity_payments.loc[
    activity_payments.client_id.isin(ids6)
    & activity_payments.txn_datetime.ge(BASE_START6)
    & activity_payments.txn_datetime.lt(END6)
].copy()

pay6['period6'] = np.where(pay6.txn_datetime.lt(CURRENT_START6),'base','current')

# Разложение на уровне клиента: нули каналов и категорий сохраняются.
def client_decomposition6(feature):
    """Decomposes per-client monthly frequency change by a categorical feature (channel/category), preserving zeros for missing values and asserting that component deltas sum to the overall change."""
    work = pay6.assign(_dimension6=pay6[feature].astype('string').fillna('Нет сведений'))
    names = sorted(work['_dimension6'].unique())
    counts = work.groupby(['client_id','_dimension6','period6'], observed=True).size()
    full = pd.MultiIndex.from_product([ids6,names,['base','current']],
        names=['client_id','_dimension6','period6'])
    wide = counts.reindex(full,fill_value=0).unstack('period6').reindex(columns=['base', 'current'])
    base = wide.base.div(12).unstack('_dimension6').reindex(index=ids6, columns=names)
    current = wide.current.div(6).unstack('_dimension6').reindex(index=ids6, columns=names)
    delta = current-base
    return base,current,delta

channel_base6,channel_current6,channel_delta6 = client_decomposition6('channel')
category_base6,category_current6,category_delta6 = client_decomposition6('category')
for channel in ['Карта','Токен']:
    idx6[f'change_{channel}6'] = channel_delta6.get(channel,pd.Series(0.0,index=ids6))
channel_losses6 = -channel_delta6.clip(upper=0).sum(axis=1)
idx6['card_loss_share6'] = (-idx6['change_Карта6'].clip(upper=0)).div(channel_losses6.replace(0,np.nan))
idx6['other_channel_growth6'] = channel_delta6.drop(columns='Карта',errors='ignore').clip(lower=0).sum(axis=1)
idx6['card_loss_not_compensated6'] = (
    idx6['change_Карта6'].lt(0)
    & idx6['other_channel_growth6'].lt(-idx6['change_Карта6']))

# Повседневные категории заранее перечислены в EVERYDAY6.
existing_everyday6 = [c for c in EVERYDAY6 if c in category_delta6]

idx6['everyday_change6'] = category_delta6[existing_everyday6].sum(axis=1)
category_losses6 = -category_delta6.clip(upper=0).sum(axis=1)
everyday_losses6 = -category_delta6[existing_everyday6].clip(upper=0).sum(axis=1)
idx6['everyday_loss_share6'] = everyday_losses6.div(category_losses6.replace(0,np.nan))
# При нулевых изменениях idxmin не должен выдавать «главную причину».
idx6['main_reduced_category6'] = category_delta6.idxmin(axis=1).where(category_delta6.min(axis=1).lt(0))

# RUB: месячные суммы успешных оплат без вычитания возвратов.
rub6 = pay6.loc[pay6.currency.astype('string').str.upper().eq('RUB').fillna(False)]
rub_sums6 = rub6.groupby(['client_id','period6']).amount.sum().unstack(fill_value=0).reindex(
    index=ids6,columns=['base','current'],fill_value=0)
idx6['rub_month_base6'] = rub_sums6.base/12
idx6['rub_month_current6'] = rub_sums6.current/6
idx6['rub_month_change6'] = idx6.rub_month_current6-idx6.rub_month_base6

# Попытки и отказы — отдельная совокупность, возвраты исключаем.
attempt6 = transactions_period.loc[
    transactions_period.client_id.isin(ids6)
    & transactions_period.txn_datetime.ge(BASE_START6)
    & transactions_period.txn_datetime.lt(END6)
].copy()
st6 = attempt6.status.astype('string').str.strip().str.casefold()

attempt6 = attempt6.loc[st6.isin(['успешно','отклонено'])].copy()
attempt6['period6'] = np.where(attempt6.txn_datetime.lt(CURRENT_START6),'base','current')
attempt6['refused6'] = attempt6.status.astype('string').str.casefold().eq('отклонено')
a6 = attempt6.groupby(['client_id','period6']).agg(attempts=('txn_id','size'),refused=('refused6','sum'))
for period,months in [('base',12),('current',6)]:
    part = a6.xs(period,level='period6').reindex(ids6,fill_value=0) if period in a6.index.get_level_values('period6') else pd.DataFrame(0,index=ids6,columns=['attempts','refused'])
    idx6[f'attempt_frequency_{period}6'] = part.attempts/months
    idx6[f'refusal_rate_{period}6'] = part.refused.div(part.attempts.replace(0,np.nan))*100
idx6['attempt_frequency_change6'] = idx6.attempt_frequency_current6-idx6.attempt_frequency_base6
campaign_features6 = idx6.reset_index()

## 7.1 Кого в кампанию не берём
Последовательно исключаем из рассмотрения: нет сопоставимой истории; нулевая база; снижение не установлено; нет явного согласия; восстановилась частота во II квартале. Одна основная причина на анкету. Это не удаление очищенных данных.


In [ ]:
reason_pre6 = pd.Series('Прошёл общие условия', index=idx6.index, dtype='string')
free6 = pd.Series(True, index=idx6.index)
conditions6 = [
    (~comparable6, 'Не сопоставим по дате регистрации'),
    (comparable6 & idx6.payments_base.eq(0), 'Нулевая базовая активность'),
    (positive6 & ~idx6.activity_declined.fillna(False), 'Снижение не установлено'),
    (~idx6.contact_allowed6, 'Нет явного согласия на коммуникацию'),
    (idx6.q2_restored6.fillna(False), 'Частота восстановилась во II квартале'),
]
for mask, label in conditions6:
    use6 = free6 & mask.fillna(False)
    reason_pre6.loc[use6] = label
    free6.loc[use6] = False
preselection_summary6 = reason_pre6.value_counts().to_frame('Клиентов')
display(preselection_summary6)
unknown_ops6 = transactions_period.loc[~transactions_period.client_id.isin(ids6)]
unknown_summary6 = pd.Series({
    'Операций без анкеты': len(unknown_ops6),
    'Уникальных непустых ID без анкеты': unknown_ops6.client_id.nunique(),
}, name='Количество')
display(unknown_summary6.to_frame())

### 7.2. Согласие, кандидаты и пересечения
Коммуникация только при явном «да». Сначала показываем пересечения, затем назначаем одну группу. Наличие 150 клиентов в исходном разрезе не гарантирует размер после согласия и пересечений.


In [ ]:
common6 = positive6 & idx6.activity_declined.fillna(False) & idx6.persistent_decline6
segment6=idx6.segment.astype('string')
app6=idx6.has_mobile_app.astype('string')
candidate_masks6=pd.DataFrame({
    'G1':common6 & segment6.eq('Массовый').fillna(False) & app6.eq('Нет').fillna(False),
    'G2':common6 & app6.eq('Да').fillna(False) & idx6.card_loss_not_compensated6 & idx6.card_loss_share6.ge(DOMINANT_SHARE6),
    'G3':common6 & idx6.everyday_change6.lt(0) & idx6.everyday_loss_share6.ge(DOMINANT_SHARE6)
         & idx6.rub_month_base6.gt(0) & idx6.rub_month_change6.lt(0),
}).fillna(False).astype(bool)
PRIORITY6=['G1','G2','G3']
GROUP_NAMES6={'G1':'Массовый без приложения: устойчивое снижение',
    'G2':'С приложением: некомпенсированное падение карты',
    'G3':'Сокращение повседневных оплат и RUB-суммы'}
contact_masks6=candidate_masks6.mul(idx6['contact_allowed6'], axis=0).astype(bool)
overlap6=contact_masks6.astype(int).T @ contact_masks6.astype(int)
display(overlap6)
fig,ax=plt.subplots(figsize=(7,4))
sns.heatmap(overlap6,annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax)
ax.set(title='Пересечения кандидатов с согласием',xlabel='Правило',ylabel='Правило')
plt.tight_layout();plt.show()

Вывод: по отдельным правилам с согласием: G1 — 697, G2 — 772, G3 — 1413. G1 и G2 не пересекаются; G3 пересекается с G1 у 487 клиентов, с G2 у 396. Количества отдельных кандидатов суммировать нельзя.


### 7.3 Назначение одной группы и распределение бюджета

Приоритет G1 → G2 → G3. Сначала назначаем одну группу, резервируем до 150 каждой, затем распределяем остаток до 2000 по приоритету. Внутри группы ранг: абсолютная потеря частоты, отставание II квартала, client_id. Если группа меньше 150, не считаем кампанию допустимой и не экспортируем результат. Пустые места не заполняем случайными клиентами.


In [ ]:
# Обнуляем старый результат: при ошибке не должен оставаться список прошлого запуска.
campaign_list=pd.DataFrame(columns=['client_id','group'])
assigned6=pd.Series(pd.NA,index=ids6,dtype='string',name='group')
for group in PRIORITY6:
    assigned6.loc[assigned6.isna() & contact_masks6[group]]=group
stages6=pd.DataFrame(index=PRIORITY6)
stages6['До согласия']=candidate_masks6.sum()
stages6['После согласия']=contact_masks6.sum()
stages6['После пересечений']=assigned6.value_counts().reindex(PRIORITY6,fill_value=0)
display(stages6)
quota6={g:min(MIN_GROUP6, int(stages6.loc[g,'После пересечений'])) for g in PRIORITY6}
remaining6=BUDGET6-sum(quota6.values())
for group in PRIORITY6:
    extra6=min(max(remaining6,0),int(stages6.loc[group,'После пересечений'])-quota6[group])
    quota6[group]+=extra6;remaining6-=extra6
parts6=[]
for group in PRIORITY6:
    ranked6=idx6.loc[assigned6.eq(group).fillna(False)].copy()
    ranked6['loss_rank6']=-ranked6.activity_change
    ranked6['end_gap_rank6']=ranked6.activity_base-ranked6.q2_frequency6
    ranked6=ranked6.reset_index().sort_values(['loss_rank6','end_gap_rank6','client_id'],
        ascending=[False,False,True],kind='mergesort')
    selected6=ranked6.head(quota6[group])[['client_id']].copy()
    selected6['group']=group;parts6.append(selected6)
campaign_list=pd.concat(parts6,ignore_index=True)
stages6['После бюджета']=campaign_list.group.value_counts().reindex(PRIORITY6,fill_value=0)
display(stages6.rename(index=GROUP_NAMES6))

campaign_ready6 = bool(
    stages6['После бюджета'].ge(MIN_GROUP6).all()
    and 3 <= campaign_list.group.nunique() <= 5
    and len(campaign_list) <= BUDGET6
    and campaign_list.client_id.is_unique)
display(pd.Series({
    'Клиентов': len(campaign_list), 'Групп': campaign_list.group.nunique(),
    'Допустима по размеру и уникальности': campaign_ready6,
}, name='Итог отбора').to_frame())

Вывод: после приоритета G1 → G2 → G3 размеры равны 697, 772 и 530. Всего 1999, три непересекающиеся группы, каждая больше 150. Бюджет никого дополнительно не исключил: размеры до и после бюджетного шага совпадают.


### 7.4 Итоговые решения по всем анкетам

Одна анкета получает одно решение. Если размер кампании недопустим, включение означает попадание в расчётный список, а не готовую кампанию.


In [ ]:
selected_ids6 = set(campaign_list.client_id)
reason6 = reason_pre6.replace('Прошёл общие условия', 'Не соответствует содержательным правилам').copy()
reason6.loc[assigned6.notna() & ~idx6.index.isin(selected_ids6)] = 'Не прошёл бюджет'
reason6.loc[idx6.index.isin(selected_ids6)] = 'Включён'
campaign_decisions6 = idx6[
    ['activity_comparable', 'payments_base', 'activity_declined', 'contact_allowed6']
].copy()
campaign_decisions6['assigned_before_budget'] = assigned6
campaign_decisions6['decision'] = reason6
exclusion_summary6 = reason6.value_counts().to_frame('Клиентов')
display(exclusion_summary6)

### 7.5. Итоговая характеристика групп
Показываем именно выбранных кандидатов. Частота по всем валютам, денежные суммы только RUB. 
Гипотезы коммуникации: G1 — удобство повседневной оплаты без требования установить приложение; 
G2 — барьеры использования карты и способы оплаты; 
G3 — восстановление повседневного использования. 


In [ ]:
final_profiles6=idx6.join(campaign_list.set_index('client_id'),how='inner')
summary6=final_profiles6.groupby('group').agg(n=('activity_base','size'),
    base_mean=('activity_base','mean'),current_mean=('activity_current','mean'),
    q2_mean=('q2_frequency6','mean'),median_change_pct=('activity_change_pct','median'),
    rub_base=('rub_month_base6','mean'),rub_current=('rub_month_current6','mean'))
display(summary6.rename(index=GROUP_NAMES6).round(3))
for group in PRIORITY6:
    group_ids6=campaign_list.loc[campaign_list.group.eq(group),'client_id']
    channel_mean6=channel_delta6.loc[group_ids6].mean().sort_values()
    category_mean6=category_delta6.loc[group_ids6].mean().sort_values()
    display(channel_mean6.to_frame('Вклад канала'))
    display(category_mean6.to_frame('Вклад категории'))
fig,ax=plt.subplots(figsize=(12,5))
for group in PRIORITY6:
    group_ids6=campaign_list.loc[campaign_list.group.eq(group),'client_id']
    ax.plot(months6.to_timestamp(),monthly6.loc[group_ids6].mean(),marker='o',label=f'{group}; n={len(group_ids6)}')
ax.axvline(CURRENT_START6,color='gray',linestyle='--')
ax.axvline(pd.Timestamp('2025-04-01'),color='gray',linestyle=':')
ax.set(title='Итоговые группы: месячная частота',xlabel='Месяц',ylabel='Успешных оплат на клиента в месяц')
ax.set_ylim(bottom=0);ax.tick_params(axis='x',rotation=45);ax.legend()
sns.despine(ax=ax);plt.tight_layout();plt.show()

### 7.6. Решения в журнале
Обновляем журнал после формирования списка и причин; экспорт — в разделе 9.


In [ ]:
add_log(key='clients_campaign_features_chapter6',
 field='q2_frequency6; persistent_decline6; card_loss_share6; everyday_loss_share6; rub_month_base6; rub_month_current6; refusal_rate_base6; refusal_rate_current6; contact_allowed6',
 issue='Нужны клиентские диагностики для правил кампании',scope='Рабочие копии анкет и связанные операции двух периодов',
 n_rows=len(idx6),value_before='Кампанийные диагностики не объединены',
 action='Созданы рабочие признаки без удаления исходных строк',value_after=f'{len(idx6)} клиентов',
 rationale='Нормирование 12/6/3 месяца; частоты по всем валютам; суммы RUB без возвратов; отказы среди успешных и отклонённых. Причины и отклик не установлены.')
add_log(key='clients_campaign_assignment_chapter6',field='group; campaign_decisions6.decision',
 issue='Нужны непересекающиеся группы с согласием и бюджетом',scope='Все анкеты: отбор и причина невключения',
 n_rows=len(idx6),value_before='Пересекающиеся кандидаты',
 action='Применены правила, согласие, приоритет, ранжирование и бюджет',
 value_after=f'Допустимость по размеру и уникальности: {campaign_ready6}; {len(campaign_list)} клиентов; размеры {campaign_list.group.value_counts().to_dict()}; квоты {quota6}; исходные строки не удалены',
 rationale=f'G1 массовый без приложения; G2 с приложением, карта >= {DOMINANT_SHARE6} потерь и не компенсирована; G3 повседневные категории >= {DOMINANT_SHARE6} потерь и RUB-сумма падает. '
 'Для всех общее и сохраняющееся к II кварталу снижение. Приоритет G1→G2→G3. '
 f'Минимум {MIN_GROUP6} каждой группе, остаток по приоритету; бюджет {BUDGET6}. '
 'Ранг: абсолютная потеря, отставание II квартала, client_id. Если группы меньше минимума, экспорт не выполняется. Восстановившиеся исключены.')
# Актуальное представление всех записей, не экспорт до будущего раздела 9.
cleaning_log=pd.DataFrame(list(cleaning_log_entries.values()),columns=LOG_COLUMNS)

In [ ]:
add_log(
    key='clients_age_geo_service_hypotheses',
    field=(
        'age_start_h; age_end_h; age_dec_projection_h; '
        'age_registration_h; возрастные когорты; '
        'городские и сервисные группировки'
    ),
    issue='Нужно проверить возрастное и географическое объяснения снижения',
    scope='Рабочие копии анкет и агрегаты главы 5',
    n_rows=len(age_h),
    value_before='Дополнительные признаки и разрезы отсутствовали',
    action='Созданы рабочие признаки и агрегаты; исходные значения сохранены',
    value_after='Удалено 0 исходных строк; правила G1–G3 не изменены',
    rationale=(
        'Возраст рассчитан на 01.01.2024, 30.06.2025 и дату регистрации. '
        '31.12.2025 — только календарная проекция известного списка. '
        'Новые клиенты учитываются по всем возрастным группам. '
        'Для географии сопоставлены одинаковые первые полугодия '
        'внутри сегмента и лояльности; неизвестные города не включены '
        'в сравнение Екатеринбурга с остальными. '
        'Оценки сгруппированы в 1–2, 3, 4–5 и отдельный пропуск; '
        'малые ячейки скрыты только на графиках, не удалены из данных. '
        'Причинные эффекты не установлены; G4 в кампанию не включена.'
    ),
)

## 8. Что проверили и что не подтвердилось

1. Общий рост неуспешности попыток по карте не подтверждён: доля отказов 2,025% → 2,017%, попыток на клиента в месяц 3,534 → 2,530. Это не доказывает отсутствие локальных проблем.
2. Полная компенсация падения других каналов токеном не подтверждена: на 5240 клиентах токен компенсирует 17,70% суммы отрицательных вкладов; среди снизивших компенсация слабее.
3. Общая гипотеза «реже, но больше» не объясняет RUB-категории: во всех девяти падают частота, месячная сумма и средний чек. При этом индивидуальная компенсация есть у 6,15–9,66% снизивших частоту категории; универсально отвергать гипотезу нельзя.
4. Массовое полное прекращение оплат не объясняет основную картину: из 3560 снизивших пять без оплат за полугодие; большинство продолжают платить.
5. Снижение не связано исключительно с календарным составом базы: при одинаковых полугодиях средний уровень ниже на 10,81%, все шесть месячных сравнений отрицательны. Сезонность и причины этим не установлены.


In [ ]:
# Таблица проверенных гипотез из текущих агрегатов, без новых отборов.
refusal_card8 = attempt_summary5.xs('Карта',level=0)
r0_8 = float(refusal_card8.loc['2024','decline_rate_pct'])
r1_8 = float(refusal_card8.loc['2025: январь–июнь','decline_rate_pct'])
loss8 = -channel_structure5['change'].clip(upper=0).sum()
growth8 = channel_structure5['change'].clip(lower=0).sum()
compensation8 = 100*growth8/loss8 if loss8 else np.nan
category_count8 = len(category_money5)
all_money_down8 = category_money5['spending_change_pct'].lt(0).all()
all_ticket_down8 = category_money5['mean_ticket_change_pct'].lt(0).all()
client_comp_range8 = client_hypothesis5['share_among_fewer_pct'].dropna()
base_positive8 = client_activity['activity_comparable'].fillna(False) & client_activity['payments_base'].gt(0)
declined_count8 = int(client_activity.loc[base_positive8,'activity_declined'].sum())
stopped_count8 = int((base_positive8 & client_activity['payments_current'].eq(0)).sum())
hypotheses_summary8 = pd.DataFrame([
    {'Гипотеза':'Общий рост отказов карты',
     'Результат':f'{r0_8:.3f}% → {r1_8:.3f}%',
     'Интерпретация':'Глобальный рост не подтверждён, если текущая доля не выше; локальные проблемы не исключены'},
    {'Гипотеза':'Полная компенсация каналов',
     'Результат':f'Компенсация {compensation8:.2f}%',
     'Интерпретация':'Частичная, если меньше 100%; сравнивать в фиксированной группе'},
    {'Гипотеза':'Реже, но больше: категории',
     'Результат':f'{CURRENCY5}; категорий {category_count8}; все месячные суммы падают: {all_money_down8}; все средние чеки падают: {all_ticket_down8}',
     'Интерпретация':f'Индивидуальное сохранение суммы при меньшей частоте: {client_comp_range8.min():.2f}–{client_comp_range8.max():.2f}% внутри категорий'},
    {'Гипотеза':'Основная картина — полное прекращение',
     'Результат':f'{stopped_count8} без оплат полугодия из {declined_count8} снизивших',
     'Интерпретация':'Снижение частоты не равно уходу'},
    {'Гипотеза':'Всё снижение объясняется выбором годовой базы',
     'Результат':f'Одинаковые полугодия: {h1_change4:.2f}%; месяцев ниже {months_lower4} из 6',
     'Интерпретация':'Проверка календарной чувствительности, не причинный вывод'},
])

# Дополнительные гипотезы: результаты текущего запуска.
age_result8 = (f'Переходная группа: {senior_change_h:+d} оплат; '
               f'новые всех возрастов: {new_total_h}; частичный баланс {senior_change_h+new_total_h:+d}; '
               f'общий баланс анкет: {int(accounting_h.loc["Итого по всем анкетам", "Изменение I пол. 2025 − I пол. 2024"]):+d}')
geo8 = idx6.loc[positive6 & idx6.city.notna()].copy()
geo8['region8'] = np.where(geo8.city.eq('Екатеринбург'), 'Екатеринбург', 'Остальные города')
for column in ['segment', 'loyalty_tier']:
    geo8[column] = geo8[column].astype('string').fillna('Нет данных')
geo8['base8'] = monthly6.loc[geo8.index, '2024-01':'2024-06'].sum(axis=1)/6
geo8['current8'] = monthly6.loc[geo8.index, '2025-01':'2025-06'].sum(axis=1)/6
geo_table8 = geo8.groupby(['segment','loyalty_tier','region8'], observed=True).agg(
    n=('base8','size'), base=('base8','mean'), current=('current8','mean'))
geo_table8['change'] = 100*(geo_table8.current/geo_table8.base.replace(0,np.nan)-1)
geo_values8 = geo_table8.change.unstack().reindex(columns=['Екатеринбург','Остальные города'])
geo_sizes8 = geo_table8.n.unstack().reindex_like(geo_values8)
geo_valid8 = geo_sizes8.ge(30).all(axis=1) & geo_values8.notna().all(axis=1)
geo_difference8 = (geo_values8['Екатеринбург']-geo_values8['Остальные города']).loc[geo_valid8]
geo_result8 = (f'Общих сочетаний: {len(geo_difference8)}; разница '
               f'{geo_difference8.min():.2f}…{geo_difference8.max():.2f} п.п.')
score8 = pd.to_numeric(idx6.satisfaction_score, errors='coerce')
g4_profile8 = positive6 & idx6.city.eq('Екатеринбург').fillna(False) & score8.between(1,3).fillna(False)
g4_eligible8 = g4_profile8 & idx6.activity_declined.fillna(False) & idx6.persistent_decline6 & idx6.contact_allowed6
service8 = idx6.loc[positive6].copy()
service8['rated8'] = pd.to_numeric(service8.satisfaction_score,errors='coerce').notna()
branch_service8 = service8.groupby('branch_code',observed=True).agg(n=('rated8','size'),rated=('rated8','sum'))
branch_enough8 = int((branch_service8.n.ge(30) & branch_service8.rated.ge(20)).sum())
extra_hypotheses8 = pd.DataFrame([
    {'Гипотеза':'Старение и отсутствие молодых объясняют основное снижение',
     'Результат':age_result8,
     'Интерпретация':'Причинное объяснение не установлено; частичный баланс не равен динамике банка; новые учитываются по всем возрастам.'},
    {'Гипотеза':'Резко обособленное снижение Екатеринбурга',
     'Результат':geo_result8,
     'Интерпретация':'Остаточный разрыв оценивается внутри одинаковых сегмента и лояльности; результат ограничен достаточно крупными сочетаниями, не доказывает отсутствия географического эффекта.'},
    {'Гипотеза':'Плохое обслуживание отдельных отделений вызвало снижение',
     'Результат':f'Отделений с n >= 30 и >= 20 оценок: {branch_enough8}; оценка отсутствует у {100*(1-service8.rated8.mean()):.2f}% рассматриваемой базы',
     'Интерпретация':'Недостаточно для причинного вывода; нет даты оценки и привязки к посещению отделения. Не считаем опровергнутой.'},
    {'Гипотеза':'Екатеринбург из-за плохих оценок',
     'Результат':f'Положительная сопоставимая база: {int(g4_profile8.sum())}; после устойчивого снижения и согласия до пересечений: {int(g4_eligible8.sum())}',
     'Интерпретация':'Правило не включено: допустимый окончательный размер не установлен, расширение оценок разведочное. Отказ от правила не опровергает сервисную гипотезу.'},
])
hypotheses_summary8 = pd.concat([hypotheses_summary8,extra_hypotheses8],ignore_index=True)
display(hypotheses_summary8)

## 9. Сохранение результатов

### 9.1.Готовность кампании - контроль

In [ ]:
display(pd.Series({
    'Клиентов кампании':len(campaign_list),
    'Групп кампании':campaign_list.group.nunique(),
    'Допустима по размеру и уникальности':campaign_ready6,
},name='Перед экспортом').to_frame())

### 9.2. Экспорт

Сохраняем две очищенные таблицы, итоговый журнал и список кампании; без JSON и отдельных агрегатов. Исходные файлы не меняем. При чтении CSV даты и типы восстанавливаются явно.


In [ ]:
processed_dir9 = Path('data/processed')
processed_dir9.mkdir(parents=True, exist_ok=True)
# Актуальный журнал после всех разделов.
cleaning_log = pd.DataFrame(list(cleaning_log_entries.values()), columns=LOG_COLUMNS,)
clients_clean9 = idx6.join(campaign_decisions6[['assigned_before_budget', 'decision']]
).reset_index()
# При недопустимой кампании не обозначаем расчётный отбор как итоговую группу.
clients_clean9['campaign_group'] = (clients_clean9.client_id.map(campaign_list.set_index('client_id')['group'])
    if campaign_ready6
    else pd.Series(pd.NA, index=clients_clean9.index, dtype='string')
)
# Очищенные данные и журнал сохраняем независимо от готовности кампании.
clients_clean9.to_csv(processed_dir9 / 'clients_clean.csv', index=False, encoding='utf-8-sig')
transactions_period.to_csv(processed_dir9 / 'transactions_clean.csv', index=False, encoding='utf-8-sig')
cleaning_log.to_csv('cleaning_log.csv', index=False, encoding='utf-8-sig')
# Итоговый список сохраняем только для допустимой кампании.
if campaign_ready6:
    campaign_list[['client_id', 'group']].to_csv('campaign_list.csv', index=False, encoding='utf-8-sig')
display(pd.Series({'Очищенные таблицы и журнал сохранены': True,'Список кампании сохранён в этом запуске': campaign_ready6,}, name='Экспорт').to_frame())

### 9.3. Итог работы и передача в отчёт

Кампания состоит из 1999 клиентов: G1 — 697, G2 — 772, G3 — 530. Все имеют явное согласие, client_id не повторяются, минимальные размеры соблюдены. Размеры после бюджетного шага совпадают с размерами после назначения: бюджетное исключение на текущих данных не понадобилось.